# บทที่ 11 โครงข่ายคอนโวลูชันสำหรับภาพทางฟิสิกส์: โน้ตบุ๊กสำหรับชั้นเรียน
โน้ตบุ๊กประกอบตำรา *วิทยาศาสตร์ข้อมูลและการเรียนรู้ของเครื่องสำหรับนักฟิสิกส์*

โน้ตบุ๊กนี้ใช้ในชั้นเรียนได้ด้วยไฟล์นี้ไฟล์เดียว ไม่ต้องดึงโค้ดจาก GitHub เนื้อหาเรียงตามบทที่ 11 ในตำรา แบ่งเป็นสี่คาบ รวม 240 นาทีตามแผนเวลาในคู่มือผู้สอน (คาบละราว 60 นาที) หัวข้อที่มีป้าย **โค้ดที่ 11.N** ตรงกับตัวอย่างโค้ดเลขเดียวกันในตำรา

## วัตถุประสงค์
เมื่อทำโน้ตบุ๊กนี้จนจบ ผู้เรียนควรสามารถ

1. อธิบายว่าการเชื่อมต่อเฉพาะที่และการใช้น้ำหนักร่วมกันทำให้โครงข่ายคอนโวลูชันใช้พารามิเตอร์น้อยกว่าและทนต่อการเลื่อนของภาพได้ดีกว่าโครงข่ายหลายชั้นอย่างไร
2. คำนวณขนาดของผลลัพธ์ จำนวนพารามิเตอร์ และขอบเขตการรับรู้ของโครงข่ายคอนโวลูชันได้
3. เขียนและฝึกโครงข่ายคอนโวลูชันด้วย PyTorch และเลือกการเพิ่มข้อมูลที่สอดคล้องกับความสมมาตรของระบบทางฟิสิกส์ได้
4. อ่านเมทริกซ์ความสับสนในเชิงฟิสิกส์ และบอกได้ว่าเมื่อใดการถ่ายโอนการเรียนรู้ช่วย และเมื่อใดลักษณะเด่นทางกายภาพที่ง่ายกว่าดีกว่า
5. ใช้ Grad-CAM ตรวจว่าโครงข่ายตัดสินจากส่วนใดของภาพ และบอกได้ว่าวิธีนี้ยืนยันอะไรได้และยืนยันอะไรไม่ได้
6. ประเมินความไม่แน่นอนด้วยการรวมกลุ่มโครงข่าย รู้ข้อจำกัดของมันกับข้อมูลนอกการแจกแจง และเขียนการ์ดแบบจำลองได้

## ข้อมูลในโน้ตบุ๊กนี้
- **ภาพกาแล็กซีจำลอง (SYNTHETIC):** ภาพขาวดำ $64\times64$ ห้าประเภท สร้างจากโพรไฟล์ของเซอร์ซิก จานเอกซ์โพเนนเชียล แขนเกลียวลอการิทึม และคาน (โค้ดที่ 11.2)
- **ภาพเมล็ดข้าวจำลอง (SYNTHETIC):** สี่ "พันธุ์" ที่ต่างกันที่ความยาวและความกว้าง (โค้ดที่ 11.7)
- **ภาพดาวฤกษ์และกาแล็กซีคู่จำลอง (SYNTHETIC):** ใช้เป็นข้อมูลนอกการแจกแจง (โค้ดที่ 11.9)
- **Galaxy10 DECaLS ของจริง (ตัวอย่างย่อย):** ภาพจริง 50 ภาพ (ประเภทละ 5 ภาพ) และป้ายกำกับของทั้ง 17,736 ภาพ ดาวน์โหลดจาก GitHub ของตำรา (ราว 7 MB) สัญญาอนุญาต CC BY 4.0 ใช้แสดงการอ่านข้อมูลด้วยโค้ดที่ 11.6 ในส่วนที่ 4 เท่านั้น
- **สายพอลิเมอร์จากบทที่ 10 (SYNTHETIC):** ใช้ในแบบฝึกหัด E11.1 นอกชั้น

หัวกราฟที่มาจากข้อมูลจำลองมีคำว่า SYNTHETIC กำกับ ตัวเลขใช้เพื่อการเรียนวิธีการ ไม่ใช่ผลการวิจัย ตัวเลขของงานจำแนกพันธุ์เมล็ดข้าวจริงในส่วนที่ 5 มาจากบทความที่ตีพิมพ์แล้ว

> **ตัวเลขอาจต่างจากตำรา:** การฝึกโครงข่ายคอนโวลูชันไม่ได้ผลเหมือนเดิมทุกหลักเมื่อเปลี่ยนเครื่องหรือรุ่นของ PyTorch แม้กำหนดเมล็ดสุ่มเดียวกัน ตำรารายงานว่าเมื่อเปลี่ยนเมล็ดสุ่มหรือคอมพิวเตอร์ โครงข่ายคอนโวลูชันได้ความแม่นยำร้อยละ 93 ถึง 99 กับภาพปกติ และ 91 ถึง 99 กับภาพที่เลื่อน ตัวเลขบน Colab จึงอาจต่างจากตำราได้หลายจุดร้อยละ แต่ข้อสรุปหลักไม่เปลี่ยน ได้แก่ ความต่างกับโครงข่ายหลายชั้นเมื่อภาพเลื่อน ผลของทางลัด และความมั่นใจเกินเหตุกับภาพนอกการแจกแจง เซลล์ตรวจคำตอบทุกเซลล์เทียบกับผลที่คำนวณบนเครื่องที่ใช้อยู่ จึงไม่ขึ้นกับความต่างนี้

## วิธีใช้
1. รันเซลล์ในส่วนที่ 0 ก่อนเสมอ
2. **แต่ละคาบเริ่มได้โดยไม่ต้องรันคาบก่อนหน้า** ถ้าเรียนต่อหลังจากปิด Colab ไปแล้ว ให้คลิกเซลล์หัวคาบที่ 1 แล้วเลือก **Runtime → Run before** (ซึ่งรันเฉพาะส่วนที่ 0) แล้วเลื่อนลงไปรันเซลล์ของคาบที่ต้องการต่อ เซลล์แรกของคาบที่ 3 และ 4 และของงานนอกชั้นเตรียมภาพกาแล็กซีและโครงข่ายที่ฝึกแล้ว ถ้ายังไม่มีไฟล์น้ำหนักที่บันทึกไว้ในคาบที่ 2 เซลล์นั้นจะฝึกโครงข่ายใหม่ (ราวสิบนาทีบน Colab) การติ๊ก `USE_DRIVE` ในส่วนที่ 0 ทำให้ไฟล์น้ำหนักไม่หายระหว่างคาบ
3. ภายในคาบ ให้รันเซลล์ทีละเซลล์ตามลำดับจากบนลงล่างด้วย `Shift + Enter`
4. ใช้หน่วยประมวลผลกลาง (CPU) ตามค่าเริ่มต้น ไม่ต้องเปลี่ยนเป็น GPU
5. เซลล์ที่มีเครื่องหมาย ✏️ เป็นแบบฝึกที่ต้องแก้โค้ดหรือเติมคำตอบเอง ส่วนเซลล์ ✅ ใช้ตรวจคำตอบ เซลล์แบบฝึกที่ยังไม่ได้เติมจะแสดงข้อความเตือน แต่ไม่ทำให้ Run all หยุด
6. เซลล์ที่มีเครื่องหมาย ⏱ ใช้เวลารันราวหนึ่งนาทีหรือมากกว่าบน Colab ระหว่างรอให้อ่านคำอธิบายของเซลล์ถัดไป บทนี้ฝึกโครงข่ายคอนโวลูชันหลายตัว และบน Colab (หน่วยประมวลผลกลางหนึ่งเธรด) การฝึกหนึ่งตัวใช้เวลาราวสิบนาที เซลล์ที่ฝึกโครงข่ายในคาบที่ 2 และโค้ดที่ 11.8 ในคาบที่ 4 จึงใช้เซลล์ละราวสิบนาที **สมาชิกตัวที่ 1 และ 2 ของการรวมกลุ่มในคาบที่ 4 (โค้ดที่ 11.9 ฉบับสามตัว) เป็นงานก่อนคาบที่ 4** ผู้เรียนรันเซลล์ท้ายคาบที่ 3 ที่บ้าน (ราว 25 นาที) โดยติ๊ก `USE_DRIVE` ไว้ แล้วในคาบที่ 4 เซลล์ของโค้ดที่ 11.9 จะโหลดน้ำหนักมาใช้ภายในไม่ถึงหนึ่งนาที ถ้าไม่พบไฟล์ เซลล์จะฝึกสมาชิกที่ขาดในชั้น (ราว 25 นาที) ส่วนโค้ดที่ 11.9 ฉบับเต็มห้าตัว (ราวหนึ่งชั่วโมง) อยู่ในงานนอกชั้น ถ้าใช้ Runtime → Run all ทั้งโน้ตบุ๊กจะใช้เวลาราวสองชั่วโมงครึ่ง
7. ถ้าเปิดโน้ตบุ๊กจากลิงก์ GitHub บน Colab จะมีคำเตือนว่าโน้ตบุ๊กไม่ได้สร้างโดย Google ให้กด **Run anyway**

> 🔒 แนวคำตอบถูกล็อกไว้จนกว่าผู้เรียนจะตอบข้อนั้นครบ ผู้สอนเปิดแนวคำตอบทั้งหมดได้ด้วยตัวเลือก `SHOW_ANSWERS` ในส่วนที่ 0

> ป้ายกำกับในกราฟเขียนเป็นภาษาอังกฤษ เพราะแบบอักษรเริ่มต้นของ matplotlib บน Colab แสดงอักษรไทยไม่ได้

## สารบัญ

| คาบ | ส่วน | เนื้อหา | หัวข้อในตำรา | เวลาโดยประมาณ |
|---|---|---|---|---|
| — | 0 | เตรียมไลบรารีและโฟลเดอร์ทำงาน | — | 2 นาที |
| 1 | 1 | จากโครงข่ายหลายชั้นสู่โครงข่ายคอนโวลูชัน | 11.1 | 30 นาที |
| 1 | 2 | ส่วนประกอบของโครงข่ายคอนโวลูชัน (โค้ดที่ 11.1) | 11.2 | 30 นาที |
| 2 | 2–3 | แบบฝึก E11.2 (ทบทวนส่วนที่ 2) แล้วฝึกโครงข่ายคอนโวลูชันกับภาพกาแล็กซีจำลอง (โค้ดที่ 11.2–11.5, E11.3) | 11.2–11.3 | 5 + 55 นาที |
| 3 | 4 | ข้อมูลจริง: Galaxy10 DECaLS (โค้ดที่ 11.6) | 11.4 | 20 นาที |
| 3 | 5 | งานวิจัยของผู้เขียน: การจำแนกพันธุ์เมล็ดข้าวไทย (โค้ดที่ 11.7, E11.5) | 11.5 | 40 นาที |
| 4 | 6 | โครงข่ายมองอะไร: Grad-CAM และการเรียนรู้ทางลัด (โค้ดที่ 11.8) | 11.6 | 30 นาที |
| 4 | 7 | ความไม่แน่นอน การ์ดแบบจำลอง และจริยธรรมของข้อมูล (โค้ดที่ 11.9) | 11.7 | 30 นาที |
| นอกชั้น | 8 | E11.1–E11.7 ส่วนที่เหลือ และโค้ดที่ 11.9 ฉบับเต็ม | 11.1–11.7 | — |

แบบฝึกที่มีเลข 11.A–11.G เป็นแบบฝึกเสริมในชั้นเรียนที่ดัดแปลงจากคำถามทบทวนท้ายบทและข้อความในตำรา ส่วนแบบฝึกที่มีเลข E11.N ตรงกับแบบฝึกหัดในตำรา

---
# ส่วนที่ 0 เตรียมไลบรารีและโฟลเดอร์ทำงาน

เซลล์ถัดไปสร้างโฟลเดอร์ทำงาน `ch11_cnn` แล้วย้ายเข้าไปทำงานในโฟลเดอร์นั้น ถ้าเปิดโน้ตบุ๊กจากโฟลเดอร์ `code/notebooks/` ในเครื่องของตนเอง เซลล์จะใช้โฟลเดอร์ `code/ch11_cnn/` ที่มีอยู่แล้ว ภาพกาแล็กซีจำลองและน้ำหนักของโครงข่ายที่ฝึกในโน้ตบุ๊กนี้จะบันทึกไว้ในโฟลเดอร์ย่อย `outputs/` เพื่อใช้ซ้ำในคาบถัดไป บน Colab ไฟล์ในโฟลเดอร์นี้จะหายเมื่อปิด session **ถ้าติ๊กช่อง `USE_DRIVE`** ในเซลล์ถัดไป โฟลเดอร์ `outputs/` จะชี้ไปที่โฟลเดอร์ `dsml_ch11_outputs` ใน Google Drive ของผู้เรียน (Colab จะขอสิทธิ์เข้าถึง Drive หนึ่งครั้ง) น้ำหนักที่ฝึกในคาบที่ 2 และในงานก่อนคาบที่ 4 จึงยังอยู่เมื่อเปิดโน้ตบุ๊กใหม่ในวันเรียน ผู้เรียนควรติ๊กช่องนี้ตั้งแต่คาบที่ 2

บทนี้ใช้ PyTorch (`torch`) ต่อจากบทที่ 10 และใช้ `h5py` อ่านไฟล์ Galaxy10 DECaLS Colab ติดตั้งทั้งสองไว้แล้ว ถ้ารันในเครื่องของตนเองและยังไม่มี เซลล์จะติดตั้งให้ บทนี้ไม่ต้องใช้ `torchvision` หรือไลบรารี Grad-CAM สำเร็จรูป เพราะเขียนทุกส่วนเอง

In [ ]:
import os, sys, subprocess, importlib.util

if os.path.basename(os.getcwd()) != "ch11_cnn":
    if os.path.isdir("../ch11_cnn"):                # เปิดจาก code/notebooks/ ในเครื่องของตนเอง
        os.chdir("../ch11_cnn")
    else:                                           # Colab หรือโฟลเดอร์ว่าง
        os.makedirs("ch11_cnn", exist_ok=True)
        os.chdir("ch11_cnn")
# เก็บโฟลเดอร์ outputs/ ไว้ใน Google Drive เพื่อให้น้ำหนักของโครงข่ายที่ฝึกแล้ว (รวมทั้งงานก่อนคาบที่ 4) ไม่หายเมื่อปิด Colab
USE_DRIVE = False  #@param {type:"boolean"}
if USE_DRIVE and importlib.util.find_spec("google.colab") is not None:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    _keep = "/content/drive/MyDrive/dsml_ch11_outputs"
    os.makedirs(_keep, exist_ok=True)
    if os.path.isdir("outputs") and not os.path.islink("outputs"):
        shutil.copytree("outputs", _keep, dirs_exist_ok=True)   # keep files made earlier in this session
        shutil.rmtree("outputs")
    if not os.path.islink("outputs"):
        os.symlink(_keep, "outputs")
    print("outputs/ ->", _keep)
os.makedirs("outputs", exist_ok=True)
print("โฟลเดอร์ทำงาน:", os.getcwd())

_need = {"sklearn": "scikit-learn", "scipy": "scipy", "torch": "torch", "h5py": "h5py"}
missing = [pip for mod, pip in _need.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("ติดตั้งเพิ่ม:", missing if missing else "ไม่มี (มีครบแล้ว)")

### การล็อกแนวคำตอบ
เซลล์ ✅ และเซลล์แนวคำตอบในโน้ตบุ๊กนี้แสดงคำตอบอ้างอิงเมื่อผู้เรียนเติมคำตอบในเซลล์ ✏️ ของข้อนั้นครบแล้วเท่านั้น ถ้ายังไม่ได้ตอบ เซลล์จะแสดงเพียงข้อความ 🔒 บอกว่ายังขาดคำตอบใด การคำนวณในเซลล์ยังทำงานตามปกติ ดังนั้น Run all จึงรันได้ครบโดยไม่เปิดเผยคำตอบ แนวคำตอบของงานเขียนและงานนอกชั้นซึ่งตรวจด้วยโค้ดไม่ได้จะเปิดเมื่อผู้สอนเลือกเท่านั้น

**สำหรับผู้สอน:** ติ๊กช่อง `SHOW_ANSWERS` ในเซลล์ถัดไป (หรือแก้เป็น `True`) แล้วรันเซลล์นั้นอีกครั้ง เพื่อแสดงแนวคำตอบทุกข้อ เช่น เมื่อเฉลยในชั้นหรือเตรียมสอน

In [ ]:
#@title ⚙️ ตัวเลือกของผู้สอน: แสดงแนวคำตอบ
SHOW_ANSWERS = False  #@param {type:"boolean"}

def _answered(v):
    """True when an exercise variable holds an answer (not None, "?", "", empty or containing blanks)."""
    if v is None or v is Ellipsis:
        return False
    if isinstance(v, str):                       # placeholders such as "?", "?-?", "..." count as blank
        return v.replace("?", "").replace("-", "").replace(".", "").strip() != ""
    if isinstance(v, dict):
        return len(v) > 0 and all(_answered(x) for x in v.values())
    if isinstance(v, (list, tuple, set, frozenset)):
        return len(v) > 0 and all(_answered(x) for x in v)
    if type(v).__name__ == "ndarray":
        return v.size > 0
    return True

def _gate(names=()):
    """Return print if the answers may be shown, otherwise print one lock message and return a no-op."""
    if SHOW_ANSWERS:
        return print
    if names:
        missing = [n for n in names if not _answered(globals().get(n))]
        if not missing:
            return print
        print("🔒 ยังไม่ได้ตอบ:", ", ".join(missing),
              "— เติมคำตอบในเซลล์ ✏️ ด้านบนให้ครบก่อน แล้วรันเซลล์นี้อีกครั้ง")
    else:
        print("🔒 แนวคำตอบข้อนี้ผู้สอนเป็นผู้เปิด (ตั้ง SHOW_ANSWERS = True ในส่วนที่ 0)")
    return lambda *args, **kwargs: None

print("แสดงแนวคำตอบทั้งหมด:", "ใช่ (โหมดผู้สอน)" if SHOW_ANSWERS else "ไม่ (แสดงเมื่อผู้เรียนตอบข้อนั้นครบแล้ว)")

In [ ]:
import time, warnings
import numpy as np
import matplotlib.pyplot as plt
import scipy
import sklearn
import torch

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

for name, mod in [("Python", sys), ("numpy", np), ("scipy", scipy), ("scikit-learn", sklearn), ("torch", torch)]:
    ver = sys.version.split()[0] if name == "Python" else mod.__version__
    print(f"{name:12s} {ver}")
print("CPU threads for torch:", torch.get_num_threads())

---
# คาบที่ 1 จากโครงข่ายหลายชั้นสู่โครงข่ายคอนโวลูชัน

> **เริ่มคาบนี้หลังจากปิด Colab ไปแล้ว:** คลิกเซลล์นี้ แล้วเลือก Runtime → Run before เพื่อรันส่วนที่ 0 ใหม่ก่อนเรียนต่อ ถ้าต้องการเริ่มคาบอื่น ให้ทำแบบเดียวกันที่เซลล์นี้ แล้วเลื่อนลงไปยังคาบนั้น

---
# ส่วนที่ 1 จากโครงข่ายหลายชั้นสู่โครงข่ายคอนโวลูชัน (หัวข้อ 11.1)

### แนวคิดสำคัญ
ภาพเป็นข้อมูลที่นักฟิสิกส์สร้างขึ้นมากที่สุดชนิดหนึ่ง ได้แก่ ภาพกาแล็กซีจากกล้องโทรทรรศน์สำรวจท้องฟ้า แผนที่ความสูงและความแข็งจากกล้องจุลทรรศน์แรงอะตอม พลังงานบนเซลล์ของแคลอริมิเตอร์ และภาพการเลี้ยวเบนของรังสีเอกซ์ ข้อมูลเหล่านี้มีลักษณะร่วมกันอย่างหนึ่ง คือความหมายของพิกเซลแต่ละตัวขึ้นกับพิกเซลรอบข้าง จุดสว่างจุดเดียวไม่บอกอะไร แต่แนวของจุดสว่างที่ต่อกันเป็นเกลียวบอกว่ากาแล็กซีมีแขน

### ทำไมโครงข่ายหลายชั้นจึงไม่เหมาะกับภาพ
ปัญหาแรกคือจำนวนพารามิเตอร์ ภาพขาวดำ $64\times64$ มีพิกเซล 4,096 ตัว ถ้าส่งเข้าโครงข่ายหลายชั้นที่มีหน่วยในชั้นซ่อนแรก 128 หน่วย ชั้นแรกชั้นเดียวก็มีน้ำหนัก $4{,}096\times128$ ตัวแล้ว สำหรับภาพจริงของ Galaxy10 ขนาด $256\times256$ พิกเซลสามแถบสี ตัวเลขนี้เพิ่มเป็นกว่ายี่สิบห้าล้าน ซึ่งมากเกินไปสำหรับข้อมูลเพียงหลักหมื่นภาพ

ปัญหาที่ลึกกว่าคือสิ่งที่โครงข่ายต้องเรียน น้ำหนักที่เชื่อมพิกเซลที่ตำแหน่ง $(10, 20)$ เป็นคนละตัวกับน้ำหนักของพิกเซลที่ $(40, 30)$ ถ้าโครงข่ายเรียนว่าแขนกังหันที่มุมบนซ้ายหมายถึงกาแล็กซีกังหัน มันต้องเรียนแยกอีกครั้งว่าแขนกังหันที่มุมล่างขวาก็หมายถึงสิ่งเดียวกัน เปรียบได้กับคนที่จำหน้าเพื่อนจากรูปถ่ายที่เพื่อนยืนกลางภาพเสมอ แล้วจำไม่ได้เมื่อเพื่อนยืนชิดขอบภาพ ส่วนที่ 3 จะวัดผลนี้ด้วยตัวเลข

### ✏️ แบบฝึก 11.A (4 นาที) จำนวนน้ำหนักในชั้นแรกของโครงข่ายหลายชั้น
คำนวณจำนวน **น้ำหนัก** (ไม่รวมไบแอส) ของชั้นเชื่อมต่อเต็มชั้นแรกที่มี 128 หน่วย เมื่อค่าเข้าเป็น (ก) ภาพขาวดำ $64\times64$ เก็บใน `my_w_64` และ (ข) ภาพสี $256\times256\times3$ ของ Galaxy10 เก็บใน `my_w_g10`

In [ ]:
my_w_64 = None    # ✏️ weights of the first layer, 64 x 64 input, 128 units
my_w_g10 = None   # ✏️ 256 x 256 x 3 input, 128 units

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_w_64', 'my_w_g10'])
import torch.nn as nn
for nm, mine, n_in in [("64 x 64      ", my_w_64, 64 * 64), ("256 x 256 x 3", my_w_g10, 256 * 256 * 3)]:
    ref = nn.Linear(n_in, 128, bias=False).weight.numel()
    _say(nm, ":", "(ยังไม่ได้ตอบ)" if mine is None else ("✔" if mine == ref else "✘"), f"— {ref:,}")

In [ ]:
#@title แนวคำตอบ 11.A
_say = _gate(['my_w_64', 'my_w_g10'])
_say('''แนวคำตอบ 11.A
(ก) 4,096 x 128 = 524,288 น้ำหนัก   (ข) 196,608 x 128 = 25,165,824 น้ำหนัก (กว่ายี่สิบห้าล้าน)
จำนวนน้ำหนักของชั้นเชื่อมต่อเต็มโตตามจำนวนพิกเซล ภาพใหญ่ขึ้นสี่เท่าต่อด้านและมีสามแถบสีจึงเพิ่มขึ้น 48 เท่า''')

### ความคิดของโครงข่ายคอนโวลูชัน
โครงข่ายคอนโวลูชัน (convolutional neural network, CNN) แก้ปัญหาทั้งสองด้วยความคิดสองข้อ

1. **การเชื่อมต่อเฉพาะที่ (local connectivity):** หน่วยแต่ละตัวในชั้นคอนโวลูชันมองเพียงหน้าต่างเล็ก ๆ ของภาพ เช่น $3\times3$ พิกเซล เพราะรูปแบบพื้นฐานของภาพ เช่น ขอบ เส้น หรือจุดสว่าง อยู่ในบริเวณเล็ก ๆ
2. **การใช้น้ำหนักร่วมกัน (weight sharing):** หน้าต่างชุดน้ำหนักเดียวกัน เรียกว่าตัวกรอง (filter) หรือเคอร์เนล (kernel) ถูกเลื่อนไปทั่วทั้งภาพ ตัวกรองที่ตรวจขอบแนวทแยงได้ที่ตำแหน่งหนึ่งจึงตรวจได้ทุกตำแหน่งโดยอัตโนมัติ เปรียบได้กับการตรวจเอกสารด้วยแว่นขยายอันเดียวที่เลื่อนไปทีละช่องทั่วทั้งหน้า

ผลของการใช้น้ำหนักร่วมกันมีชื่อว่าความแปรตามการเลื่อน (translation equivariance) ถ้าเลื่อนภาพเข้าไปห้าพิกเซล แผนที่ที่ได้จากชั้นคอนโวลูชันก็เลื่อนไปห้าพิกเซลตามโดยค่าไม่เปลี่ยน ถ้าสุดท้ายนำแผนที่นั้นมาเฉลี่ยทั้งภาพ ซึ่งเรียกว่าการรวมแบบเฉลี่ยทั้งหมด (global average pooling) ผลลัพธ์ก็ไม่เปลี่ยนเลยเมื่อภาพเลื่อน ซึ่งเรียกว่าความไม่แปรเปลี่ยนต่อการเลื่อน (translation invariance) ชั้นคอนโวลูชันให้ความแปรตาม ส่วนความไม่แปรเปลี่ยนมาจากการรวมในตอนท้าย งานจำแนกกาแล็กซีต้องการความไม่แปรเปลี่ยน เพราะชนิดของกาแล็กซีไม่ขึ้นกับตำแหน่งในภาพ แต่งานหาตำแหน่งของดาวในภาพต้องการความแปรตาม เพราะคำตอบต้องเลื่อนตามดาว

เซลล์ถัดไปตรวจสมบัติทั้งสองด้วยชั้นที่ยังไม่ได้ฝึก (น้ำหนักสุ่ม) และภาพจุดสว่างหนึ่งจุด แล้วเทียบกับชั้นเชื่อมต่อเต็มซึ่งไม่มีสมบัตินี้

In [ ]:
import torch.nn as nn

torch.manual_seed(0)
yy, xx = np.mgrid[:64, :64]
blob = np.exp(-((xx - 24)**2 + (yy - 30)**2) / 20).astype(np.float32)   # one bright spot, off-centre
x = torch.tensor(blob[None, None])
x_shift = torch.roll(x, 5, dims=3)                     # the same spot, 5 px to the right

conv = nn.Conv2d(1, 8, 3, padding=1)                   # untrained: random weights
dense = nn.Linear(64 * 64, 8)
with torch.no_grad():
    a, b = conv(x), conv(x_shift)
    print("conv layer : max |conv(shifted x) - shifted conv(x)| =", f"{(b - torch.roll(a, 5, dims=3)).abs().max().item():.1e}")
    print("             map averaged over the image, original:", a.mean(dim=(2, 3))[0, :4].numpy().round(4))
    print("             map averaged over the image, shifted :", b.mean(dim=(2, 3))[0, :4].numpy().round(4))
    d0, d1 = dense(x.flatten(1)), dense(x_shift.flatten(1))
    print("dense layer: output, original:", d0[0, :4].numpy().round(4))
    print("             output, shifted :", d1[0, :4].numpy().round(4))

fig, ax = plt.subplots(1, 4, figsize=(10, 2.6))
for i, (img, t) in enumerate([(x, "image"), (a[:, :1], "conv map 0"), (x_shift, "shifted image"), (b[:, :1], "conv map 0 (shifted)")]):
    ax[i].imshow(img[0, 0].detach().numpy(), cmap="gray", origin="lower"); ax[i].set_title(t, fontsize=9); ax[i].axis("off")
fig.tight_layout(); plt.show()

ผลของเซลล์บนยืนยันความคิดทั้งสองด้วยตัวเลข แผนที่ของชั้นคอนโวลูชันเลื่อนตามภาพโดยค่าไม่เปลี่ยน (ผลต่างเป็นศูนย์ในระดับความละเอียดของเลขทศนิยม) ค่าเฉลี่ยทั้งแผนที่จึงเท่าเดิม ขณะที่ผลลัพธ์ของชั้นเชื่อมต่อเต็มเปลี่ยนไปทั้งหมด เพราะจุดสว่างไปตกบนน้ำหนักชุดใหม่

> **ความเข้าใจคลาดเคลื่อนที่พบบ่อย:** "โครงข่ายคอนโวลูชันไม่แปรเปลี่ยนต่อการแปลงทุกชนิดของภาพ" โครงสร้างของโครงข่ายคอนโวลูชันรับประกันเฉพาะความสมมาตรต่อการเลื่อน และแม้แต่ข้อนี้ก็รับประกันไม่สมบูรณ์เพราะผลของขอบภาพและการลดขนาดแบบรวม มันไม่รับประกันความไม่แปรเปลี่ยนต่อการหมุน การสะท้อน หรือการเปลี่ยนขนาด กาแล็กซีกังหันที่หมุนไปสี่สิบห้าองศาเป็นรูปแบบใหม่สำหรับตัวกรองที่เรียนจากภาพในทิศเดิม ความสมมาตรเหล่านี้ต้องสอนผ่านการเพิ่มข้อมูล (ส่วนที่ 3) หรือสร้างเข้าไปในโครงสร้างด้วยโครงข่ายที่แปรตามกลุ่มของการแปลง (group-equivariant CNN) ซึ่งอยู่นอกขอบเขตของบทนี้

### จากเวกเตอร์ระยะห่างของพอลิเมอร์สู่ภาพ
ความคิดเรื่องโครงสร้างเชิงปริภูมิไม่ได้จำกัดอยู่ที่ภาพถ่าย ในบทที่ 10 โครงข่ายรับระยะห่างระหว่างมอนอเมอร์ 435 คู่เป็นเวกเตอร์ยาว ซึ่งทิ้งข้อมูลว่าคู่ใดอยู่ใกล้คู่ใดในสาย ระยะห่างเหล่านั้นเรียงเป็นเมทริกซ์ระยะห่างขนาด $30\times30$ ได้ โดยสมาชิกแถว $i$ หลัก $j$ คือระยะระหว่างมอนอเมอร์ $i$ กับ $j$ เมทริกซ์นี้คือภาพ แถบใกล้เส้นทแยงบอกโครงสร้างเฉพาะที่ของสาย ส่วนบริเวณไกลเส้นทแยงบอกว่าปลายทั้งสองของสายเข้ามาใกล้กันหรือไม่ ซึ่งเป็นลายเซ็นของการยุบตัว

เซลล์ถัดไปวาดเมทริกซ์ระยะห่างของสายของเล่นสองเส้น (ไม่ใช่ข้อมูลของบทที่ 10) คือสายที่เดินสุ่มแบบขดลวดและสายเดียวกันที่ถูกบีบให้อัดแน่นเป็นก้อน แบบฝึกหัด E11.1 ในงานนอกชั้นใช้ข้อมูลจริงของบทที่ 10

In [ ]:
rng = np.random.default_rng(1)
steps = rng.normal(size=(30, 3)); steps /= np.linalg.norm(steps, axis=1, keepdims=True)
coil = np.cumsum(steps, axis=0)                              # random-walk chain, bond length 1
globule = coil - coil.mean(0)
globule = globule / np.linalg.norm(globule, axis=1, keepdims=True).max() * 1.6   # squeezed into a small ball
vmax = np.linalg.norm(coil[:, None] - coil[None], axis=2).max()     # one colour scale for both panels
fig, ax = plt.subplots(1, 2, figsize=(7.5, 3.3))
for a_, c_, t in [(ax[0], coil, "coil-like chain"), (ax[1], globule, "compact globule")]:
    D_ = np.linalg.norm(c_[:, None] - c_[None], axis=2)
    im = a_.imshow(D_, cmap="viridis", origin="upper", vmin=0, vmax=vmax)
    a_.set_title(f"distance matrix: {t}\n(SYNTHETIC toy chain)", fontsize=9)
    a_.set_xlabel("monomer j"); a_.set_ylabel("monomer i"); a_.grid(False); fig.colorbar(im, ax=a_, shrink=0.8)
fig.tight_layout(); plt.show()

In [ ]:
#@title คำถามทบทวนข้อ 2 (แนวคำตอบ)
print('''คำถามทบทวนข้อ 2: ข้อมูลทางฟิสิกส์ชนิดใดเหมาะกับโครงข่ายคอนโวลูชัน และชนิดใดไม่เหมาะ''')
_say = _gate([])
_say('''คำถามทบทวนข้อ 2 — แนวคำตอบ: เหมาะกับข้อมูลที่อยู่บนตารางปกติและมีโครงสร้างเฉพาะที่ที่ซ้ำกันในหลายตำแหน่ง เช่น ภาพกาแล็กซี
แผนที่ AFM ภาพจากแคลอริมิเตอร์ ภาพการเลี้ยวเบน และเมทริกซ์ระยะห่างของพอลิเมอร์ ส่วนอนุกรมเวลาและสเปกตรัม
ใช้คอนโวลูชันหนึ่งมิติได้ ข้อมูลที่ไม่เหมาะคือข้อมูลตารางที่ลำดับของคอลัมน์ไม่มีความหมาย เช่น ลักษณะเด่นสี่ตัว
ของข้อมูลฮิกส์ เพราะไม่มี "เพื่อนบ้าน" ให้ตัวกรองใช้''')

---
# ส่วนที่ 2 ส่วนประกอบของโครงข่ายคอนโวลูชัน (หัวข้อ 11.2)

### ชั้นคอนโวลูชัน
> **นิยาม (ชั้นคอนโวลูชัน)** ชั้นคอนโวลูชันที่มีตัวกรอง $\mathbf K$ ขนาด $k\times k$ รับภาพ $x$ แล้วให้แผนที่ลักษณะเด่น (feature map)
> $$y_{ij} = \sum_{m=0}^{k-1}\sum_{n=0}^{k-1} K_{mn}\,x_{is+m,\; js+n} + b$$
> เมื่อ $s$ คือระยะก้าว (stride) และ $b$ คือไบแอส ขอบของภาพมักเติมศูนย์ไว้ $p$ แถว (padding) ภาพกว้าง $n$ พิกเซลจึงให้แผนที่กว้าง $\lfloor (n + 2p - k)/s\rfloor + 1$ พิกเซล ถ้าภาพเข้ามี $C_{\text{in}}$ ช่องสัญญาณ (channels) เช่น สามแถบสี ตัวกรองแต่ละตัวมีขนาด $C_{\text{in}}\times k\times k$ ชั้นที่มีตัวกรอง $C_{\text{out}}$ ตัวจึงมีพารามิเตอร์ $C_{\text{out}}(C_{\text{in}}k^2 + 1)$ ตัว ซึ่งไม่ขึ้นกับขนาดของภาพเลย

ในทางคณิตศาสตร์ สมการข้างต้นคือสหสัมพันธ์ไขว้ (cross-correlation) ไม่ใช่คอนโวลูชันแท้ซึ่งต้องกลับด้านตัวกรองก่อน แต่เนื่องจากตัวกรองเป็นพารามิเตอร์ที่เรียนรู้ได้ ความต่างนี้จึงไม่มีผลในทางปฏิบัติ ตัวกรองบางแบบที่โครงข่ายเรียนได้ในชั้นแรกมีหน้าตาคุ้นเคยสำหรับนักฟิสิกส์ เช่น ตัวกรองที่คล้ายตัวดำเนินการลาปลาซตรวจจุดสว่าง และตัวกรองที่คล้ายอนุพันธ์ในทิศหนึ่งตรวจขอบ สิ่งที่ต่างจากการประมวลผลภาพแบบดั้งเดิมคือผู้ออกแบบไม่ได้เลือกตัวกรองเอง โครงข่ายเลือกตัวกรองที่ช่วยลดความสูญเสียของงานที่กำลังเรียน

### ✏️ แบบฝึก 11.B (6 นาที) ขนาดของผลลัพธ์และจำนวนพารามิเตอร์
ชั้นคอนโวลูชันรับภาพสามแถบสีและมีตัวกรอง 16 ตัว ใช้สูตรในนิยามคำนวณความกว้างของแผนที่ผลลัพธ์สำหรับสี่กรณีในตาราง แล้วเก็บใน `my_out` ตามลำดับ และคำนวณจำนวนพารามิเตอร์ของกรณีที่สอง ($k = 5$) เก็บใน `my_npar_k5`

| กรณี | ความกว้างภาพ $n$ | ตัวกรอง $k$ | เติมขอบ $p$ | ระยะก้าว $s$ |
|---|---|---|---|---|
| 1 | 64 | 3 | 1 | 1 |
| 2 | 64 | 5 | 0 | 1 |
| 3 | 64 | 3 | 1 | 2 |
| 4 | 69 | 3 | 1 | 1 |

In [ ]:
my_out = [None, None, None, None]   # ✏️ output widths of cases 1-4
my_npar_k5 = None                   # ✏️ parameters of case 2: C_out (C_in k^2 + 1)

In [ ]:
#@markdown ✅ ตรวจคำตอบ (เทียบกับ PyTorch)
_say = _gate(['my_npar_k5', 'my_out'])
for i, (n, k, p, s) in enumerate([(64, 3, 1, 1), (64, 5, 0, 1), (64, 3, 1, 2), (69, 3, 1, 1)]):
    c_ = nn.Conv2d(3, 16, k, stride=s, padding=p)
    w_ = c_(torch.zeros(1, 3, n, n)).shape[-1]
    mine = my_out[i]
    _say(f"case {i + 1} (n={n}, k={k}, p={p}, s={s}):", "(ยังไม่ได้ตอบ)" if mine is None else ("✔" if mine == w_ else "✘"),
          f"— PyTorch gives width {w_}")
_ref = sum(q.numel() for q in nn.Conv2d(3, 16, 5).parameters())
_say("parameters, k = 5:", "(ยังไม่ได้ตอบ)" if my_npar_k5 is None else ("✔" if my_npar_k5 == _ref else "✘"), f"— {_ref:,}")

In [ ]:
#@title แนวคำตอบ 11.B
_say = _gate(['my_npar_k5', 'my_out'])
_say('''แนวคำตอบ 11.B
กรณี 1: (64 + 2 - 3)/1 + 1 = 64  เติมขอบ 1 แถวกับตัวกรอง 3 x 3 รักษาขนาดภาพไว้
กรณี 2: (64 + 0 - 5)/1 + 1 = 60  ไม่เติมขอบ ภาพจึงหดลง k - 1 = 4 พิกเซล
กรณี 3: floor((64 + 2 - 3)/2) + 1 = 32  ระยะก้าว 2 ลดขนาดลงครึ่งหนึ่ง
กรณี 4: (69 + 2 - 3)/1 + 1 = 69
จำนวนพารามิเตอร์ของกรณี 2: 16 x (3 x 25 + 1) = 1,216 ตัว ไม่ว่าภาพจะกว้าง 64 หรือ 1,000 พิกเซล''')

### แผนที่ลักษณะเด่นและขอบเขตการรับรู้
ชั้นคอนโวลูชันหนึ่งชั้นมีตัวกรองหลายตัว ตัวกรองแต่ละตัวให้แผนที่ลักษณะเด่นหนึ่งแผ่น และแผ่นเหล่านั้นซ้อนกันเป็นช่องสัญญาณของภาพเข้าสำหรับชั้นถัดไป เมื่อซ้อนชั้นคอนโวลูชันหลายชั้นสลับกับการลดขนาด หน่วยในชั้นลึกจะ "มองเห็น" บริเวณของภาพต้นฉบับกว้างขึ้นเรื่อย ๆ บริเวณนี้เรียกว่าขอบเขตการรับรู้ (receptive field) งานศึกษาจำนวนมากพบว่าชั้นต้น ๆ เรียนรูปแบบง่าย เช่น ขอบและจุด ชั้นกลางเรียนพื้นผิวและส่วนโค้ง และชั้นลึกเรียนรูปแบบที่ซับซ้อนขึ้น ภาพนี้เป็นแนวโน้มทั่วไป ไม่ใช่กฎที่รับประกัน

### การรวม: การลดขนาดแบบหยาบลง
ชั้นรวม (pooling layer) ลดขนาดของแผนที่ลักษณะเด่น การรวมแบบสูงสุด (max pooling) ขนาด $2\times2$ เก็บค่าสูงสุดของทุกหน้าต่าง $2\times2$ ภาพจึงเล็กลงครึ่งหนึ่งในแต่ละด้าน ความหมายของมันคือ "รูปแบบนี้ปรากฏที่ใดที่หนึ่งในบริเวณนี้หรือไม่" โดยไม่สนตำแหน่งแน่นอน ส่วนการรวมแบบเฉลี่ย (average pooling) เก็บค่าเฉลี่ยแทน นักฟิสิกส์อาจนึกถึงการแปลงแบบบล็อกสปินของคาดานอฟในการจัดกลุ่มใหม่ (renormalization group) ซึ่งแทนสปินหลายตัวด้วยสปินตัวเดียวแล้วดูพฤติกรรมที่มาตราส่วนใหญ่ขึ้น ความคล้ายนี้มีผู้ศึกษาในเชิงทฤษฎี แต่ควรใช้เป็นอุปมาเท่านั้น การจัดกลุ่มใหม่ในฟิสิกส์มีกฎการแปลงที่กำหนดจากแฮมิลโทเนียน ขณะที่การรวมในโครงข่ายเป็นการลดขนาดแบบตายตัว และสิ่งที่ถูกเก็บไว้ถูกกำหนดโดยตัวกรองที่เรียนมาเพื่องานจำแนก

### สร้างโครงข่ายคอนโวลูชันด้วย PyTorch
โค้ดที่ 11.1 สร้างโครงข่ายคอนโวลูชันสำหรับภาพขาวดำ $64\times64$ ห้าประเภท พร้อมโครงข่ายหลายชั้นสำหรับเปรียบเทียบ ส่วนสกัดลักษณะเด่น (`features`) มีคอนโวลูชันสี่ชั้น แต่ละชั้นตามด้วย ReLU และสามชั้นแรกตามด้วยการรวมแบบสูงสุด ส่วนหัว (`head`) รับค่าเฉลี่ยของแผนที่ลักษณะเด่นทั้ง 64 แผ่นจากการรวมแบบเฉลี่ยทั้งหมด แล้วให้คะแนนของแต่ละประเภท

### โค้ดที่ 11.1 — โครงข่ายคอนโวลูชันและโครงข่ายหลายชั้นสำหรับภาพ 64×64

In [ ]:
import torch
import torch.nn as nn

class GalaxyCNN(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 64 -> 32
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 32 -> 16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 16 -> 8
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU())                   # 64 maps, 8x8
        self.head = nn.Linear(64, n_classes)
    def forward(self, x):
        return self.head(self.features(x).mean(dim=(2, 3)))   # global average pooling

class MLP(nn.Module):                                           # for comparison
    def __init__(self, n_classes=5):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(),
                                 nn.Linear(64 * 64, 128), nn.ReLU(),
                                 nn.Linear(128, 64), nn.ReLU(),
                                 nn.Linear(64, n_classes))
    def forward(self, x):
        return self.net(x)

for Model in (MLP, GalaxyCNN):
    print(Model.__name__, sum(p.numel() for p in Model().parameters()), "parameters")

In [ ]:
cnn_, mlp_ = GalaxyCNN(), MLP()
print("GalaxyCNN parameters per conv layer:", [sum(q.numel() for q in l.parameters()) for l in cnn_.features if isinstance(l, nn.Conv2d)],
      "| head:", sum(q.numel() for q in cnn_.head.parameters()))
first = sum(q.numel() for q in mlp_.net[1].parameters()); total = sum(q.numel() for q in mlp_.parameters())
print(f"MLP first layer: {first:,} of {total:,} parameters ({100 * first / total:.1f} %)")
print(f"ratio MLP / CNN = {total / sum(q.numel() for q in cnn_.parameters()):.2f}")

ผลที่พิมพ์ออกมาบอกความต่างของสองแบบจำลองอย่างชัดเจน โครงข่ายหลายชั้นมีพารามิเตอร์ 532,997 ตัว โดยร้อยละ 98 อยู่ในชั้นแรก ส่วนโครงข่ายคอนโวลูชันมีเพียง 60,549 ตัว น้อยกว่าเกือบเก้าเท่า และจำนวนนี้จะไม่เปลี่ยนเลยถ้าภาพใหญ่ขึ้น ส่วนหัวที่ใช้การรวมแบบเฉลี่ยทั้งหมดมีพารามิเตอร์เพียง $64\times5 + 5 = 325$ ตัว ต่างจากการออกแบบในร่างเดิมหลายแบบที่แผ่แผนที่ลักษณะเด่นเป็นเวกเตอร์ยาวแล้วต่อกับชั้นเชื่อมต่อเต็ม ซึ่งเพิ่มพารามิเตอร์หลายแสนตัวและทิ้งความไม่แปรเปลี่ยนต่อการเลื่อนไปบางส่วน

เซลล์ถัดไปวัดขอบเขตการรับรู้ของ `GalaxyCNN` ด้วยเกรเดียนต์ วิธีคือหาอนุพันธ์ของหน่วยกลางแผนที่ของแต่ละชั้นเทียบกับภาพเข้า แล้วนับความกว้างของบริเวณที่อนุพันธ์ไม่เป็นศูนย์ (ใช้การรวมแบบเฉลี่ยแทนการรวมแบบสูงสุด และตั้งน้ำหนักเป็นบวกทั้งหมด เพื่อให้ทุกพิกเซลในขอบเขตมีอนุพันธ์ไม่เป็นศูนย์ เรขาคณิตของขอบเขตเหมือนเดิม)

In [ ]:
m_ = GalaxyCNN()
f_ = nn.Sequential(*[nn.AvgPool2d(2) if isinstance(l, nn.MaxPool2d) else l for l in m_.features])   # same geometry
for q in f_.parameters():
    nn.init.constant_(q, 0.01) if q.dim() == 1 else nn.init.uniform_(q, 0.5, 1.0)
rf = []
for i, l in enumerate(f_):
    if isinstance(l, (nn.Conv2d, nn.AvgPool2d)):
        x_ = torch.ones(1, 1, 64, 64, requires_grad=True)
        y_ = f_[:i + 2 if isinstance(l, nn.Conv2d) else i + 1](x_)
        c = y_.shape[-1] // 2
        y_[0, 0, c, c].backward()
        rows = (x_.grad[0, 0] != 0).any(1).nonzero()
        rf.append(('conv' if isinstance(l, nn.Conv2d) else 'pool', int(rows.max() - rows.min() + 1)))
print("receptive field (pixels) after each layer:", rf)

ขอบเขตการรับรู้ของผลลัพธ์ของคอนโวลูชันโตจาก 3 พิกเซลในชั้นแรกเป็น 8, 18 และ 38 พิกเซลในชั้นที่สอง สาม และสี่ (หลังการรวมแต่ละครั้งคือ 4, 10 และ 22) หน่วยในชั้นสุดท้ายจึงมองเห็นกาแล็กซีเกือบทั้งดวง ขอบเขตโตเร็วขึ้นหลังการรวมแต่ละครั้ง เพราะตัวกรอง $3\times3$ บนแผนที่ที่หดลงครึ่งหนึ่งครอบคลุมพิกเซลของภาพต้นฉบับมากขึ้นเป็นสองเท่า

In [ ]:
#@title คำถามทบทวนข้อ 1 และ 3 (แนวคำตอบ)
print('''คำถามทบทวนข้อ 1: ความแปรตามการเลื่อนกับความไม่แปรเปลี่ยนต่อการเลื่อนต่างกันอย่างไร และส่วนใดของโครงข่ายให้สมบัติแต่ละอย่าง

คำถามทบทวนข้อ 3: ทำไมจำนวนพารามิเตอร์ของชั้นคอนโวลูชันจึงไม่ขึ้นกับขนาดของภาพ และขอบเขตการรับรู้โตอย่างไรตามความลึก''')
_say = _gate([])
_say('''คำถามทบทวนข้อ 1 — แนวคำตอบ: ความแปรตามคือเมื่อภาพเลื่อน ผลลัพธ์เลื่อนตาม ซึ่งชั้นคอนโวลูชันให้ผ่านการใช้น้ำหนักร่วมกัน ความไม่แปรเปลี่ยน
คือผลลัพธ์ไม่เปลี่ยนเลย ซึ่งได้จากการรวมแบบเฉลี่ยทั้งหมดในตอนท้าย งานจำแนกต้องการความไม่แปรเปลี่ยน ส่วนงานหาตำแหน่ง
ต้องการความแปรตาม

คำถามทบทวนข้อ 3 — แนวคำตอบ: เพราะตัวกรองชุดเดียวกันถูกใช้ทุกตำแหน่ง พารามิเตอร์จึงขึ้นกับขนาดตัวกรองและจำนวนช่องสัญญาณเท่านั้น
ขอบเขตการรับรู้โตขึ้นทุกชั้นคอนโวลูชัน และโตเร็วขึ้นหลังการรวมแต่ละครั้ง ในโครงข่ายของบทนี้คือ 3, 8, 18 และ 38 พิกเซล''')

---
# คาบที่ 2 ฝึกโครงข่ายคอนโวลูชันกับภาพกาแล็กซีจำลอง

> **เริ่มคาบนี้หลังจากปิด Colab ไปแล้ว:** คลิกเซลล์หัวคาบที่ 1 แล้วเลือก Runtime → Run before จากนั้นกลับมารันเซลล์ซ่อนถัดไป ซึ่งนิยาม `GalaxyCNN` และ `MLP` ของโค้ดที่ 11.1 ใหม่ (ถ้าเพิ่งเรียนคาบที่ 1 ต่อเนื่องมา การรันซ้ำไม่มีผลเสีย)

In [ ]:
#@title นิยาม GalaxyCNN และ MLP จากโค้ดที่ 11.1 (ดับเบิลคลิกเพื่อดูโค้ด)
import torch
import torch.nn as nn

class GalaxyCNN(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 64 -> 32
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 32 -> 16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 16 -> 8
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU())                   # 64 maps, 8x8
        self.head = nn.Linear(64, n_classes)
    def forward(self, x):
        return self.head(self.features(x).mean(dim=(2, 3)))   # global average pooling

class MLP(nn.Module):                                           # for comparison
    def __init__(self, n_classes=5):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(),
                                 nn.Linear(64 * 64, 128), nn.ReLU(),
                                 nn.Linear(128, 64), nn.ReLU(),
                                 nn.Linear(64, n_classes))
    def forward(self, x):
        return self.net(x)

print('GalaxyCNN, MLP พร้อมใช้')

### ✏️ แบบฝึก E11.2 ในชั้นเรียน (5 นาที ทบทวนส่วนที่ 2 ต้นคาบ) ออกแบบโครงข่ายสำหรับภาพสีของ Galaxy10
ภาพของ Galaxy10 DECaLS มีสามแถบสี ($g$, $r$, $z$) และสิบประเภท สมมติว่าลดขนาดภาพเหลือ $64\times64$ แล้วดัดแปลง `GalaxyCNN` ให้รับภาพนี้ โดยเปลี่ยนชั้นแรกเป็น `nn.Conv2d(3, 16, 3, padding=1)` และใช้ `n_classes=10` คำนวณด้วยมือ (ใช้สูตร $C_{\text{out}}(C_{\text{in}}k^2 + 1)$) จำนวนพารามิเตอร์ของชั้นแรกเก็บใน `my_first_g10` ของส่วนหัวเก็บใน `my_head_g10` และของทั้งโครงข่ายเก็บใน `my_total_g10` แล้วตรวจกับ PyTorch (ส่วนที่เปรียบเทียบกับแบบในร่างเดิมเป็นงานนอกชั้น)

In [ ]:
my_first_g10 = None   # ✏️ nn.Conv2d(3, 16, 3, padding=1)
my_head_g10 = None    # ✏️ nn.Linear(64, 10)
my_total_g10 = None   # ✏️ whole network

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_first_g10', 'my_head_g10', 'my_total_g10'])
g10_ = GalaxyCNN(n_classes=10)
g10_.features[0] = nn.Conv2d(3, 16, 3, padding=1)
_refs = [("first conv layer", my_first_g10, sum(q.numel() for q in g10_.features[0].parameters())),
         ("head            ", my_head_g10, sum(q.numel() for q in g10_.head.parameters())),
         ("total           ", my_total_g10, sum(q.numel() for q in g10_.parameters()))]
for nm, mine, ref in _refs:
    _say(nm, ":", "(ยังไม่ได้ตอบ)" if mine is None else ("✔" if mine == ref else "✘"), f"— {ref:,}")

In [ ]:
#@title แนวคำตอบ E11.2 (ส่วนในชั้นเรียน)
_say = _gate(['my_first_g10', 'my_head_g10', 'my_total_g10'])
_say('''แนวคำตอบ E11.2 (ส่วนในชั้นเรียน)
ชั้นแรกเพิ่มจาก 16(1 x 9 + 1) = 160 เป็น 16(3 x 9 + 1) = 448 ตัว ส่วนหัวที่มีสิบประเภทมี 64 x 10 + 10 = 650 ตัว
ชั้นอื่นเหมือนเดิม คือ 4,640, 18,496 และ 36,928 ตัว รวมเป็น 61,162 ตัว
ชั้นที่มีพารามิเตอร์มากที่สุดคือคอนโวลูชันชั้นสุดท้าย การเพิ่มแถบสีเพิ่มพารามิเตอร์เพียง 288 ตัว เพราะกระทบเฉพาะชั้นแรก''')

---
# ส่วนที่ 3 ฝึกโครงข่ายคอนโวลูชันกับภาพกาแล็กซีจำลอง (หัวข้อ 11.3)

### ภาพกาแล็กซีจำลองจากแบบจำลองทางดาราศาสตร์
ก่อนใช้ข้อมูลจริงซึ่งมีขนาดใหญ่และฝึกช้า ส่วนนี้ใช้ภาพกาแล็กซีจำลองที่สร้างจากแบบจำลองมาตรฐานของดาราศาสตร์ ข้อดีของภาพจำลองคือเรารู้คำตอบที่ถูกต้องของทุกภาพ รู้ว่าลักษณะใดแยกประเภทใดออกจากกัน และทดลองเปลี่ยนเงื่อนไขได้ตามต้องการ ซึ่งทำให้ตรวจได้ว่าโครงข่ายเรียนสิ่งที่ควรเรียนหรือไม่

ความสว่างพื้นผิวของกาแล็กซีส่วนใหญ่อธิบายได้ดีด้วยโพรไฟล์ของเซอร์ซิก (Sérsic profile)
$$I(r) = I_e \exp\Big\{-b_n\Big[\Big(\frac{r}{r_e}\Big)^{1/n} - 1\Big]\Big\}, \qquad b_n \approx 2n - \tfrac13$$
เมื่อ $r_e$ คือรัศมีที่ครอบแสงครึ่งหนึ่ง และดัชนี $n$ กำหนดรูปร่าง $n = 4$ คือโพรไฟล์ของเดอ วอกูเลอร์ ซึ่งพบในกาแล็กซีรีและส่วนป่องกลาง ความสว่างของมันลดลงช้าที่ขอบนอกจึงมี "ปีก" กว้าง ส่วน $n = 1$ คือจานแบบเอกซ์โพเนนเชียล ซึ่งพบในจานของกาแล็กซีกังหัน

| ประเภท | แบบจำลองในโค้ดที่ 11.2 |
|---|---|
| 0 เรียบกลม (smooth round) | เซอร์ซิก $n = 4$ อัตราส่วนแกน $q$ 0.8–1.0 |
| 1 เรียบรูปซิการ์ (smooth cigar) | เซอร์ซิก $n$ 1–4 อัตราส่วนแกน $q$ 0.2–0.4 |
| 2 จานที่เห็นด้านข้าง (edge-on disk) | จานเอกซ์โพเนนเชียลที่แบนมาก ($q$ 0.08–0.15) กับส่วนป่องเล็ก ๆ |
| 3 กังหัน (spiral) | จานที่มีแขนสองแขนตามเกลียวลอการิทึม $\theta = \ln r/\tan\alpha$ ($\alpha$ คือมุมพิตช์) เอียงด้วย $q = \cos i$ |
| 4 กังหันมีคาน (barred spiral) | เหมือนประเภท 3 และเพิ่มคานรูปยาวที่ใจกลาง |

ทุกภาพหมุนด้วยมุมสุ่ม เบลอด้วยฟังก์ชันการกระจายของจุด (point spread function) แบบเกาส์ซึ่งแทนผลของบรรยากาศ และเติมสัญญาณรบกวนของท้องฟ้า ความสว่างถูกบีบด้วยฟังก์ชัน $\sinh^{-1}$ ซึ่งนักดาราศาสตร์ใช้แสดงทั้งใจกลางที่สว่างมากและขอบที่จางในภาพเดียว

### โค้ดที่ 11.2 — ภาพกาแล็กซีจำลองห้าประเภท

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter
CLASSES = ['smooth round', 'smooth cigar', 'edge-on disk', 'spiral', 'barred spiral']

def make_galaxy(cls, rng, size=64):
    """One noisy 64x64 image of a toy galaxy of class cls (0-4)."""
    y, x = np.mgrid[:size, :size] - (size - 1) / 2
    x0, y0 = rng.normal(0, 1.5, 2)                     # small centring error
    x, y = x - x0, y - y0
    pa = rng.uniform(0, np.pi)                          # position angle
    xr = x * np.cos(pa) + y * np.sin(pa)
    yr = -x * np.sin(pa) + y * np.cos(pa)
    def sersic(re, n, q):
        r = np.sqrt(xr**2 + (yr / q)**2)
        b = 2 * n - 1 / 3
        return np.exp(-b * ((r / re)**(1 / n) - 1))
    if cls == 0:
        img = sersic(rng.uniform(6, 10), 4, rng.uniform(0.8, 1.0))
    elif cls == 1:
        img = sersic(rng.uniform(6, 10), rng.uniform(1, 4), rng.uniform(0.2, 0.4))
    elif cls == 2:
        img = sersic(rng.uniform(10, 15), 1, rng.uniform(0.08, 0.15)) \
              + 0.3 * sersic(2.5, 2, 0.6)
    else:
        q = rng.uniform(0.45, 1.0)                      # cos(inclination)
        re = rng.uniform(10, 15)
        r = np.sqrt(xr**2 + (yr / q)**2)
        theta = np.arctan2(yr / q, xr)
        pitch = rng.uniform(0.25, 0.45)                 # arm winding
        arms = 0.5 * (1 + np.cos(2 * (theta - np.log(r + 1e-3) / np.tan(pitch))))
        arms = arms * (1 - np.exp(-(r / (0.4 * re))**2))  # arms start outside the centre
        disk = np.exp(-r / (re / 1.68)) * (0.4 + 1.2 * arms**2)
        img = disk + 1.5 * sersic(2.0, 2, 0.7 + 0.3 * q) / np.exp(11 / 3)   # bulge, rounder than disk
        if cls == 4:                                     # bar along the disk
            bar_len = rng.uniform(0.4, 0.6) * re
            img += 1.2 * np.exp(-(xr / bar_len)**4 - (yr / (0.3 * bar_len))**2)
    img = np.arcsinh(img / 0.3)                        # astronomical asinh stretch
    img = img / img.max() * rng.uniform(0.6, 1.0)
    img = gaussian_filter(img, rng.uniform(0.8, 1.5))   # seeing (PSF)
    img = img + rng.normal(0, 0.05, img.shape)          # sky + read noise
    return img.astype(np.float32)

def make_galaxy_set(n_per_class, seed=0):
    rng = np.random.default_rng(seed)
    X = np.array([make_galaxy(c, rng) for c in range(5) for _ in range(n_per_class)])
    y = np.repeat(np.arange(5), n_per_class)
    p = rng.permutation(len(y))
    return X[p], y[p]

X_train, y_train = make_galaxy_set(1000, seed=1)     # 5000 images
X_val, y_val = make_galaxy_set(250, seed=2)          # 1250 images
X_test, y_test = make_galaxy_set(400, seed=3)        # 2000 images

In [ ]:
np.savez_compressed("outputs/galaxies.npz", X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
                    X_test=X_test, y_test=y_test)
print("train / validation / test:", X_train.shape, X_val.shape, X_test.shape, "| class counts (train):", np.bincount(y_train))
print("saved outputs/galaxies.npz (used again in periods 3-4)")
fig, ax = plt.subplots(3, 5, figsize=(10, 6.3))
for c in range(5):
    for r, i in enumerate(np.where(y_test == c)[0][:3]):
        ax[r, c].imshow(X_test[i], cmap='gray', origin='lower'); ax[r, c].axis('off')
    ax[0, c].set_title(CLASSES[c], fontsize=10)
fig.suptitle("toy galaxy images (SYNTHETIC)", fontsize=10)
fig.tight_layout(); plt.show()

ขอให้สังเกตว่านี่เป็นแบบจำลองของเล่น ในเครื่องมือจริง การเบลอจากบรรยากาศเกิดกับความสว่างเชิงเส้นก่อนการบีบใด ๆ ส่วนโค้งนี้บีบก่อนเบลอเพื่อให้ภาพขนาดเล็กเห็นโครงสร้างชัด นอกจากนี้ยังไม่มีดาวฤกษ์ข้างเคียง กาแล็กซีพื้นหลัง รังสีคอสมิก หรือความต่างของแถบสี ข้อจำกัดเหล่านี้ไม่กระทบจุดประสงค์ของส่วนนี้ ซึ่งคือการเข้าใจกลไกของโครงข่าย แต่หมายความว่าตัวเลขความแม่นยำในส่วนนี้ใช้เทียบกับงานจริงไม่ได้

### การฝึกและการเฝ้าดูข้อมูลตรวจสอบ
วงรอบการฝึกในโค้ดที่ 11.3 เหมือนกับบทที่ 10 ทุกประการ สิ่งที่เพิ่มเข้ามามีสองอย่าง อย่างแรกคือฟังก์ชัน `fit` บันทึกความแม่นยำกับข้อมูลตรวจสอบทุกเอพอก เพื่อดูพลวัตของการฝึก อย่างที่สองคือทางเลือกการเพิ่มข้อมูล ซึ่งจะใช้ในตอนท้ายของส่วนนี้ เซลล์ถัดไปฝึกโครงข่ายหลายชั้น 15 เอพอกและโครงข่ายคอนโวลูชัน 30 เอพอก (⏱ ราวสิบนาทีบน Colab) ระหว่างรอให้อ่านคำอธิบายถัดไป

### โค้ดที่ 11.3 — การฝึกโครงข่ายพร้อมการเพิ่มข้อมูลและการเฝ้าดูข้อมูลตรวจสอบ

In [ ]:
# ⏱ ราวสิบนาทีบน Colab
_t0 = time.time()
def augment(xb):
    """Random 90-degree rotation and mirror flip: exact symmetries of the pixel grid."""
    xb = torch.rot90(xb, np.random.randint(4), dims=(2, 3))
    if np.random.rand() < 0.5:
        xb = torch.flip(xb, dims=(3,))
    return xb

def fit(model, X, y, X_val, y_val, epochs=30, lr=2e-3, aug=False, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(X[:, None]), torch.tensor(y)),
        batch_size=64, shuffle=True)
    Xv = torch.tensor(X_val[:, None])
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    val_acc = []
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            if aug:
                xb = augment(xb)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val_acc.append(np.mean(model(Xv).argmax(1).numpy() == y_val))
    return val_acc

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(torch.tensor(X[:, None])), dim=1).numpy()

models = {}
for name, Model, lr, epochs in [('MLP', MLP, 1e-3, 15), ('CNN', GalaxyCNN, 2e-3, 30)]:
    torch.manual_seed(0)
    models[name] = Model()
    fit(models[name], X_train, y_train, X_val, y_val, epochs=epochs, lr=lr)

print(f"MLP and CNN trained in {time.time() - _t0:.0f} s")

In [ ]:
for name in models:
    torch.save(models[name].state_dict(), f"outputs/{name}.pt")
print("saved outputs/MLP.pt, outputs/CNN.pt (used again in periods 3-4)")

การฝึกโครงข่ายคอนโวลูชันใช้เวลานานกว่าโครงข่ายหลายชั้นหลายสิบเท่า ทั้งที่มีพารามิเตอร์น้อยกว่ามาก เพราะการคำนวณคอนโวลูชันบนแผนที่ขนาดใหญ่ในชั้นต้น ๆ แพงกว่าการคูณเมทริกซ์ธรรมดา จำนวนพารามิเตอร์จึงไม่ใช่ตัวบอกเวลาฝึก

### ผลลัพธ์และเมทริกซ์ความสับสน
โค้ดที่ 11.4 ประเมินทั้งสองโครงข่ายด้วยภาพทดสอบ 2,000 ภาพ และด้วยภาพชุดเดียวกันที่เลื่อนกาแล็กซีออกจากกลางภาพแบบสุ่มไม่เกิน 12 พิกเซลในแต่ละแกน

### โค้ดที่ 11.4 — ความแม่นยำ ภาพที่เลื่อน และเมทริกซ์ความสับสน

In [ ]:
from sklearn.metrics import confusion_matrix

rng = np.random.default_rng(5)
X_shift = np.array([np.roll(img, tuple(rng.integers(-12, 13, 2)), axis=(0, 1))
                    for img in X_test])                  # galaxies moved off-centre
for name, model in models.items():
    pred = predict(model, X_test).argmax(1)
    acc_shift = np.mean(predict(model, X_shift).argmax(1) == y_test)
    print(f"{name}: test {np.mean(pred == y_test):.3f}, shifted {acc_shift:.3f}")
    print(confusion_matrix(y_test, pred))

ในตำรา (เมล็ดสุ่ม 0) โครงข่ายคอนโวลูชันได้ความแม่นยำร้อยละ 95.9 กับภาพทดสอบ ส่วนโครงข่ายหลายชั้นได้ร้อยละ 92.8 ทั้งที่มีพารามิเตอร์มากกว่าเกือบเก้าเท่า ความต่างที่ชัดกว่านั้นมากอยู่ที่ภาพที่เลื่อนออกจากกลาง โครงข่ายหลายชั้นเหลือเพียงร้อยละ 39.3 เพราะน้ำหนักของมันผูกกับตำแหน่งของพิกเซล ขณะที่โครงข่ายคอนโวลูชันได้ร้อยละ 96.5 ไม่ลดลงเลย ผลนี้คือความไม่แปรเปลี่ยนต่อการเลื่อนในส่วนที่ 1 ที่เห็นเป็นตัวเลข

ตัวเลขทั้งหมดนี้มาจากการฝึกครั้งเดียว เมื่อฝึกซ้ำด้วยเมล็ดสุ่ม 0 ถึง 4 โครงข่ายคอนโวลูชันได้ร้อยละ 95.3 ถึง 99.0 กับภาพปกติ และ 91.2 ถึง 99.1 กับภาพที่เลื่อน ส่วนโครงข่ายหลายชั้นได้ร้อยละ 92.0 ถึง 93.0 และ 37.2 ถึง 39.6 ความได้เปรียบกับภาพที่อยู่กลางภาพจึงมีจริงแต่ใกล้กับความแกว่งระหว่างการฝึก และความไม่แปรเปลี่ยนก็ไม่สมบูรณ์ บางการฝึกเสียไปราวห้าจุดเพราะผลของขอบภาพและการรวม แต่ความต่างจากโครงข่ายหลายชั้นซึ่งเสียไปกว่าครึ่งในทุกการฝึกนั้นชัดเจนเสมอ **ตัวเลขบน Colab จึงอาจต่างจากตำราหลายจุดร้อยละ ให้เทียบกับช่วงข้างต้น ไม่ใช่กับตัวเลขเดี่ยว** และอาจต่ำกว่าช่วงนี้ได้ด้วย เมื่อผู้เขียนทดสอบบน Colab โครงข่ายคอนโวลูชันได้ร้อยละ 90.1 กับภาพปกติ และ 91.8 กับภาพที่เลื่อน โดยความผิดพลาดหลักย้ายไปที่กังหันมีคานถูกทายเป็นกังหัน 111 ภาพ แต่ข้อสรุปหลักยังเหมือนเดิม คือโครงข่ายคอนโวลูชันแทบไม่เสียความแม่นยำเมื่อภาพเลื่อน ขณะที่โครงข่ายหลายชั้นเสียไปกว่าครึ่ง

ในตำรา เมทริกซ์ความสับสนของโครงข่ายคอนโวลูชัน (แถวคือประเภทจริง หลักคือประเภทที่ทำนาย) มีความผิดพลาดส่วนใหญ่อยู่ที่แถวเดียว คือกาแล็กซีเรียบรูปซิการ์ ซึ่งถูกทายเป็นจานที่เห็นด้านข้าง 29 ภาพ และเป็นกังหันมีคาน 35 ภาพ ความสับสนทั้งสองมีเหตุผลทางฟิสิกส์ของภาพ ซิการ์ที่แบนมาก ($q \approx 0.2$) มีหน้าตาใกล้กับจานที่เห็นด้านข้าง ส่วนคานของกาแล็กซีกังหันเป็นโครงสร้างรูปยาวเรียบที่ใจกลาง ซึ่งเมื่อแขนกังหันจางลงในสัญญาณรบกวน ส่วนที่เหลือก็คล้ายซิการ์ เมื่อฝึกด้วยเมล็ดสุ่มอื่น ช่องที่ผิดมากที่สุดของโครงข่ายคอนโวลูชันย้ายไปมาในสามคู่ คือซิการ์กับจานที่เห็นด้านข้าง ซิการ์กับกังหันมีคาน และกังหันมีคานกับกังหัน ส่วนโครงข่ายหลายชั้นผิดมากที่สุดที่จานที่เห็นด้านข้างซึ่งถูกทายเป็นซิการ์ทุกครั้ง การอ่านเมทริกซ์ความสับสนแบบนี้ ซึ่งถามว่าความผิดพลาดแต่ละช่องมีเหตุผลทางฟิสิกส์หรือไม่ ช่วยแยกความกำกวมที่มีอยู่จริงในข้อมูลออกจากความบกพร่องของแบบจำลอง

### ✏️ แบบฝึก 11.C (6 นาที) อ่านเมทริกซ์ความสับสนของเครื่องที่ใช้อยู่
จากเมทริกซ์ความสับสนของ **CNN** ที่โค้ดที่ 11.4 พิมพ์ออกมาบนเครื่องของตนเอง (ลำดับประเภทคือ `CLASSES`)
1. หาช่องนอกเส้นทแยงที่มีค่ามากที่สุด เก็บเป็นทูเพิล `(ประเภทจริง, ประเภทที่ทำนาย)` ใน `my_top_pair` เช่น `(1, 4)`
2. คำนวณค่าการเรียกคืน (recall) ของประเภทจริงในข้อ 1 เก็บใน `my_recall`

แล้วพิจารณาว่าช่องนั้นเป็นหนึ่งในสามคู่ที่ตำราอธิบายด้วยฟิสิกส์ของภาพหรือไม่

In [ ]:
my_top_pair = None   # ✏️ (true class, predicted class) of the largest off-diagonal cell, CNN
my_recall = None     # ✏️ recall of that true class

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_recall', 'my_top_pair'])
_C = confusion_matrix(y_test, predict(models['CNN'], X_test).argmax(1))
_off = _C.copy(); np.fill_diagonal(_off, -1)
_ij = tuple(int(v) for v in np.unravel_index(_off.argmax(), _off.shape))
_rec = _C[_ij[0], _ij[0]] / _C[_ij[0]].sum()
_say("largest off-diagonal cell:", "(ยังไม่ได้ตอบ)" if my_top_pair is None else ("✔" if tuple(my_top_pair) == _ij else "✘"),
      f"— {CLASSES[_ij[0]]} -> {CLASSES[_ij[1]]}: {_C[_ij]} images")
_say("recall of that class     :", "(ยังไม่ได้ตอบ)" if my_recall is None else ("✔" if abs(my_recall - _rec) < 1e-3 else "✘"),
      f"— {_rec:.3f}")
_pairs = [{1, 2}, {1, 4}, {3, 4}]
_say("one of the three physical pairs of the book (cigar/edge-on, cigar/barred, barred/spiral):", set(_ij) in _pairs)

In [ ]:
#@title แนวคำตอบ 11.C
_say = _gate(['my_recall', 'my_top_pair'])
_say('''แนวคำตอบ 11.C
คำตอบขึ้นกับการฝึกบนเครื่องที่ใช้ ในตำรา (เมล็ดสุ่ม 0) ช่องที่มากที่สุดคือซิการ์ถูกทายเป็นกังหันมีคาน 35 ภาพ
ค่าการเรียกคืนของซิการ์คือ 332/400 = 0.83 บนเครื่องอื่นช่องที่มากที่สุดอาจเป็นกังหันมีคานถูกทายเป็นซิการ์
หรือเป็นกังหัน แต่มักอยู่ในสามคู่ที่มีเหตุผลทางฟิสิกส์ของภาพ ได้แก่ ซิการ์กับจานที่เห็นด้านข้าง (รูปยาวแบนทั้งคู่)
ซิการ์กับกังหันมีคาน (คานเรียบรูปยาวเมื่อแขนจาง) และกังหันมีคานกับกังหัน (คานสั้นหรือจางจนมองไม่เห็น)''')

### การเพิ่มข้อมูลที่สอดคล้องกับความสมมาตรทางฟิสิกส์
เมื่อข้อมูลฝึกมีน้อย การเพิ่มข้อมูล (data augmentation) เป็นวิธีปรับค่าเกินที่ได้ผลมากสำหรับภาพ ความคิดคือสร้างภาพฝึกใหม่จากภาพเดิมด้วยการแปลงที่ไม่เปลี่ยนคำตอบ สำหรับกาแล็กซี การหมุนภาพไม่เปลี่ยนชนิดของกาแล็กซี เพราะทิศของกาแล็กซีบนท้องฟ้าเป็นเรื่องบังเอิญของตำแหน่งผู้สังเกต การสะท้อนภาพก็ไม่เปลี่ยนชนิด แม้จะเปลี่ยนทิศการหมุนของแขนกังหัน ซึ่งไม่ใช่เกณฑ์ของการจำแนก ฟังก์ชัน `augment` ในโค้ดที่ 11.3 ใช้การหมุนทีละ 90 องศาและการสะท้อน ซึ่งเป็นความสมมาตรแม่นตรงของตารางพิกเซลสี่เหลี่ยม จึงไม่ต้องประมาณค่าพิกเซลใหม่

หลักสำคัญคือ **การเพิ่มข้อมูลต้องสอดคล้องกับความสมมาตรของระบบที่วัด** ไม่ใช่กับความสมมาตรของภาพโดยทั่วไป

| ข้อมูล | การแปลงที่ไม่ควรใช้ | เหตุผล |
|---|---|---|
| ภาพจากแคลอริมิเตอร์ทรงกระบอก | สะท้อนในแนวที่กลับทิศของสนามแม่เหล็ก | กลับเครื่องหมายของประจุที่อนุมานจากความโค้งของรอย (หมุนรอบแกนลำอนุภาคได้) |
| ภาพเวลา–ความถี่ของคลื่นความโน้มถ่วง | หมุนภาพหรือกลับแกนเวลา | แกนทั้งสองมีความหมายทางฟิสิกส์ต่างกัน |
| สเปกตรัมที่แสดงเป็นภาพ | เลื่อนตามแกนความยาวคลื่น | การเลื่อนคือการเปลี่ยนเรดชิฟต์ ซึ่งอาจเป็นคำตอบที่ต้องการหาเสียเอง |
| ภาพทางฟิสิกส์ทั่วไป | ปรับความสว่างแบบสุ่ม | ความสว่างมักเป็นปริมาณที่วัดได้และมีความหมาย |

โค้ดที่ 11.5 ทดสอบผลของการเพิ่มข้อมูลเมื่อมีภาพฝึกเพียง 100 ภาพต่อประเภท โดยฝึก 60 เอพอกทั้งแบบไม่เพิ่มและเพิ่มข้อมูล (⏱ ราวสามถึงห้านาทีบน Colab)

### โค้ดที่ 11.5 — ผลของการเพิ่มข้อมูลเมื่อภาพฝึกมีน้อย

In [ ]:
# ⏱ ราวสามถึงห้านาทีบน Colab
small = np.concatenate([np.where(y_train == c)[0][:100] for c in range(5)])   # 100 per class
for aug in (False, True):
    torch.manual_seed(0)
    model = GalaxyCNN()
    val_acc = fit(model, X_train[small], y_train[small], X_val, y_val, epochs=60, aug=aug)
    test_acc = np.mean(predict(model, X_test).argmax(1) == y_test)
    print(f"augmentation={aug}: best val {max(val_acc):.3f}, "
          f"last val {val_acc[-1]:.3f}, test {test_acc:.3f}")

ในตำรา เมื่อไม่เพิ่มข้อมูล ความแม่นยำกับข้อมูลทดสอบคือร้อยละ 73.7 และเมื่อเพิ่มข้อมูลคือร้อยละ 77.1 ห่างจากผลที่ได้ด้วยภาพฝึกครบ 1,000 ภาพต่อประเภทมาก ข้อมูลที่หายไปเก้าในสิบส่วนจึงชดเชยด้วยการเพิ่มข้อมูลได้เพียงบางส่วน ประโยชน์ที่ได้ในตัวอย่างนี้ไม่มากนัก เพราะภาพจำลองมีมุมการหมุนสุ่มอยู่แล้ว โครงข่ายจึงเห็นกาแล็กซีหลายทิศแม้ไม่เพิ่มข้อมูล การเพิ่มข้อมูลจะมีผลมากกว่านี้เมื่อข้อมูลฝึกมีทิศที่ลำเอียง เช่น ภาพจากการสำรวจที่มักจัดให้แกนยาวของกาแล็กซีอยู่แนวนอน

ตัวเลขอีกคู่หนึ่งที่โค้ดพิมพ์ออกมาสำคัญไม่แพ้กัน ความแม่นยำกับข้อมูลตรวจสอบแกว่งมากระหว่างเอพอก ในตำรา ค่าสูงสุดตลอด 60 เอพอกคือร้อยละ 81.4 และ 82.9 แต่ค่าที่เอพอกสุดท้ายคือร้อยละ 74.9 และ 77.8

### ✏️ แบบฝึก 11.D (4 นาที) ขนาดของอคติจากการเลือกเอพอก
หลังโค้ดที่ 11.5 ตัวแปร `val_acc` (ความแม่นยำกับข้อมูลตรวจสอบทุกเอพอก) และ `test_acc` เป็นของการฝึกรอบสุดท้าย (เพิ่มข้อมูล) คำนวณผลต่างระหว่างความแม่นยำสูงสุดกับข้อมูลตรวจสอบและความแม่นยำกับข้อมูลทดสอบ เก็บใน `my_gap` และหาเอพอกที่ให้ค่าสูงสุด (นับจาก 1) เก็บใน `my_best_epoch` แล้ววาดกราฟ `val_acc` เทียบกับเอพอก พร้อมเส้นแนวนอนที่ `test_acc`

In [ ]:
my_gap = None          # ✏️ max(val_acc) - test_acc
my_best_epoch = None   # ✏️ epoch (1-60) of the best validation accuracy
# ✏️ plot val_acc vs epoch and a horizontal line at test_acc

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_best_epoch', 'my_gap'])
_gap = max(val_acc) - test_acc; _be = int(np.argmax(val_acc)) + 1
_say("gap best validation - test:", "(ยังไม่ได้ตอบ)" if my_gap is None else ("✔" if abs(my_gap - _gap) < 1e-6 else "✘"),
      f"— {100 * _gap:.1f} percentage points")
_say("best epoch               :", "(ยังไม่ได้ตอบ)" if my_best_epoch is None else ("✔" if my_best_epoch == _be else "✘"), f"— {_be}")

In [ ]:
#@title แนวคำตอบ 11.D
_say = _gate(['my_best_epoch', 'my_gap'])
_say('''แนวคำตอบ 11.D
my_gap = max(val_acc) - test_acc ;  my_best_epoch = int(np.argmax(val_acc)) + 1
plt.plot(np.arange(1, 61), val_acc); plt.axhline(test_acc, ls='--')
ในตำรา ผลต่างคือ 82.9 - 77.1 = 5.8 จุดร้อยละ (และ 81.4 - 73.7 = 7.7 จุดเมื่อไม่เพิ่มข้อมูล)
ถ้ารายงานค่าสูงสุดกับข้อมูลตรวจสอบเป็นผลของแบบจำลอง ตัวเลขจะสูงกว่าความจริงราวห้าถึงแปดจุดร้อยละ
กราฟแสดงว่าค่าสูงสุดเป็นยอดแหลมหนึ่งจุดท่ามกลางความแกว่ง ไม่ใช่ระดับที่แบบจำลองรักษาไว้ได้''')

> **ความเข้าใจคลาดเคลื่อนที่พบบ่อย:** "เลือกเอพอกที่ความแม่นยำกับข้อมูลตรวจสอบสูงที่สุด แล้วรายงานตัวเลขนั้นเป็นผลของแบบจำลอง" ความแม่นยำกับข้อมูลตรวจสอบแกว่งจากเอพอกหนึ่งไปอีกเอพอกหนึ่งตามความบังเอิญของข้อมูลชุดนั้น เมื่อเลือกเอพอกที่ดีที่สุดจากหลายสิบหรือหลายร้อยเอพอก ค่าที่เลือกได้มีอคติเข้าข้างตัวเอง ด้วยเหตุผลเดียวกับการเลือกไฮเพอร์พารามิเตอร์ในบทที่ 9 ตัวเลขที่รายงานต้องมาจากข้อมูลทดสอบที่ไม่ได้ใช้ในการเลือกใด ๆ เลย ตัวอย่างเมล็ดข้าวในส่วนที่ 5 จะกลับมาที่ประเด็นนี้อีกครั้ง

### ✏️ แบบฝึก E11.3 ในชั้นเรียน (10 นาที) หาข้อผิดพลาดในการเตรียมข้อมูลภาพ
นักศึกษาคนหนึ่งเตรียมข้อมูลสำหรับฝึกโครงข่ายคอนโวลูชันดังนี้

1. คำนวณค่าเฉลี่ยและส่วนเบี่ยงเบนมาตรฐานของความสว่างจากภาพ **ทั้งหมด** แล้วใช้ปรับมาตรฐานทุกภาพ
2. แบ่งข้อมูลเป็นชุดฝึกและชุดทดสอบ (test set) โดยภาพของกาแล็กซีดวงเดียวกันที่ถ่ายจากสองการสำรวจอาจตกอยู่คนละชุด
3. ใช้การเพิ่มข้อมูลแบบหมุนสุ่มและปรับความสว่างสุ่มร้อยละ 30 กับทั้งชุดฝึกและชุดทดสอบ
4. ใช้การตั้งค่าเดียวกันนี้กับงานจำแนกภาพเวลา–ความถี่ของสัญญาณจากเครื่องตรวจวัดคลื่นความโน้มถ่วง

ในชั้นเรียน ให้ตอบสองข้อ (1) ขั้นใดเป็นการรั่วไหลของข้อมูลที่รุนแรงที่สุด เก็บเลขขั้นใน `my_worst_leak` (2) ขั้นใดผิดโดยหลักการทางฟิสิกส์ เพราะใช้การแปลงที่ขัดกับความหมายของแกนของข้อมูล เก็บเลขขั้นใน `my_unphysical` (การเขียนปัญหา ผลกระทบ และวิธีแก้ของทุกขั้นเป็นงานนอกชั้น)

In [ ]:
my_worst_leak = None   # ✏️ step number 1-4
my_unphysical = None   # ✏️ step number 1-4

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_unphysical', 'my_worst_leak'])
for nm, mine, ok in [("most severe leak ", my_worst_leak, 2), ("wrong in principle", my_unphysical, 4)]:
    _say(nm, ":", "(ยังไม่ได้ตอบ)" if mine is None else ("✔" if mine == ok else "✘"))

In [ ]:
#@title แนวคำตอบ E11.3 (ส่วนในชั้นเรียน)
_say = _gate(['my_unphysical', 'my_worst_leak'])
_say('''แนวคำตอบ E11.3 (ส่วนในชั้นเรียน)
(1) ขั้นที่ 2: โครงข่ายอาจจำกาแล็กซีดวงนั้นได้จากรายละเอียดเฉพาะตัว ผลกับชุดทดสอบจึงดีเกินจริง
    วิธีแก้คือแบ่งตามวัตถุ ไม่ใช่ตามภาพ (การแบ่งตามกลุ่มในบทที่ 9) ขั้นที่ 1 ก็รั่วไหลเช่นกัน แต่ผลมักเล็ก
(2) ขั้นที่ 4: ภาพเวลา-ความถี่มีแกนที่มีความหมายต่างกัน การหมุนภาพสร้างสัญญาณที่ความถี่เปลี่ยนตามเวลา
    ในแบบที่ไม่มีระบบดาวคู่ใดทำได้ โครงข่ายจะเรียนจากข้อมูลที่ขัดกับฟิสิกส์ โดยอาจไม่มีตัวเลขใดเตือน
ทุกการแปลงที่ใช้เพิ่มข้อมูลคือการอ้างว่า "คำตอบไม่เปลี่ยนภายใต้การแปลงนี้" ซึ่งต้องมีเหตุผลทางฟิสิกส์รองรับ''')

In [ ]:
#@title คำถามทบทวนข้อ 4 (แนวคำตอบ)
print('''คำถามทบทวนข้อ 4: จงยกตัวอย่างการเพิ่มข้อมูลที่ถูกต้องและที่ผิดสำหรับข้อมูลทางฟิสิกส์ พร้อมเหตุผล''')
_say = _gate([])
_say('''คำถามทบทวนข้อ 4 — แนวคำตอบ: การหมุนและสะท้อนภาพกาแล็กซีถูกต้อง เพราะทิศของกาแล็กซีบนท้องฟ้าไม่เกี่ยวกับชนิด การหมุนภาพ
เวลา-ความถี่หรือการเลื่อนสเปกตรัมตามแกนความยาวคลื่นผิด เพราะแกนมีความหมายทางฟิสิกส์ และการเลื่อนสเปกตรัม
คือการเปลี่ยนเรดชิฟต์''')

---
# คาบที่ 3 ข้อมูลจริงและงานวิจัยของผู้เขียน

> **เริ่มคาบนี้หลังจากปิด Colab ไปแล้ว:** คลิกเซลล์หัวคาบที่ 1 แล้วเลือก Runtime → Run before จากนั้นกลับมารันเซลล์ซ่อนถัดไป

เซลล์ซ่อนถัดไปเตรียมสิ่งที่คาบนี้ต้องใช้ ได้แก่ `GalaxyCNN` และ `MLP` (โค้ดที่ 11.1) ภาพกาแล็กซีจำลอง (โค้ดที่ 11.2 อ่านจาก `outputs/galaxies.npz`) ฟังก์ชัน `fit` และ `predict` และโครงข่ายที่ฝึกแล้ว `models['CNN']` (โค้ดที่ 11.3 อ่านจาก `outputs/CNN.pt`) ถ้ายังไม่มีไฟล์ เซลล์จะสร้างภาพและฝึกโครงข่ายใหม่ (⏱ ราวสิบนาที) ดับเบิลคลิกเพื่อดูโค้ดได้

In [ ]:
#@title เตรียมโค้ดที่ 11.1–11.3 และโครงข่ายที่ฝึกแล้ว (ดับเบิลคลิกเพื่อดูโค้ด)
import os, time
import numpy as np
# ---- Listing 11.1: GalaxyCNN and MLP
import torch
import torch.nn as nn

class GalaxyCNN(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 64 -> 32
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 32 -> 16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 16 -> 8
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU())                   # 64 maps, 8x8
        self.head = nn.Linear(64, n_classes)
    def forward(self, x):
        return self.head(self.features(x).mean(dim=(2, 3)))   # global average pooling

class MLP(nn.Module):                                           # for comparison
    def __init__(self, n_classes=5):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(),
                                 nn.Linear(64 * 64, 128), nn.ReLU(),
                                 nn.Linear(128, 64), nn.ReLU(),
                                 nn.Linear(64, n_classes))
    def forward(self, x):
        return self.net(x)

# ---- Listing 11.2: toy galaxies (from outputs/galaxies.npz if saved in period 2)
import numpy as np
from scipy.ndimage import gaussian_filter
CLASSES = ['smooth round', 'smooth cigar', 'edge-on disk', 'spiral', 'barred spiral']

def make_galaxy(cls, rng, size=64):
    """One noisy 64x64 image of a toy galaxy of class cls (0-4)."""
    y, x = np.mgrid[:size, :size] - (size - 1) / 2
    x0, y0 = rng.normal(0, 1.5, 2)                     # small centring error
    x, y = x - x0, y - y0
    pa = rng.uniform(0, np.pi)                          # position angle
    xr = x * np.cos(pa) + y * np.sin(pa)
    yr = -x * np.sin(pa) + y * np.cos(pa)
    def sersic(re, n, q):
        r = np.sqrt(xr**2 + (yr / q)**2)
        b = 2 * n - 1 / 3
        return np.exp(-b * ((r / re)**(1 / n) - 1))
    if cls == 0:
        img = sersic(rng.uniform(6, 10), 4, rng.uniform(0.8, 1.0))
    elif cls == 1:
        img = sersic(rng.uniform(6, 10), rng.uniform(1, 4), rng.uniform(0.2, 0.4))
    elif cls == 2:
        img = sersic(rng.uniform(10, 15), 1, rng.uniform(0.08, 0.15)) \
              + 0.3 * sersic(2.5, 2, 0.6)
    else:
        q = rng.uniform(0.45, 1.0)                      # cos(inclination)
        re = rng.uniform(10, 15)
        r = np.sqrt(xr**2 + (yr / q)**2)
        theta = np.arctan2(yr / q, xr)
        pitch = rng.uniform(0.25, 0.45)                 # arm winding
        arms = 0.5 * (1 + np.cos(2 * (theta - np.log(r + 1e-3) / np.tan(pitch))))
        arms = arms * (1 - np.exp(-(r / (0.4 * re))**2))  # arms start outside the centre
        disk = np.exp(-r / (re / 1.68)) * (0.4 + 1.2 * arms**2)
        img = disk + 1.5 * sersic(2.0, 2, 0.7 + 0.3 * q) / np.exp(11 / 3)   # bulge, rounder than disk
        if cls == 4:                                     # bar along the disk
            bar_len = rng.uniform(0.4, 0.6) * re
            img += 1.2 * np.exp(-(xr / bar_len)**4 - (yr / (0.3 * bar_len))**2)
    img = np.arcsinh(img / 0.3)                        # astronomical asinh stretch
    img = img / img.max() * rng.uniform(0.6, 1.0)
    img = gaussian_filter(img, rng.uniform(0.8, 1.5))   # seeing (PSF)
    img = img + rng.normal(0, 0.05, img.shape)          # sky + read noise
    return img.astype(np.float32)

def make_galaxy_set(n_per_class, seed=0):
    rng = np.random.default_rng(seed)
    X = np.array([make_galaxy(c, rng) for c in range(5) for _ in range(n_per_class)])
    y = np.repeat(np.arange(5), n_per_class)
    p = rng.permutation(len(y))
    return X[p], y[p]

if os.path.exists("outputs/galaxies.npz"):
    _d = np.load("outputs/galaxies.npz")
    X_train, y_train, X_val, y_val, X_test, y_test = (_d[k] for k in ("X_train", "y_train", "X_val", "y_val", "X_test", "y_test"))
else:
    X_train, y_train = make_galaxy_set(1000, seed=1)
    X_val, y_val = make_galaxy_set(250, seed=2)
    X_test, y_test = make_galaxy_set(400, seed=3)
    np.savez_compressed("outputs/galaxies.npz", X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
                        X_test=X_test, y_test=y_test)
print("toy galaxies (SYNTHETIC):", X_train.shape, X_val.shape, X_test.shape)

# ---- Listing 11.3: augment, fit, predict, and the trained MLP and CNN
def augment(xb):
    """Random 90-degree rotation and mirror flip: exact symmetries of the pixel grid."""
    xb = torch.rot90(xb, np.random.randint(4), dims=(2, 3))
    if np.random.rand() < 0.5:
        xb = torch.flip(xb, dims=(3,))
    return xb

def fit(model, X, y, X_val, y_val, epochs=30, lr=2e-3, aug=False, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(X[:, None]), torch.tensor(y)),
        batch_size=64, shuffle=True)
    Xv = torch.tensor(X_val[:, None])
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    val_acc = []
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            if aug:
                xb = augment(xb)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val_acc.append(np.mean(model(Xv).argmax(1).numpy() == y_val))
    return val_acc

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(torch.tensor(X[:, None])), dim=1).numpy()

models = {}
for name, Model, lr, epochs in [('MLP', MLP, 1e-3, 15), ('CNN', GalaxyCNN, 2e-3, 30)]:
    torch.manual_seed(0)
    models[name] = Model()
    if os.path.exists(f"outputs/{name}.pt"):
        models[name].load_state_dict(torch.load(f"outputs/{name}.pt"))
        print(f"loaded outputs/{name}.pt")
    else:
        print(f"outputs/{name}.pt not found: training {name} as in Listing 11.3 ...")
        _t0 = time.time()
        fit(models[name], X_train, y_train, X_val, y_val, epochs=epochs, lr=lr)
        torch.save(models[name].state_dict(), f"outputs/{name}.pt")
        print(f"  trained and saved outputs/{name}.pt ({time.time() - _t0:.0f} s)")
    models[name].eval()

---
# ส่วนที่ 4 ข้อมูลจริง: Galaxy10 DECaLS (หัวข้อ 11.4)

### รูปร่างของกาแล็กซีในเชิงฟิสิกส์
รูปร่างของกาแล็กซีบันทึกประวัติการเกิดและวิวัฒนาการของมัน ลำดับของฮับเบิล (Hubble sequence) แบ่งกาแล็กซีเป็นกาแล็กซีรี กาแล็กซีรูปเลนส์ กาแล็กซีกังหัน และกาแล็กซีกังหันมีคาน กาแล็กซีรีส่วนใหญ่ประกอบด้วยดาวอายุมาก โคจรในทิศทางสุ่ม และมีแก๊สเย็นน้อย กาแล็กซีกังหันมีจานที่หมุนอย่างเป็นระเบียบ มีแก๊สและการเกิดดาวใหม่ในแขน ส่วนคานเป็นโครงสร้างที่เกิดจากความไม่เสถียรของจาน และมีบทบาทในการพาแก๊สเข้าสู่ใจกลาง กาแล็กซีที่ถูกรบกวนหรือกำลังรวมตัวกันบอกว่าอันตรกิริยาโน้มถ่วงกับกาแล็กซีข้างเคียงเกิดขึ้นเมื่อไม่นานมานี้

รูปร่างที่เห็นในภาพไม่ได้ขึ้นกับตัวกาแล็กซีเพียงอย่างเดียว กาแล็กซีกังหันที่เห็นด้านข้างดูเป็นเส้นบาง ๆ ซึ่งแยกจากกาแล็กซีรีรูปซิการ์ได้ยาก คานที่ชี้เข้าหาผู้สังเกตมองไม่เห็น และแขนกังหันที่จางจะหายไปถ้ากาแล็กซีอยู่ไกลหรือภาพมีสัญญาณรบกวนมาก ความสับสนบางส่วนของตัวจำแนกจึงเป็นความกำกวมที่มีอยู่จริงในข้อมูล ไม่ใช่ความบกพร่องของตัวจำแนก

### ชุดข้อมูล Galaxy10 DECaLS
Galaxy10 DECaLS เป็นชุดข้อมูลภาพสีของกาแล็กซี 17,736 ภาพ ขนาด $256\times256$ พิกเซล สามแถบสี คือ $g$, $r$ และ $z$ จากการสำรวจ DECaLS ของโครงการ DESI Legacy Imaging Surveys ป้ายกำกับสิบประเภทได้จากการลงคะแนนของอาสาสมัครในโครงการ Galaxy Zoo DECaLS โดยคัดเฉพาะกาแล็กซีที่อาสาสมัครเห็นตรงกันมากพอ จำนวนภาพของแต่ละประเภทไม่เท่ากัน ตั้งแต่ 334 ภาพสำหรับกาแล็กซีเรียบรูปซิการ์ ไปจนถึง 2,645 ภาพสำหรับกาแล็กซีเรียบกลม ชุดข้อมูลเผยแพร่บน Zenodo ภายใต้สัญญาอนุญาตครีเอทีฟคอมมอนส์แบบแสดงที่มา 4.0 (CC BY 4.0) มีไฟล์สองรุ่น ขนาดรุ่นละราว 2.7 GB รุ่น `Galaxy10_DECals_NoDuplicated.h5` ตัดภาพที่ซ้ำกันออก การใช้รุ่นที่ยังมีภาพซ้ำเสี่ยงต่อการที่ภาพเดียวกันตกอยู่ทั้งในชุดฝึกและชุดทดสอบ ซึ่งเป็นการรั่วไหลแบบเดียวกับขั้นที่ 2 ของแบบฝึก E11.3 ชุดข้อมูลนี้มีรุ่นก่อนหน้าคือ Galaxy10 SDSS ซึ่งมีภาพ $69\times69$ พิกเซล ผู้ที่พบตัวเลข $69\times69$ ในเอกสารอื่นควรตรวจว่าหมายถึงรุ่นใด

โค้ดที่ 11.6 อ่านไฟล์ข้อมูลและลดขนาดภาพเหลือ $64\times64$ ด้วยการเฉลี่ยทีละบล็อก $4\times4$ พิกเซล โดยอ่านทีละ 1,000 ภาพ เพราะถ้าแปลงภาพทั้งหมดเป็นเลขทศนิยมพร้อมกันจะต้องใช้หน่วยความจำราว 14 GB การลดขนาดทำให้ฝึกบนหน่วยประมวลผลกลางได้ แต่แลกกับรายละเอียดที่หายไป เช่น แขนกังหันที่แคบ

**ในชั้นเรียนใช้ตัวอย่างย่อย** ไฟล์เต็มใหญ่เกินกว่าจะดาวน์โหลดในคาบ เซลล์ถัดไปจึงดาวน์โหลดตัวอย่างย่อยจาก GitHub ของตำรา (ราว 7 MB) ซึ่งมีภาพจริงที่ไม่ดัดแปลง 50 ภาพ (ประเภทละ 5 ภาพแรก) และป้ายกำกับของทั้ง 17,736 ภาพ แล้วบันทึกในโฟลเดอร์ย่อย `outputs/galaxy10_sample/` ด้วยชื่อ `Galaxy10_DECals.h5` เพื่อให้โค้ดที่ 11.6 รันได้โดยไม่ต้องแก้ ผลที่ได้จึงมี 50 ภาพ ไม่ใช่ 17,736 ภาพ ผู้ที่ทำแบบฝึกหัด E11.4 ต้องดาวน์โหลดไฟล์เต็มเอง

In [ ]:
import shutil, urllib.request
os.makedirs("outputs/galaxy10_sample", exist_ok=True)
_dst = "outputs/galaxy10_sample/Galaxy10_DECals.h5"          # the 50-image sample, named as in Listing 11.6
_local = "../ch01_intro/sample/galaxy10_decals_sample.h5"
_url = ("https://raw.githubusercontent.com/busarapa-stack/dsml-for-physicists/main/"
        "ch01_intro/sample/galaxy10_decals_sample.h5")
if not os.path.exists(_dst):
    if os.path.exists(_local):
        shutil.copy(_local, _dst); print("copied the sample from", _local)
    else:
        urllib.request.urlretrieve(_url, _dst); print("downloaded the sample from GitHub")
print(f"{_dst}: {os.path.getsize(_dst) / 1e6:.1f} MB")

### โค้ดที่ 11.6 — การอ่านและลดขนาดข้อมูล Galaxy10 DECaLS
เซลล์ถัดไปย้ายเข้าโฟลเดอร์ `outputs/galaxy10_sample/` รันโค้ดที่ 11.6 แล้วย้ายกลับ

In [ ]:
os.chdir("outputs/galaxy10_sample")                # the 50-image sample is here
import h5py
import torch.nn.functional as F

# 1,000 images at a time: all 17,736 as float32 at once would need about 14 GB
with h5py.File('Galaxy10_DECals.h5', 'r') as f:   # about 2.7 GB, from Zenodo
    labels = f['ans'][:].astype(np.int64)          # (17736,) classes 0-9
    x = torch.empty(len(labels), 3, 64, 64)
    for i in range(0, len(labels), 1000):          # images: (17736, 256, 256, 3) uint8, g r z
        batch = torch.tensor(f['images'][i:i + 1000]).permute(0, 3, 1, 2).float() / 255.0
        x[i:i + 1000] = F.avg_pool2d(batch, kernel_size=4)   # downsample to 64 x 64
print(x.shape, np.bincount(labels))

from sklearn.model_selection import train_test_split
idx_tr, idx_te = train_test_split(np.arange(len(labels)), test_size=0.2,
                                  stratify=labels, random_state=42)
# GalaxyCNN needs nn.Conv2d(3, 16, ...) as its first layer and n_classes=10
os.chdir("../..")

In [ ]:
import h5py
G10_NAMES = ['disturbed', 'merging', 'round smooth', 'in-between round smooth', 'cigar-shaped smooth',
             'barred spiral', 'unbarred tight spiral', 'unbarred loose spiral', 'edge-on without bulge',
             'edge-on with bulge']
with h5py.File("outputs/galaxy10_sample/Galaxy10_DECals.h5", "r") as f:
    ans_full = f["ans_full"][:].astype(np.int64)          # labels of all 17,736 images of the full file
counts_full = np.bincount(ans_full, minlength=10)
for n_, c_ in zip(G10_NAMES, counts_full):
    print(f"  {n_:25s} {c_:5d}")
print(f"full file: {len(ans_full):,} images (sample in this notebook: {len(labels)} images, {x.shape})")

fig, ax = plt.subplots(2, 5, figsize=(10, 4.6))
for c in range(10):
    i = np.where(labels == c)[0][0]
    a_ = ax.flat[c]
    a_.imshow(np.clip(x[i].permute(1, 2, 0).numpy() * 2.0, 0, 1), origin="upper")   # 64 x 64, brightened x2 for display
    a_.set_title(G10_NAMES[c], fontsize=8); a_.axis("off")
fig.suptitle("Galaxy10 DECaLS (real data, CC BY 4.0), downsampled to 64 x 64", fontsize=10)
fig.tight_layout(); plt.show()

ภาพที่ลดขนาดเหลือ $64\times64$ ยังเห็นรูปร่างหลักของแต่ละประเภท แต่แขนกังหันที่แคบเริ่มเลือน ซึ่งอาจทำให้แยกกาแล็กซีกังหันแขนแน่นกับแขนหลวมได้ยากขึ้น ภาพจริงยังมีสิ่งที่ภาพจำลองในส่วนที่ 3 ไม่มี ได้แก่ ดาวฤกษ์ข้างเคียง กาแล็กซีพื้นหลัง และความต่างของสีระหว่างส่วนป่องกับจาน

### ✏️ แบบฝึก 11.E (4 นาที) ความไม่สมดุลของประเภท
จาก `counts_full` (จำนวนภาพของแต่ละประเภทในไฟล์เต็ม) คำนวณอัตราส่วนของจำนวนภาพในประเภทที่ใหญ่ที่สุดต่อประเภทที่เล็กที่สุด เก็บใน `my_ratio` และความแม่นยำของตัวจำแนกที่ทายประเภทที่ใหญ่ที่สุดทุกภาพ เก็บใน `my_acc_major` แล้วพิจารณาว่าควรรายงานตัวชี้วัดใดคู่กับความแม่นยำ (บทที่ 6)

In [ ]:
my_ratio = None       # ✏️ largest class / smallest class
my_acc_major = None   # ✏️ accuracy of always predicting the largest class

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_acc_major', 'my_ratio'])
_r = counts_full.max() / counts_full.min(); _a = counts_full.max() / counts_full.sum()
_say("largest / smallest :", "(ยังไม่ได้ตอบ)" if my_ratio is None else ("✔" if abs(my_ratio - _r) < 1e-6 else "✘"),
      f"— {counts_full.max()} / {counts_full.min()} = {_r:.2f}")
_say("majority-class acc.:", "(ยังไม่ได้ตอบ)" if my_acc_major is None else ("✔" if abs(my_acc_major - _a) < 1e-6 else "✘"),
      f"— {_a:.3f}")

In [ ]:
#@title แนวคำตอบ 11.E
_say = _gate(['my_acc_major', 'my_ratio'])
_say('''แนวคำตอบ 11.E
ประเภทที่ใหญ่ที่สุดคือเรียบกลม 2,645 ภาพ ที่เล็กที่สุดคือเรียบรูปซิการ์ 334 ภาพ อัตราส่วนราว 7.9 เท่า
การทายเรียบกลมทุกภาพได้ความแม่นยำ 2,645/17,736 = 0.149 ซึ่งเป็นเส้นฐานที่ต่ำ แต่ค่าการเรียกคืนของประเภทเล็ก
ซ่อนอยู่ในความแม่นยำรวมได้ง่าย ควรรายงานความแม่นยำแบบสมดุล (ค่าเฉลี่ยของค่าการเรียกคืนทุกประเภท)
และเมทริกซ์ความสับสนคู่กับความแม่นยำรวมเสมอ''')

### แบบฝึกหัด E11.4 (โครงงานนอกชั้น)
แบบฝึกหัด E11.4 เป็นโครงงานจำแนกกาแล็กซีด้วยไฟล์เต็มของ Galaxy10 DECaLS (3–4 ชั่วโมง) รายละเอียดอยู่ในส่วนที่ 8

---
# ส่วนที่ 5 งานวิจัยของผู้เขียน: การจำแนกพันธุ์เมล็ดข้าวไทยด้วยโครงข่ายคอนโวลูชัน (หัวข้อ 11.5)

### เทคนิคเดียวกันกับภาพต่างโดเมน
โครงข่ายคอนโวลูชันไม่รู้ว่าภาพที่รับเข้ามาเป็นกาแล็กซี เมล็ดพืช หรือแผนที่ของกล้องจุลทรรศน์ มันรู้เพียงว่าข้อมูลเป็นตารางของตัวเลขที่ความหมายขึ้นกับเพื่อนบ้าน สถาปัตยกรรมเดียวกันจึงใช้ได้กับภาพจากหลายสาขา สิ่งที่เปลี่ยนไปตามโดเมนคือทุกอย่างรอบ ๆ โครงข่าย ได้แก่ การได้มาซึ่งภาพ การทำความสะอาดข้อมูล ความสมมาตรที่ใช้เพิ่มข้อมูลได้ ตัวชี้วัดที่ตรงกับเป้าหมาย และความหมายของความผิดพลาด ส่วนนี้ใช้งานวิจัยของผู้เขียนและคณะเรื่องการจำแนกพันธุ์เมล็ดข้าวไทย (Kanchana et al., 2021) เป็นกรณีศึกษา

### ปัญหา ข้อมูล และการเตรียมภาพ
ข้าวขาวดอกมะลิ 105 เป็นข้าวหอมมะลิไทยที่มีราคาสูง จึงมีแรงจูงใจให้ปลอมปนด้วยข้าวพันธุ์อื่นที่ราคาต่ำกว่า มาตรฐานสินค้าเกษตรของไทยกำหนดให้ข้าวหอมมะลิไทยชนิดข้าวเปลือกมีความบริสุทธิ์ไม่น้อยกว่าร้อยละ 95 การตรวจด้วยลักษณะทางกายภาพและปริมาณแอมิโลสแยกพันธุ์ที่คล้ายกันไม่ได้ ส่วนการตรวจดีเอ็นเอแม่นยำแต่มีค่าใช้จ่ายสูง งานนี้จึงทดลองใช้ภาพถ่ายของเมล็ดข้าวเปลือกสี่พันธุ์ ได้แก่ ขาวดอกมะลิ 105 (KDML105) ปทุมธานี 1 (PTT1) กข23 (RD23) และ กข57 (RD57) ร่วมกับโครงข่ายคอนโวลูชัน

ส่วนที่นักฟิสิกส์จะคุ้นเคยคือการควบคุมการวัด

- ภาพถ่ายด้วยสแกนเนอร์แบบระนาบรุ่นเดียวกันทุกภาพ ซึ่งให้แสง มาตราส่วน และระยะที่คงที่
- เมล็ดถูกจัดเรียงด้วยแผ่นอะคริลิกที่เจาะช่องขนาดใกล้เมล็ด เมล็ดจึงไม่ซ้อนกัน
- แยกภาพเมล็ดแต่ละเมล็ดด้วยการหาขอบแบบแคนนี (Canny edge detection) และการหาเส้นรอบรูป (contour) แล้วตัดภาพที่คะแนนมาตรฐานของพื้นที่เมล็ดเกิน 3 ออก เพื่อคัดเมล็ดที่แตกหักหรือติดกัน ซึ่งคือการตรวจค่าผิดปกติตามหลักในบทที่ 3
- ข้อมูลที่ได้มีภาพฝึก 725 ภาพ และภาพตรวจสอบ 312 ภาพ แต่ละพันธุ์มีภาพฝึก 150 ถึง 223 ภาพ

โครงข่ายที่ใช้มีสิบชั้น ประกอบด้วยชั้นคอนโวลูชันที่ใช้ฟังก์ชันกระตุ้น SELU ชั้นรวมแบบสูงสุด ชั้นแผ่ และชั้นเชื่อมต่อเต็ม ฝึกด้วย Keras และ TensorFlow เป็นเวลา 300 รอบ

### ผลและการอ่านผลอย่างระวัง
ความแม่นยำสูงสุดกับชุดตรวจสอบคือร้อยละ 84 ที่รอบที่ 267 ความแม่นยำรายพันธุ์ ซึ่งคือค่าการเรียกคืน (recall) ของแต่ละพันธุ์ คือร้อยละ 99 สำหรับ กข57 ร้อยละ 94 สำหรับขาวดอกมะลิ 105 ร้อยละ 79 สำหรับปทุมธานี 1 และร้อยละ 65 สำหรับ กข23 ความสับสนเกือบทั้งหมดอยู่ระหว่าง กข23 กับปทุมธานี 1 จากเมล็ด กข23 ในชุดตรวจสอบ 77 เมล็ด มี 14 เมล็ดถูกทายเป็นปทุมธานี 1 และจากเมล็ดปทุมธานี 1 จำนวน 80 เมล็ด มี 12 เมล็ดถูกทายเป็น กข23 ผู้วิจัยอธิบายว่าเมล็ดของสองพันธุ์นี้มีลักษณะทางกายภาพคล้ายกันมาก ซึ่งเป็นการอ่านเมทริกซ์ความสับสนแบบเดียวกับที่ทำกับกาแล็กซีในส่วนที่ 3

เมื่อมองงานนี้ด้วยเครื่องมือจากบทที่ 9 มีสามประเด็นที่ผู้เขียนเห็นว่าควรปรับปรุง และยกมาเป็นบทเรียน

1. ตัวเลขร้อยละ 84 ได้จากการเลือกรอบที่ความแม่นยำกับชุดตรวจสอบสูงที่สุดในสามร้อยรอบ ซึ่งมีอคติเข้าข้างตัวเองตามที่แบบฝึก 11.D แสดงด้วยตัวเลข งานที่ทำต่อควรแยกชุดทดสอบที่ไม่ได้ใช้ในการเลือกใด ๆ ไว้ต่างหาก
2. ข้อมูลฝึก 725 ภาพน้อยสำหรับโครงข่ายขนาดนี้ ซึ่งผู้วิจัยก็ระบุไว้ในงานเองว่าอาจเกิดการเรียนรู้เกิน และเสนอให้เพิ่มข้อมูล ใช้การถ่ายโอนการเรียนรู้ และใช้ดรอปเอาต์
3. การตรวจการปลอมปนสนใจคำถามว่า "ในบรรดาเมล็ดที่แบบจำลองบอกว่าเป็นขาวดอกมะลิ 105 มีเมล็ดพันธุ์อื่นปนอยู่เท่าใด" ซึ่งคือความแม่นยำเชิงบวก (precision) ของพันธุ์นี้ ไม่ใช่ค่าการเรียกคืน และต้องประเมินที่สัดส่วนการปลอมปนที่เป็นจริงในตลาด ผู้วิจัยเองก็เสนอให้สุ่มตรวจดีเอ็นเอเพื่อยืนยันป้ายกำกับ แบบเดียวกับที่ต้องคิดถึงการลงคะแนนของอาสาสมัครใน Galaxy Zoo

### ✏️ แบบฝึก E11.5 ในชั้นเรียน (10 นาที) อ่านผลการจำแนกพันธุ์เมล็ดข้าว
ชุดตรวจสอบมีขาวดอกมะลิ 105 จำนวน 65 เมล็ด กข23 จำนวน 77 เมล็ด กข57 จำนวน 90 เมล็ด และปทุมธานี 1 จำนวน 80 เมล็ด ค่าการเรียกคืนของสี่พันธุ์ตามลำดับเดียวกันคือ 0.94, 0.65, 0.99 และ 0.79

1. คำนวณจำนวนเมล็ดที่ทายถูกของแต่ละพันธุ์ (ปัดเป็นจำนวนเต็ม) เก็บเป็นลิสต์ใน `my_correct` ตามลำดับ KDML105, RD23, RD57, PTT1
2. คำนวณความแม่นยำรวม เก็บใน `my_acc_rice` และตรวจว่าได้ร้อยละ 84 จริง
3. คำนวณความแม่นยำแบบสมดุลจากจำนวนที่ปัดแล้ว เก็บใน `my_bal_rice`

(ส่วนที่ถามถึงความแม่นยำเชิงบวกและการตรวจความบริสุทธิ์ร้อยละ 95 เป็นงานนอกชั้น)

In [ ]:
n_val = np.array([65, 77, 90, 80])            # KDML105, RD23, RD57, PTT1
recall_pub = np.array([0.94, 0.65, 0.99, 0.79])
my_correct = None     # ✏️ list of 4 integers
my_acc_rice = None    # ✏️ overall accuracy
my_bal_rice = None    # ✏️ balanced accuracy from the rounded counts

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_acc_rice', 'my_bal_rice', 'my_correct'])
_k = np.round(recall_pub * n_val).astype(int)
_say("correct per variety:", "(ยังไม่ได้ตอบ)" if my_correct is None else ("✔" if list(my_correct) == list(_k) else "✘"), "—", _k)
_say("overall accuracy   :", "(ยังไม่ได้ตอบ)" if my_acc_rice is None else ("✔" if abs(my_acc_rice - _k.sum() / n_val.sum()) < 1e-3 else "✘"),
      f"— {_k.sum()} / {n_val.sum()} = {_k.sum() / n_val.sum():.3f}")
_say("balanced accuracy  :", "(ยังไม่ได้ตอบ)" if my_bal_rice is None else ("✔" if abs(my_bal_rice - np.mean(_k / n_val)) < 1e-3 else "✘"),
      f"— {np.mean(_k / n_val):.3f}")

In [ ]:
#@title แนวคำตอบ E11.5 (ส่วนในชั้นเรียน)
_say = _gate(['my_acc_rice', 'my_bal_rice', 'my_correct'])
_say('''แนวคำตอบ E11.5 (ส่วนในชั้นเรียน)
0.94 x 65 = 61, 0.65 x 77 = 50, 0.99 x 90 = 89, 0.79 x 80 = 63 เมล็ด รวม 263 จาก 312 = ร้อยละ 84.3 ตรงกับที่รายงาน
ความแม่นยำแบบสมดุล = (0.938 + 0.649 + 0.989 + 0.788)/4 = 0.841 ใกล้กับความแม่นยำรวม เพราะชุดตรวจสอบ
ไม่ได้ไม่สมดุลมาก ตัวเลขทั้งสองยังมาจากการเลือกรอบด้วยชุดตรวจสอบ จึงน่าจะสูงกว่าความแม่นยำกับข้าวใหม่บ้าง''')

### การถ่ายโอนการเรียนรู้: เมื่อใดช่วย และเมื่อใดไม่พอ
การถ่ายโอนการเรียนรู้ (transfer learning) คือการนำโครงข่ายที่ฝึกกับงานหนึ่งซึ่งมีข้อมูลมาก มาเป็นจุดเริ่มต้นของงานใหม่ที่มีข้อมูลน้อย เหตุผลที่มันได้ผลคือชั้นต้น ๆ ของโครงข่ายเรียนรูปแบบทั่วไป เช่น ขอบ จุด และความยาวรี ซึ่งมีประโยชน์กับภาพหลายชนิด ยอซินสกีและคณะ (Yosinski et al., 2014) แสดงว่ายิ่งชั้นลึกขึ้น ลักษณะเด่นยิ่งเฉพาะกับงานเดิมและถ่ายโอนได้น้อยลง ทางปฏิบัติมีสองแบบหลัก แบบแรกคือตรึงส่วนสกัดลักษณะเด่นไว้ แล้วฝึกเฉพาะส่วนหัวใหม่ แบบที่สองคือปรับละเอียด (fine-tune) ทุกชั้นด้วยอัตราการเรียนรู้ที่เล็ก เปรียบได้กับช่างภาพที่ถ่ายภาพท้องฟ้ามานาน เมื่อต้องไปถ่ายภาพเมล็ดพืช เขาไม่ต้องเรียนการโฟกัสและการตั้งแสงใหม่ แต่ต้องเรียนว่าอะไรในภาพเมล็ดที่ควรสังเกต

โค้ดที่ 11.7 ทดลองความคิดนี้ด้วยภาพเมล็ดข้าวจำลองสี่ "พันธุ์" ซึ่งต่างกันที่ความยาวเฉลี่ยและความกว้างเฉลี่ยเพียงเล็กน้อย โดยมีภาพฝึกเพียง 25 ภาพต่อพันธุ์ แล้วเปรียบเทียบสี่วิธี คือโครงข่ายที่ฝึกจากศูนย์ โครงข่ายกาแล็กซี (`models['CNN']` จากคาบที่ 2) ที่ตรึงส่วนสกัดลักษณะเด่น โครงข่ายกาแล็กซีที่ปรับละเอียดทุกชั้น และเส้นฐานที่ใช้ลักษณะเด่นทางกายภาพสามตัว คือความยาว ความกว้าง และพื้นที่ที่วัดจากโมเมนต์อันดับสองของภาพ ร่วมกับการถดถอยโลจิสติก (⏱ ราวสองถึงสี่นาทีบน Colab)

### โค้ดที่ 11.7 — การถ่ายโอนการเรียนรู้ไปยังภาพเมล็ดข้าวจำลองเทียบกับลักษณะเด่นทางกายภาพ

In [ ]:
# ⏱ ราวสองถึงสี่นาทีบน Colab
SEED_TYPES = [(30, 8.0), (30, 9.0), (26, 8.0), (26, 9.5)]   # mean length, width (px)

def make_seed(cls, rng, size=64):
    """A toy rice-grain image: pointed ellipse with husk texture, random orientation."""
    L, W = rng.normal(SEED_TYPES[cls][0], 1.5), rng.normal(SEED_TYPES[cls][1], 0.5)
    y, x = np.mgrid[:size, :size] - (size - 1) / 2
    a = rng.uniform(0, np.pi)
    u, v = x * np.cos(a) + y * np.sin(a), -x * np.sin(a) + y * np.cos(a)
    inside = np.abs(2 * u / L)**2.2 + np.abs(2 * v / W)**2 < 1
    grain = inside * (0.7 + 0.3 * np.cos(np.pi * v / W)**2)
    grain *= 1 + 0.15 * gaussian_filter(rng.normal(size=grain.shape), 1.0)
    return (gaussian_filter(grain, 0.8) + rng.normal(0, 0.05, grain.shape)).astype(np.float32)

def make_seed_set(n_per_class, seed=0):
    rng = np.random.default_rng(seed)
    X = np.array([make_seed(c, rng) for c in range(4) for _ in range(n_per_class)])
    y = np.repeat(np.arange(4), n_per_class)
    p = rng.permutation(len(y))
    return X[p], y[p]

Xs_tr, ys_tr = make_seed_set(25, seed=21)          # only 100 labelled grains
Xs_te, ys_te = make_seed_set(200, seed=23)

def size_features(X):                              # length, width, area from image moments
    out = []
    for img in X:
        yy, xx = np.nonzero(img > 0.35)
        ev = np.sort(np.linalg.eigvalsh(np.cov(np.vstack([xx, yy]))))[::-1]
        out.append([4 * np.sqrt(ev[0]), 4 * np.sqrt(ev[1]), len(xx)])
    return np.array(out)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
baseline.fit(size_features(Xs_tr), ys_tr)
print("size features + logistic regression:",
      np.mean(baseline.predict(size_features(Xs_te)) == ys_te).round(3))

for strategy in ('scratch', 'frozen', 'fine-tune'):
    accs = []
    for s in range(3):
        torch.manual_seed(s)
        model = GalaxyCNN(n_classes=4)
        if strategy != 'scratch':                  # start from the galaxy CNN
            model = GalaxyCNN()
            model.load_state_dict(models['CNN'].state_dict())
            model.head = nn.Linear(64, 4)          # new head for 4 seed types
            if strategy == 'frozen':
                for p in model.features.parameters():
                    p.requires_grad = False
        torch.manual_seed(s); np.random.seed(s)
        opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=2e-3)
        Xt, yt = torch.tensor(Xs_tr[:, None]), torch.tensor(ys_tr)
        for epoch in range(60):
            model.train()
            perm = np.random.permutation(len(yt))
            for b in range(0, len(yt), 32):
                i = perm[b:b + 32]
                loss = nn.functional.cross_entropy(model(Xt[i]), yt[i])
                opt.zero_grad(); loss.backward(); opt.step()
        accs.append(np.mean(predict(model, Xs_te).argmax(1) == ys_te))
    print(f"{strategy:9s}: test accuracy {np.round(accs, 3)}, mean {np.mean(accs):.3f}")

In [ ]:
F_te = size_features(Xs_te)
fig, ax = plt.subplots(1, 5, figsize=(12, 2.6), gridspec_kw={"width_ratios": [1, 1, 1, 1, 1.6]})
for c in range(4):
    ax[c].imshow(Xs_tr[np.where(ys_tr == c)[0][0]], cmap='gray', origin='lower'); ax[c].axis('off')
    ax[c].set_title(f"type {c}: L {SEED_TYPES[c][0]}, W {SEED_TYPES[c][1]}", fontsize=8)
for c in range(4):
    ax[4].scatter(F_te[ys_te == c, 0], F_te[ys_te == c, 1], s=6, alpha=0.6, label=f"type {c}")
ax[4].set(xlabel="length from moments (px)", ylabel="width (px)", title="size features, test grains (SYNTHETIC)")
ax[4].title.set_fontsize(8); ax[4].legend(fontsize=7, markerscale=2)
fig.tight_layout(); plt.show()

ผลในตำราเป็นบทเรียนที่ดี เมื่อมีภาพฝึกเพียง 100 ภาพ โครงข่ายที่ฝึกจากศูนย์ได้ความแม่นยำเฉลี่ยร้อยละ 31 เกือบเท่ากับการเดาสุ่มซึ่งได้ร้อยละ 25 โครงข่ายกาแล็กซีที่ตรึงส่วนสกัดลักษณะเด่นได้ร้อยละ 41 และที่ปรับละเอียดทุกชั้นได้ร้อยละ 51 การถ่ายโอนการเรียนรู้จึงช่วยได้จริง ลักษณะเด่นที่เรียนจากกาแล็กซี เช่น ตัวตรวจความยาวรีของซิการ์และจาน มีประโยชน์กับเมล็ดข้าวด้วย **แต่เส้นฐานที่ใช้ความยาว ความกว้าง และพื้นที่ร่วมกับการถดถอยโลจิสติกได้ร้อยละ 80 ดีกว่าทุกวิธีของโครงข่ายมาก** ความแม่นยำของโครงข่ายยังแกว่งมากระหว่างเมล็ดสุ่มสามค่า เช่น การปรับละเอียดได้ตั้งแต่ร้อยละ 43 ถึง 62 (ตัวเลขบน Colab อาจต่างจากนี้ แต่ลำดับของสี่วิธีมักเหมือนเดิม)

กราฟทางขวาบอกเหตุผล ต้องยอมรับตรง ๆ ว่าตัวอย่างนี้เอื้อต่อเส้นฐาน เพราะ "พันธุ์" จำลองต่างกันที่ขนาดเท่านั้นโดยการสร้าง ลักษณะเด่นสามตัวจึงวัดสิ่งที่แยกประเภทได้พอดี ในข้าวจริง พันธุ์ต่างกันที่สี พื้นผิวของเปลือก รูปร่างของปลายเมล็ด และลักษณะอื่นที่วัดด้วยมือได้ยาก ซึ่งเป็นเหตุผลที่งานจริงใช้โครงข่ายคอนโวลูชัน บทเรียนที่ถูกต้องจึงไม่ใช่ "ลักษณะเด่นทางกายภาพดีกว่าโครงข่ายเสมอ" แต่คือ **ต้องเทียบกับเส้นฐานทางกายภาพที่ดีที่สุดที่ทำได้ทุกครั้ง** ถ้าโครงข่ายไม่ชนะเส้นฐานนั้นอย่างชัดเจน ข้อมูลอาจยังไม่มากพอ หรือความต่างของประเภทอาจอยู่ที่ลักษณะที่วัดได้ง่ายอยู่แล้ว

In [ ]:
#@title คำถามทบทวนข้อ 5 (แนวคำตอบ)
print('''คำถามทบทวนข้อ 5: ในงานจำแนกพันธุ์เมล็ดข้าว ทำไมความแม่นยำรายพันธุ์ของขาวดอกมะลิ 105 จึงยังไม่พอสำหรับการตรวจการปลอมปน''')
_say = _gate([])
_say('''คำถามทบทวนข้อ 5 — แนวคำตอบ: ตัวเลขร้อยละ 94 คือค่าการเรียกคืน ซึ่งบอกว่าเมล็ดขาวดอกมะลิ 105 จริงถูกจำได้เท่าใด การตรวจการปลอมปน
ต้องรู้ว่าเมล็ดที่ถูกเรียกว่าขาวดอกมะลิ 105 มีพันธุ์อื่นปนอยู่เท่าใด ซึ่งคือความแม่นยำเชิงบวก และต้องประเมิน
ที่สัดส่วนการปลอมปนจริงในตลาด''')

---
### งานก่อนคาบที่ 4: ฝึกสมาชิกตัวที่ 1 และ 2 ของการรวมกลุ่ม (⏱ ราว 25 นาทีบน Colab)
คาบที่ 4 ใช้การรวมกลุ่มโครงข่ายคอนโวลูชันสามตัว (โค้ดที่ 11.9 ฉบับย่อ) สมาชิกตัวแรกคือ `models['CNN']` ที่ฝึกไว้แล้วในคาบที่ 2 ส่วนสมาชิกตัวที่ 1 และ 2 ต้องฝึกใหม่ด้วยเมล็ดสุ่ม 1 และ 2 ซึ่งใช้เวลาราวตัวละสิบสองนาทีบน Colab การฝึกในชั้นจะกินเวลาเกือบครึ่งคาบ จึงให้ผู้เรียนรันเซลล์ถัดไปก่อนเข้าคาบที่ 4

ขั้นตอน: (1) ติ๊กช่อง `USE_DRIVE` ในส่วนที่ 0 แล้วรันส่วนที่ 0 ใหม่ เพื่อให้ไฟล์น้ำหนักอยู่ใน Google Drive และไม่หายเมื่อปิด Colab (2) คลิกเซลล์หัวคาบที่ 3 แล้วเลือก Runtime → Run before จากนั้นรันเซลล์ซ่อนแรกของคาบที่ 3 (3) รันเซลล์ถัดไป แล้วปล่อยให้ทำงานจนเสร็จ เซลล์บันทึกน้ำหนักเป็น `outputs/ens_member_1.pt` และ `outputs/ens_member_2.pt` ถ้ามีไฟล์อยู่แล้ว เซลล์จะข้ามสมาชิกตัวนั้น

ขั้นตอนการฝึกเหมือนกับในโค้ดที่ 11.9 ทุกประการ (เมล็ดสุ่มเดียวกันกำหนดทั้งน้ำหนักเริ่มต้นและลำดับของข้อมูล) ผลในคาบที่ 4 จึงเหมือนกับการฝึกในชั้น

In [ ]:
# ⏱ ราว 25 นาทีบน Colab — งานก่อนคาบที่ 4 (ฝึกสมาชิกตัวที่ 1 และ 2 ของโค้ดที่ 11.9 ฉบับย่อ)
_t0 = time.time()
for s in (1, 2):
    path = f"outputs/ens_member_{s}.pt"
    if os.path.exists(path):
        print(f"member {s}: {path} มีอยู่แล้ว ข้ามการฝึก")
        continue
    torch.manual_seed(s)
    m_ = GalaxyCNN()
    fit(m_, X_train, y_train, X_val, y_val, seed=s)
    torch.save(m_.state_dict(), path)
    print(f"member {s}: test accuracy {np.mean(predict(m_, X_test).argmax(1) == y_test):.3f}, saved {path} ({time.time() - _t0:.0f} s)")

---
# คาบที่ 4 โครงข่ายมองอะไร และโครงข่ายรู้ตัวหรือไม่ว่าไม่รู้

> **เริ่มคาบนี้หลังจากปิด Colab ไปแล้ว:** คลิกเซลล์หัวคาบที่ 1 แล้วเลือก Runtime → Run before จากนั้นกลับมารันเซลล์ซ่อนถัดไป

เซลล์ซ่อนถัดไปเหมือนกับเซลล์แรกของคาบที่ 3 คือเตรียม `GalaxyCNN` ภาพกาแล็กซีจำลอง `fit`, `predict` และ `models['CNN']` ที่ฝึกแล้ว (ถ้ายังไม่มีไฟล์ใน `outputs/` จะฝึกใหม่ ⏱ ราวสิบนาที การติ๊ก `USE_DRIVE` ในส่วนที่ 0 ทำให้ไฟล์ไม่หายระหว่างคาบ)

In [ ]:
#@title เตรียมโค้ดที่ 11.1–11.3 และโครงข่ายที่ฝึกแล้ว (ดับเบิลคลิกเพื่อดูโค้ด)
import os, time
import numpy as np
# ---- Listing 11.1: GalaxyCNN and MLP
import torch
import torch.nn as nn

class GalaxyCNN(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 64 -> 32
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 32 -> 16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 16 -> 8
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU())                   # 64 maps, 8x8
        self.head = nn.Linear(64, n_classes)
    def forward(self, x):
        return self.head(self.features(x).mean(dim=(2, 3)))   # global average pooling

class MLP(nn.Module):                                           # for comparison
    def __init__(self, n_classes=5):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(),
                                 nn.Linear(64 * 64, 128), nn.ReLU(),
                                 nn.Linear(128, 64), nn.ReLU(),
                                 nn.Linear(64, n_classes))
    def forward(self, x):
        return self.net(x)

# ---- Listing 11.2: toy galaxies (from outputs/galaxies.npz if saved in period 2)
import numpy as np
from scipy.ndimage import gaussian_filter
CLASSES = ['smooth round', 'smooth cigar', 'edge-on disk', 'spiral', 'barred spiral']

def make_galaxy(cls, rng, size=64):
    """One noisy 64x64 image of a toy galaxy of class cls (0-4)."""
    y, x = np.mgrid[:size, :size] - (size - 1) / 2
    x0, y0 = rng.normal(0, 1.5, 2)                     # small centring error
    x, y = x - x0, y - y0
    pa = rng.uniform(0, np.pi)                          # position angle
    xr = x * np.cos(pa) + y * np.sin(pa)
    yr = -x * np.sin(pa) + y * np.cos(pa)
    def sersic(re, n, q):
        r = np.sqrt(xr**2 + (yr / q)**2)
        b = 2 * n - 1 / 3
        return np.exp(-b * ((r / re)**(1 / n) - 1))
    if cls == 0:
        img = sersic(rng.uniform(6, 10), 4, rng.uniform(0.8, 1.0))
    elif cls == 1:
        img = sersic(rng.uniform(6, 10), rng.uniform(1, 4), rng.uniform(0.2, 0.4))
    elif cls == 2:
        img = sersic(rng.uniform(10, 15), 1, rng.uniform(0.08, 0.15)) \
              + 0.3 * sersic(2.5, 2, 0.6)
    else:
        q = rng.uniform(0.45, 1.0)                      # cos(inclination)
        re = rng.uniform(10, 15)
        r = np.sqrt(xr**2 + (yr / q)**2)
        theta = np.arctan2(yr / q, xr)
        pitch = rng.uniform(0.25, 0.45)                 # arm winding
        arms = 0.5 * (1 + np.cos(2 * (theta - np.log(r + 1e-3) / np.tan(pitch))))
        arms = arms * (1 - np.exp(-(r / (0.4 * re))**2))  # arms start outside the centre
        disk = np.exp(-r / (re / 1.68)) * (0.4 + 1.2 * arms**2)
        img = disk + 1.5 * sersic(2.0, 2, 0.7 + 0.3 * q) / np.exp(11 / 3)   # bulge, rounder than disk
        if cls == 4:                                     # bar along the disk
            bar_len = rng.uniform(0.4, 0.6) * re
            img += 1.2 * np.exp(-(xr / bar_len)**4 - (yr / (0.3 * bar_len))**2)
    img = np.arcsinh(img / 0.3)                        # astronomical asinh stretch
    img = img / img.max() * rng.uniform(0.6, 1.0)
    img = gaussian_filter(img, rng.uniform(0.8, 1.5))   # seeing (PSF)
    img = img + rng.normal(0, 0.05, img.shape)          # sky + read noise
    return img.astype(np.float32)

def make_galaxy_set(n_per_class, seed=0):
    rng = np.random.default_rng(seed)
    X = np.array([make_galaxy(c, rng) for c in range(5) for _ in range(n_per_class)])
    y = np.repeat(np.arange(5), n_per_class)
    p = rng.permutation(len(y))
    return X[p], y[p]

if os.path.exists("outputs/galaxies.npz"):
    _d = np.load("outputs/galaxies.npz")
    X_train, y_train, X_val, y_val, X_test, y_test = (_d[k] for k in ("X_train", "y_train", "X_val", "y_val", "X_test", "y_test"))
else:
    X_train, y_train = make_galaxy_set(1000, seed=1)
    X_val, y_val = make_galaxy_set(250, seed=2)
    X_test, y_test = make_galaxy_set(400, seed=3)
    np.savez_compressed("outputs/galaxies.npz", X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
                        X_test=X_test, y_test=y_test)
print("toy galaxies (SYNTHETIC):", X_train.shape, X_val.shape, X_test.shape)

# ---- Listing 11.3: augment, fit, predict, and the trained MLP and CNN
def augment(xb):
    """Random 90-degree rotation and mirror flip: exact symmetries of the pixel grid."""
    xb = torch.rot90(xb, np.random.randint(4), dims=(2, 3))
    if np.random.rand() < 0.5:
        xb = torch.flip(xb, dims=(3,))
    return xb

def fit(model, X, y, X_val, y_val, epochs=30, lr=2e-3, aug=False, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(X[:, None]), torch.tensor(y)),
        batch_size=64, shuffle=True)
    Xv = torch.tensor(X_val[:, None])
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    val_acc = []
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            if aug:
                xb = augment(xb)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val_acc.append(np.mean(model(Xv).argmax(1).numpy() == y_val))
    return val_acc

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(torch.tensor(X[:, None])), dim=1).numpy()

models = {}
for name, Model, lr, epochs in [('MLP', MLP, 1e-3, 15), ('CNN', GalaxyCNN, 2e-3, 30)]:
    torch.manual_seed(0)
    models[name] = Model()
    if os.path.exists(f"outputs/{name}.pt"):
        models[name].load_state_dict(torch.load(f"outputs/{name}.pt"))
        print(f"loaded outputs/{name}.pt")
    else:
        print(f"outputs/{name}.pt not found: training {name} as in Listing 11.3 ...")
        _t0 = time.time()
        fit(models[name], X_train, y_train, X_val, y_val, epochs=epochs, lr=lr)
        torch.save(models[name].state_dict(), f"outputs/{name}.pt")
        print(f"  trained and saved outputs/{name}.pt ({time.time() - _t0:.0f} s)")
    models[name].eval()

---
# ส่วนที่ 6 โครงข่ายมองอะไร: Grad-CAM และการเรียนรู้ทางลัด (หัวข้อ 11.6)

### การดูแผนที่ลักษณะเด่น
วิธีตรวจโครงข่ายที่ง่ายที่สุดคือวาดแผนที่ลักษณะเด่นของแต่ละชั้นเมื่อป้อนภาพหนึ่งภาพ เซลล์ถัดไปวาดแผนที่ 8 แผ่นแรกของคอนโวลูชันชั้นแรก (หลัง ReLU) และ 8 แผ่นแรกของชั้นสุดท้าย เมื่อป้อนภาพกาแล็กซีกังหันมีคานหนึ่งภาพให้ `models['CNN']`

In [ ]:
cnn = models['CNN']
img0 = torch.tensor(X_test[y_test == 4][:1, None])
with torch.no_grad():
    maps1 = cnn.features[:2](img0)[0]          # first conv + ReLU: 16 maps, 64 x 64
    maps4 = cnn.features(img0)[0]              # last layer: 64 maps, 8 x 8
fig, ax = plt.subplots(2, 9, figsize=(12, 3.8))
for r, (maps, t) in enumerate([(maps1, "layer 1 (64x64)"), (maps4, "layer 4 (8x8)")]):
    ax[r, 0].imshow(img0[0, 0], cmap="gray", origin="lower"); ax[r, 0].set_title("input", fontsize=8)
    for k in range(8):
        ax[r, k + 1].imshow(maps[k].numpy(), cmap="viridis", origin="lower"); ax[r, k + 1].set_title(f"{t}, map {k}", fontsize=6)
for a_ in ax.flat:
    a_.axis("off")
fig.suptitle("feature maps of the trained CNN for one barred spiral (SYNTHETIC)", fontsize=9)
fig.tight_layout(); plt.show()

แผนที่ของชั้นแรกดูเหมือนภาพต้นฉบับที่ผ่านตัวกรองเน้นขอบหรือจุดสว่าง บางแผ่นเป็นศูนย์ทั้งแผ่น เพราะ ReLU ตัดค่าลบทิ้ง ตัวกรองนั้นจึงไม่ตอบสนองต่อภาพนี้เลย ส่วนแผนที่ของชั้นสุดท้ายมีขนาดเพียง $8\times8$ และอ่านความหมายได้ยาก วิธีนี้บอกว่าโครงข่ายตอบสนองต่ออะไร แต่ไม่ได้บอกว่าส่วนใดของภาพมีผลต่อการตัดสินใจของประเภทที่สนใจ คำถามหลังตอบได้ดีกว่าด้วยแผนที่ความสำคัญเชิงปริภูมิ

### Grad-CAM
Grad-CAM (gradient-weighted class activation mapping) สร้างแผนที่ความร้อนบนภาพ ซึ่งบอกว่าบริเวณใดทำให้คะแนนของประเภทที่เลือกสูงขึ้น ความคิดของมันต่อยอดจากแผนที่การกระตุ้นของประเภท (class activation map) ของโจวและคณะ (Zhou et al., 2016)

1. หาอนุพันธ์ของคะแนนของประเภทนั้นเทียบกับแผนที่ลักษณะเด่นของชั้นคอนโวลูชันสุดท้าย แล้วเฉลี่ยทั่วทั้งแผนที่ ได้น้ำหนักความสำคัญของแผนที่แต่ละแผ่น
2. รวมแผนที่ทั้งหมดแบบถ่วงน้ำหนักด้วยน้ำหนักนั้น ตัดส่วนที่ติดลบทิ้ง แล้วขยายกลับเป็นขนาดของภาพต้นฉบับ

สำหรับโครงข่ายในบทนี้ซึ่งใช้การรวมแบบเฉลี่ยทั้งหมดตามด้วยชั้นเชิงเส้นชั้นเดียว Grad-CAM ให้ผลเท่ากับแผนที่การกระตุ้นของประเภทพอดี แผนที่ที่ได้มีความละเอียดเพียง $8\times8$ ตามขนาดของชั้นสุดท้าย จึงบอกได้เพียงบริเวณอย่างหยาบ

การใช้ Grad-CAM ที่มีประโยชน์ที่สุดไม่ใช่การยืนยันว่าโครงข่าย "ฉลาด" แต่เป็นการจับได้ว่าโครงข่ายเรียนทางลัด (shortcut learning) คือเรียนจากลักษณะที่บังเอิญสัมพันธ์กับป้ายกำกับในข้อมูลฝึก แต่ไม่เกี่ยวกับคำถามทางวิทยาศาสตร์ อุปมาที่มักใช้คือม้าชื่อฮันส์ผู้ฉลาดในต้นศตวรรษที่ยี่สิบ ซึ่งดูเหมือนคิดเลขได้ แต่จริง ๆ แล้วอ่านท่าทางของผู้ถาม ในข้อมูลทางฟิสิกส์ ทางลัดเกิดได้ง่ายกว่าที่คิด ถ้ากาแล็กซีประเภทหนึ่งส่วนใหญ่ถ่ายด้วยกล้องตัวหนึ่ง โครงข่ายอาจเรียนลายเซ็นของกล้องแทนรูปร่าง ถ้าภาพของเมล็ดข้าวพันธุ์หนึ่งสแกนในวันที่กระจกสแกนเนอร์มีรอยฝุ่น โครงข่ายอาจเรียนรอยฝุ่น

โค้ดที่ 11.8 สร้างทางลัดนี้ขึ้นอย่างจงใจ ภาพกาแล็กซีกังหันมีคานร้อยละ 90 ในชุดฝึกและชุดตรวจสอบถูกเติมจุดสว่างขนาด $3\times3$ พิกเซลที่มุมบนซ้าย ซึ่งอาจแทนดาวฤกษ์ข้างเคียงหรือตำหนิของเครื่องมือ แล้วฝึกโครงข่ายใหม่ด้วยข้อมูลนี้ (⏱ ราวสิบนาทีบน Colab)

### โค้ดที่ 11.8 — Grad-CAM และการทดลองทางลัดด้วยจุดสว่าง

In [ ]:
# ⏱ ราวสิบนาทีบน Colab
import torch.nn.functional as F

def grad_cam(model, x, target):
    """Grad-CAM map (64 x 64, unit sum) of class `target` for one image x (1,1,64,64)."""
    model.eval()
    A = model.features(x)                          # last feature maps (1, 64, 8, 8)
    A.retain_grad()
    score = model.head(A.mean(dim=(2, 3)))[0, target]
    model.zero_grad()
    score.backward()
    w = A.grad.mean(dim=(2, 3), keepdim=True)      # importance of each feature map
    cam = F.relu((w * A).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)
    cam = cam[0, 0].detach().numpy()
    return cam / (cam.sum() + 1e-12)

def add_star(X):                                   # a 3x3 bright spot near one corner
    X = X.copy(); X[:, 4:7, 4:7] += 1.5
    return X

rng = np.random.default_rng(7)                     # 90% of barred training images get it
X_tr_s = X_train.copy(); m = (y_train == 4) & (rng.random(len(y_train)) < 0.9)
X_tr_s[m] = add_star(X_tr_s[m])
X_va_s = X_val.copy(); m = (y_val == 4) & (rng.random(len(y_val)) < 0.9)
X_va_s[m] = add_star(X_va_s[m])
torch.manual_seed(0)
shortcut = GalaxyCNN()
val_acc = fit(shortcut, X_tr_s, y_train, X_va_s, y_val)
print("validation accuracy (with the artefact):", round(val_acc[-1], 3))
pred = predict(shortcut, X_test).argmax(1)
print("clean test:", np.mean(pred == y_test).round(3),
      " barred recall:", np.mean(pred[y_test == 4] == 4).round(3))
pred = predict(shortcut, add_star(X_test)).argmax(1)
print("spot on every test image: accuracy", np.mean(pred == y_test).round(3),
      " fraction called barred", np.mean(pred == 4).round(3))

yy, xx = np.mgrid[:64, :64]
corner = (yy < 16) & (xx < 16)                     # 6% of the image area
for name, model in [('clean CNN', models['CNN']), ('shortcut CNN', shortcut)]:
    frac = [grad_cam(model, torch.tensor(img[None, None]), 4)[corner].sum()
            for img in add_star(X_test[y_test == 4][:100])]
    print(f"{name}: share of Grad-CAM in the corner = {np.mean(frac):.3f}")

In [ ]:
img = add_star(X_test[y_test == 4][:1])[0]
fig, ax = plt.subplots(1, 3, figsize=(8, 2.8))
ax[0].imshow(img, cmap='gray', origin='upper'); ax[0].set_title('barred spiral + spot', fontsize=9)
for a, (name, model) in zip(ax[1:], [('clean CNN', models['CNN']), ('shortcut CNN', shortcut)]):
    a.imshow(img, cmap='gray', origin='upper')
    a.imshow(grad_cam(model, torch.tensor(img[None, None]), 4), cmap='inferno', alpha=0.55, origin='upper')
    a.set_title(f'Grad-CAM, {name}', fontsize=9)
for a in ax:
    a.axis('off')
fig.suptitle("Grad-CAM of the class 'barred spiral' (SYNTHETIC)", fontsize=9)
fig.tight_layout(); plt.show()

ผลในตำราแสดงอันตรายของทางลัดได้ชัดเจน

- ความแม่นยำกับชุดตรวจสอบซึ่งมีจุดสว่างแบบเดียวกันคือร้อยละ 96.8 ไม่ต่างจากโครงข่ายปกติ ผู้ที่ดูเพียงตัวเลขนี้จะไม่รู้เลยว่ามีปัญหา
- กับภาพทดสอบที่สะอาด ความแม่นยำรวมยังดูดีที่ร้อยละ 94.9 แต่ค่าการเรียกคืนของกาแล็กซีกังหันมีคานลดจากร้อยละ 97.8 ของโครงข่ายปกติเหลือร้อยละ 91
- เมื่อเติมจุดสว่างให้ภาพทดสอบ **ทุก** ภาพ โครงข่ายทายทุกภาพเป็นกาแล็กซีกังหันมีคาน ความแม่นยำเหลือร้อยละ 20 เท่ากับการทายประเภทเดียวตลอด
- แผนที่ Grad-CAM ของโครงข่ายที่เรียนทางลัดมีสัดส่วนอยู่ในสี่เหลี่ยม $16\times16$ พิกเซลที่มุมบนซ้ายถึงร้อยละ 53 ขณะที่โครงข่ายปกติมีสัดส่วนในบริเวณเดียวกันเพียงร้อยละ 5 ผู้ที่ดูแผนที่ Grad-CAM เพียงไม่กี่ภาพจะเห็นทันทีว่าโครงข่ายมองผิดที่

### ✏️ แบบฝึก 11.F (5 นาที) Grad-CAM กระจุกที่มุมมากกว่าพื้นที่กี่เท่า
ถ้าแผนที่กระจายสม่ำเสมอทั่วภาพ สัดส่วนในสี่เหลี่ยมที่มุมจะเท่ากับสัดส่วนพื้นที่ของสี่เหลี่ยมนั้น คำนวณสัดส่วนพื้นที่ของ `corner` เก็บใน `my_corner_area` แล้วคำนวณว่าสัดส่วนของ Grad-CAM ในมุมของโครงข่ายที่เรียนทางลัดมากกว่าสัดส่วนพื้นที่กี่เท่า เก็บใน `my_enrich` (หลังโค้ดที่ 11.8 ตัวแปร `frac` เป็นสัดส่วนของ Grad-CAM ในมุมของภาพ 100 ภาพสำหรับโครงข่ายตัวสุดท้ายในวงรอบ คือโครงข่ายที่เรียนทางลัด)

In [ ]:
my_corner_area = None   # ✏️ fraction of the 64 x 64 image inside `corner`
my_enrich = None        # ✏️ mean share of Grad-CAM in the corner (shortcut CNN) / area fraction

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_corner_area', 'my_enrich'])
_area = corner.mean(); _en = np.mean(frac) / _area
_say("corner area fraction:", "(ยังไม่ได้ตอบ)" if my_corner_area is None else ("✔" if abs(my_corner_area - _area) < 1e-6 else "✘"),
      f"— {_area:.4f}")
_say("enrichment          :", "(ยังไม่ได้ตอบ)" if my_enrich is None else ("✔" if abs(my_enrich - _en) < 1e-3 else "✘"),
      f"— {_en:.1f} times the area share")

In [ ]:
#@title แนวคำตอบ 11.F
_say = _gate(['my_corner_area', 'my_enrich'])
_say('''แนวคำตอบ 11.F
my_corner_area = corner.mean() = 256/4096 = 0.0625 (ร้อยละ 6)
my_enrich = np.mean(frac) / corner.mean() ในตำราคือ 0.532/0.0625 = 8.5 เท่า ส่วนโครงข่ายปกติได้ 0.052/0.0625 = 0.8 เท่า
(บน Colab ผู้เขียนได้ 0.523 และ 0.108 คือ 8.4 และ 1.7 เท่า) โครงข่ายปกติมีสัดส่วนในมุมใกล้กับพื้นที่ของมุม
เพราะมุมภาพเป็นท้องฟ้าว่าง ส่วนแผนที่ที่กระจุกมากกว่าพื้นที่หลายเท่าในบริเวณที่ไม่มีวัตถุเป็นสัญญาณเตือนของทางลัดที่ชัดเจน''')

เซลล์ถัดไปยืนยันข้อความว่า Grad-CAM ของโครงข่ายนี้เท่ากับแผนที่การกระตุ้นของประเภท ซึ่งคำนวณได้โดยตรงจากน้ำหนักของส่วนหัวโดยไม่ต้องหาอนุพันธ์ (เพราะเกรเดียนต์ของคะแนนเทียบกับแผนที่แต่ละแผ่นคือน้ำหนักของส่วนหัวหารด้วยจำนวนพิกเซล $8\times8$)

In [ ]:
x1 = torch.tensor(X_test[y_test == 4][:1, None])
with torch.no_grad():
    A_ = cnn.features(x1)
    cam_ = torch.relu((cnn.head.weight[4][None, :, None, None] * A_).sum(1, keepdim=True))
    cam_ = nn.functional.interpolate(cam_, size=(64, 64), mode='bilinear', align_corners=False)[0, 0].numpy()
cam_ = cam_ / cam_.sum()
print(f"last feature maps {tuple(A_.shape[-2:])}; max |Grad-CAM - CAM| (both unit sum) = {np.abs(grad_cam(cnn, x1, 4) - cam_).max():.1e}")

> **ความเข้าใจคลาดเคลื่อนที่พบบ่อย:** "ถ้า Grad-CAM ชี้ไปที่ตัวกาแล็กซี แปลว่าโครงข่ายเรียนฟิสิกส์ที่ถูกต้อง" แผนที่ Grad-CAM บอกว่าบริเวณใดมีผลต่อคะแนน ไม่ได้บอกว่าโครงข่ายใช้ลักษณะใดของบริเวณนั้น แผนที่ที่ชี้ไปที่ใจกลางของกาแล็กซีกังหันมีคานเข้ากันได้กับการที่โครงข่ายดูคาน แต่ก็เข้ากันได้กับการที่มันดูความสว่างของใจกลางหรือขนาดของกาแล็กซีเช่นกัน แผนที่ความสำคัญบางวิธียังให้ภาพที่แทบไม่เปลี่ยนเมื่อสุ่มน้ำหนักของโครงข่ายใหม่ทั้งหมด ซึ่งแปลว่าภาพนั้นสะท้อนโครงสร้างของภาพเข้ามากกว่าสิ่งที่โครงข่ายเรียน Grad-CAM จึงเป็นเครื่องมือหาปัญหาที่ดี แต่เป็นหลักฐานยืนยันความถูกต้องที่อ่อน การยืนยันที่ดีกว่าคือการทดลองควบคุม เช่น การเติมหรือลบลักษณะที่สงสัยแล้วดูว่าการทำนายเปลี่ยนอย่างไร ดังที่โค้ดที่ 11.8 ทำกับจุดสว่าง

In [ ]:
#@title คำถามทบทวนข้อ 6 (แนวคำตอบ)
print('''คำถามทบทวนข้อ 6: Grad-CAM ช่วยอะไรได้ในตัวอย่างจุดสว่าง และทำไมจึงไม่ควรใช้เป็นหลักฐานว่าโครงข่ายเรียนฟิสิกส์ถูก''')
_say = _gate([])
_say('''คำถามทบทวนข้อ 6 — แนวคำตอบ: Grad-CAM แสดงว่าโครงข่ายที่เรียนทางลัดให้น้ำหนักกับมุมที่มีจุดสว่างมากผิดปกติ จึงช่วยหาปัญหาได้ดี
แต่แผนที่บอกเพียงบริเวณ ไม่บอกลักษณะที่โครงข่ายใช้ และบางวิธีให้แผนที่ที่ไม่เปลี่ยนแม้สุ่มน้ำหนักใหม่
การยืนยันต้องใช้การทดลองควบคุม เช่น เติมหรือลบลักษณะที่สงสัยแล้วดูผล''')

---
# ส่วนที่ 7 ความไม่แน่นอน การ์ดแบบจำลอง และจริยธรรมของข้อมูล (หัวข้อ 11.7)

### การรวมกลุ่มโครงข่ายคอนโวลูชันและข้อมูลนอกการแจกแจง
บทที่ 10 ได้แนะนำการรวมกลุ่มโครงข่ายเพื่อประมาณความไม่แน่นอนจากการฝึก ส่วนนี้ใช้วิธีเดียวกันกับโครงข่ายคอนโวลูชัน และเพิ่มการทดสอบที่สำคัญมากในงานจริง คือข้อมูลนอกการแจกแจง (out-of-distribution) ของข้อมูลฝึก ในการสำรวจท้องฟ้าจริง ภาพที่ตัดมารอบวัตถุไม่ได้เป็นกาแล็กซีห้าประเภทที่โครงข่ายรู้จักเสมอ บางภาพเป็นดาวฤกษ์ในกาแล็กซีของเราเอง บางภาพเป็นกาแล็กซีสองดวงที่กำลังรวมกัน และบางภาพเป็นตำหนิของเครื่องมือ โครงข่ายที่ดีควรบอกได้ว่า "ไม่รู้" กับภาพเหล่านี้

โค้ดที่ 11.9 ในตำราฝึกโครงข่ายห้าตัวด้วยเมล็ดสุ่ม 0 ถึง 4 แล้วป้อนภาพของแหล่งกำเนิดแบบจุด ซึ่งแทนดาวฤกษ์ และภาพของกาแล็กซีเรียบสองดวงที่อยู่ติดกัน ซึ่งแทนการรวมตัว

**ฉบับย่อสำหรับชั้นเรียน:** บน Colab การฝึกโครงข่ายคอนโวลูชันหนึ่งตัวใช้เวลาราวสิบนาที ฉบับเต็มห้าตัวจึงใช้ราวหนึ่งชั่วโมง ในชั้นเรียนจึงใช้การรวมกลุ่มสามตัว สมาชิกตัวแรกคือ `models['CNN']` ที่ฝึกไว้แล้วในคาบที่ 2 ซึ่งใช้เมล็ดสุ่ม 0 และการตั้งค่าเดียวกับสมาชิกตัวแรกของโค้ดที่ 11.9 จึงไม่ต้องฝึกใหม่ สมาชิกที่ใช้เมล็ดสุ่ม 1 และ 2 ผู้เรียนฝึกไว้แล้วในงานก่อนคาบที่ 4 (ท้ายคาบที่ 3) เซลล์ถัดไปจึงเพียงโหลดน้ำหนักจาก `outputs/` ถ้าไม่พบไฟล์ เซลล์จะฝึกสมาชิกที่ขาดตามโค้ดที่ 11.9 แล้วบันทึกไว้ โค้ดส่วนอื่นเหมือนโค้ดที่ 11.9 ทุกบรรทัด ส่วนโค้ดที่ 11.9 ฉบับเต็มห้าตัวอยู่ในงานนอกชั้น (ส่วนที่ 8) ข้อสรุปของทั้งสองฉบับเหมือนกัน

> ⏱ **ถ้าทำงานก่อนคาบที่ 4 แล้ว เซลล์ถัดไปใช้เวลาไม่ถึงหนึ่งนาที** ถ้ายังไม่ได้ทำ เซลล์จะฝึกสมาชิกสองตัวในชั้นซึ่งใช้เวลาราว 25 นาที ในกรณีนั้นให้เริ่มรันทันที แล้วอ่านและอภิปรายหัวข้อ "การ์ดแบบจำลอง" และ "จริยธรรมและสัญญาอนุญาตของข้อมูล" ด้านล่างระหว่างรอ หรือผู้สอนรันเซลล์นี้ล่วงหน้าแล้วฉายผลให้ผู้เรียนดู

### โค้ดที่ 11.9 ฉบับย่อสำหรับชั้นเรียน (สมาชิกสามตัว)

In [ ]:
# ⏱ ไม่ถึงหนึ่งนาทีถ้าทำงานก่อนคาบที่ 4 แล้ว (ถ้ายังไม่ได้ทำ ราว 25 นาทีบน Colab)
_t0 = time.time()
def point_source(rng):                              # a star: PSF only, no extended light
    img = np.zeros((64, 64))
    c = (31.5 + rng.normal(0, 1.5, 2)).astype(int)
    img[c[0], c[1]] = rng.uniform(20, 60)
    img = np.arcsinh(gaussian_filter(img, 1.2) / 0.3)
    img = img / img.max() * rng.uniform(0.6, 1.0)
    return (img + rng.normal(0, 0.05, img.shape)).astype(np.float32)

def merger(rng):                                    # two smooth galaxies side by side
    a, b = make_galaxy(0, rng), make_galaxy(0, rng)
    return np.maximum(a, np.roll(b, rng.integers(10, 16), axis=1))

rng = np.random.default_rng(11)
X_star = np.array([point_source(rng) for _ in range(200)])
X_merge = np.array([merger(rng) for _ in range(200)])

members = [models['CNN']]                           # in class: member 0 = the CNN of period 2 (seed 0)
print('member 0', np.mean(predict(members[0], X_test).argmax(1) == y_test).round(3))
for s in (1, 2):                                    # in class: two more members (Listing 11.9: range(5))
    torch.manual_seed(s)
    members.append(GalaxyCNN())
    if os.path.exists(f"outputs/ens_member_{s}.pt"):   # trained before period 4 (end of period 3)
        members[-1].load_state_dict(torch.load(f"outputs/ens_member_{s}.pt"))
    else:                                           # not found: train now, about 12 min each on Colab
        fit(members[-1], X_train, y_train, X_val, y_val, seed=s)
        torch.save(members[-1].state_dict(), f"outputs/ens_member_{s}.pt")
    print("member", s, np.mean(predict(members[-1], X_test).argmax(1) == y_test).round(3))

def ensemble(X):
    P = np.mean([predict(m, X) for m in members], axis=0)
    return P, -(P * np.log(P + 1e-12)).sum(axis=1)   # mean probability, entropy

P, H = ensemble(X_test)
ok = P.argmax(1) == y_test
print("ensemble accuracy", ok.mean().round(3),
      "| entropy: correct", H[ok].mean().round(3), "wrong", H[~ok].mean().round(3))
print("test images with max prob > 0.9:", np.mean(P.max(1) > 0.9).round(3))
for name, X in [('stars', X_star), ('mergers', X_merge)]:
    Pq, Hq = ensemble(X)
    print(name, "max prob > 0.9:", np.mean(Pq.max(1) > 0.9).round(2),
          "| predicted classes:", np.bincount(Pq.argmax(1), minlength=5))

print(f"ensemble of {len(members)} members ready in {time.time() - _t0:.0f} s")

### การ์ดแบบจำลอง (อ่านระหว่างรอโค้ดที่ 11.9)
แบบจำลองที่ฝึกเสร็จเป็นเครื่องมือวัดชนิดหนึ่ง เครื่องมือวัดในห้องปฏิบัติการต้องมีคู่มือที่บอกช่วงการวัด ความแม่นยำ เงื่อนไขการใช้ และการปรับเทียบ แบบจำลองก็ต้องมีเอกสารแบบเดียวกัน มิตเชลล์และคณะ (Mitchell et al., 2019) เสนอการ์ดแบบจำลอง (model card) เป็นเอกสารสั้นที่มาพร้อมแบบจำลองทุกตัว ตารางถัดไปเป็นตัวอย่างการ์ดของโครงข่ายในบทนี้ (ตัวเลขจากตำรา) ขอให้สังเกตว่าส่วนที่มีประโยชน์ที่สุดของการ์ดไม่ใช่ตัวเลขความแม่นยำ แต่เป็นส่วนที่บอกว่า **ไม่ควร** ใช้แบบจำลองกับอะไร

| หัวข้อ | เนื้อหา |
|---|---|
| รายละเอียดแบบจำลอง | `GalaxyCNN` คอนโวลูชันสี่ชั้นกับการรวมแบบเฉลี่ยทั้งหมด 60,549 พารามิเตอร์ ฝึกด้วย Adam 30 เอพอก เมล็ดสุ่ม 0 ถึง 4 สำหรับการรวมกลุ่ม |
| การใช้งานที่ตั้งใจ | จำแนกภาพขาวดำ $64\times64$ ของกาแล็กซีที่อยู่กลางภาพโดยประมาณเป็นห้าประเภท เพื่อการเรียนการสอน |
| นอกขอบเขต | ภาพจริงจากการสำรวจท้องฟ้า ภาพหลายแถบสี ดาวฤกษ์ กาแล็กซีที่กำลังรวมตัว วัตถุที่ถูกเลนส์ความโน้มถ่วง ภาพที่มีวัตถุหลายดวง |
| ข้อมูลฝึก | ภาพจำลอง 5,000 ภาพจากโพรไฟล์ของเซอร์ซิก จานเอกซ์โพเนนเชียล แขนเกลียวลอการิทึม และคาน ประเภทละเท่ากัน |
| ผลการประเมิน | ความแม่นยำร้อยละ 95.9 กับภาพจำลองทดสอบ 2,000 ภาพ (95.3 ถึง 99.0 เมื่อเปลี่ยนเมล็ดสุ่ม) และร้อยละ 97.9 สำหรับการรวมกลุ่มห้าตัว ความสับสนหลักคือซิการ์กับจานที่เห็นด้านข้างและกังหันมีคาน |
| ข้อจำกัดที่รู้ | ให้ความน่าจะเป็นสูงกับดาวฤกษ์และกาแล็กซีคู่ ทั้งที่ไม่ใช่ประเภทใดของมัน ไวต่อจุดสว่างที่สัมพันธ์กับป้ายกำกับในข้อมูลฝึก |
| ข้อพิจารณาเชิงจริยธรรม | ไม่มีข้อมูลส่วนบุคคล ถ้าฝึกใหม่ด้วย Galaxy10 ต้องอ้างถึงชุดข้อมูลและอาสาสมัครของ Galaxy Zoo ตามสัญญาอนุญาต |

คำถามอภิปรายระหว่างรอ: ถ้าจะเขียนการ์ดของโครงข่ายที่ฝึกบน Colab ในคาบนี้ แถวใดต้องแก้ตัวเลข และแถวใดไม่ต้องแก้ เพราะเหตุใด

### จริยธรรมและสัญญาอนุญาตของข้อมูล (อ่านระหว่างรอโค้ดที่ 11.9)
ข้อมูลทางวิทยาศาสตร์ส่วนใหญ่ไม่มีข้อมูลส่วนบุคคล แต่ไม่ได้หมายความว่าไม่มีประเด็นเชิงจริยธรรม

- **สัญญาอนุญาต:** Galaxy10 DECaLS เผยแพร่ภายใต้ CC BY 4.0 ซึ่งอนุญาตให้ใช้และดัดแปลงได้โดยต้องแสดงที่มา การแสดงที่มาที่ถูกต้องครอบคลุมทั้งผู้จัดทำชุดข้อมูล โครงการสำรวจที่ถ่ายภาพ และโครงการ Galaxy Zoo ซึ่งป้ายกำกับมาจากแรงงานอาสาของประชาชนจำนวนมาก
- **การคัดเลือกข้อมูล:** ภาพที่อาสาสมัครเห็นไม่ตรงกันถูกคัดออกไปจากชุดข้อมูล ผลคือชุดข้อมูลมีแต่ภาพที่ "ง่าย" กว่าท้องฟ้าจริง ตัวเลขความแม่นยำที่ได้จึงสูงกว่าที่จะได้เมื่อใช้กับการสำรวจทั้งหมด
- **ข้อตกลงกับผู้ให้ตัวอย่าง:** ตัวอย่างเมล็ดข้าวในงานของผู้เขียนได้รับความอนุเคราะห์จากศูนย์วิจัยข้าวและศูนย์เมล็ดพันธุ์ข้าว การเผยแพร่ภาพหรือแบบจำลองต่อต้องคำนึงถึงข้อตกลงนั้น
- **ผลต่อคนจริง:** ถ้าแบบจำลองถูกใช้ตัดสินคุณภาพของข้าวที่ส่งผลต่อรายได้ของเกษตรกร ความผิดพลาดของมันก็มีผลต่อคนจริง

บทที่ 12 จะรวมประเด็นเหล่านี้เป็นกรอบที่สมบูรณ์ พร้อมการ์ดชุดข้อมูล (dataset card) ซึ่งเป็นเอกสารคู่กับการ์ดแบบจำลอง

### ผลของการรวมกลุ่ม (อ่านเมื่อโค้ดที่ 11.9 รันเสร็จ)
ตัวเลขในย่อหน้านี้มาจากโค้ดที่ 11.9 ฉบับเต็มห้าตัวในตำรา ฉบับย่อสามตัวในชั้นเรียนได้ผลในทิศเดียวกัน ในตำรา สมาชิกทั้งห้าตัวมีความแม่นยำร้อยละ 95.2 ถึง 99.0 ความต่างระหว่างสมาชิกถึงสี่จุดร้อยละนี้เกิดจากเมล็ดสุ่มอย่างเดียว ซึ่งเตือนว่าตัวเลขจากการฝึกครั้งเดียวไม่ควรรายงานโดยไม่มีความไม่แน่นอนกำกับ การรวมกลุ่มให้ความแม่นยำร้อยละ 97.9 สูงกว่าสมาชิกเกือบทุกตัว และเอนโทรปีของการทำนายเฉลี่ยของภาพที่ทายผิดคือ 0.68 เทียบกับ 0.14 ของภาพที่ทายถูก การรวมกลุ่มจึงชี้ภาพที่ยากได้ดี ภาพที่เอนโทรปีสูงควรส่งให้ผู้เชี่ยวชาญตรวจ ซึ่งเป็นการแบ่งงานระหว่างแบบจำลองกับมนุษย์ที่ใช้กันในการสำรวจท้องฟ้าขนาดใหญ่

ภาพนอกการแจกแจงให้ผลตรงข้ามกับที่หวัง ในตำรา ดาวฤกษ์ทั้ง 200 ภาพถูกทายเป็นกาแล็กซีเรียบรูปซิการ์ และร้อยละ 94 ได้ความน่าจะเป็นสูงสุดเกิน 0.9 กาแล็กซีคู่ 192 จาก 200 ภาพถูกทายเป็นกาแล็กซีเรียบกลม และร้อยละ 88 ได้ความน่าจะเป็นเกิน 0.9 เทียบกับภาพทดสอบปกติซึ่งมีเพียงร้อยละ 83 ที่ได้ความน่าจะเป็นเกิน 0.9 การรวมกลุ่มจึง **มั่นใจกับภาพที่มันไม่เคยเห็นพอ ๆ กับหรือมากกว่าภาพที่มันถูกฝึกมา** เมื่อรันบนคอมพิวเตอร์อีกเครื่อง ตัวเลขเปลี่ยนไปบ้าง แต่ดาวฤกษ์ทั้ง 200 ภาพยังถูกทายเป็นซิการ์ และกาแล็กซีคู่ 196 ภาพยังถูกทายเป็นเรียบกลม สมาชิกทั้งห้าตัวเห็นตรงกัน เพราะทุกตัวเรียนลักษณะเด่นคล้ายกัน ดาวฤกษ์เป็นจุดสว่างเล็ก ๆ ที่ถูกเบลอเป็นรูปรีเล็กน้อยจากสัญญาณรบกวน ส่วนกาแล็กซีคู่มีโพรไฟล์เรียบแบบเดียวกับกาแล็กซีเรียบกลมทุกประการ

เซลล์ถัดไปวาดการแจกแจงของเอนโทรปีสำหรับภาพทดสอบที่ทายถูก ภาพทดสอบที่ทายผิด ดาวฤกษ์ และกาแล็กซีคู่ พร้อมตัวอย่างภาพนอกการแจกแจง

In [ ]:
H_star, H_merge = ensemble(X_star)[1], ensemble(X_merge)[1]
fig, ax = plt.subplots(1, 3, figsize=(11, 3), gridspec_kw={"width_ratios": [2.2, 1, 1]})
bins = np.linspace(0, np.log(5), 30)
for h_, lab in [(H[ok], "test, correct"), (H[~ok], "test, wrong"), (H_star, "stars (OOD)"), (H_merge, "mergers (OOD)")]:
    ax[0].hist(h_, bins=bins, histtype="step", density=True, lw=1.5, label=f"{lab} (mean {h_.mean():.2f})")
ax[0].set(xlabel="entropy of the ensemble prediction (max ln 5 = 1.61)", ylabel="density",
          title="deep ensemble of 5 CNNs (SYNTHETIC)"); ax[0].legend(fontsize=7)
for a_, X_, t in [(ax[1], X_star, "star"), (ax[2], X_merge, "merger")]:
    a_.imshow(X_[0], cmap="gray", origin="lower"); a_.axis("off")
    a_.set_title(f"{t}: called {CLASSES[ensemble(X_[:1])[0].argmax()]}", fontsize=8)
fig.tight_layout(); plt.show()

### ✏️ แบบฝึก 11.G (5 นาที) ความมั่นใจกับภาพที่ไม่เคยเห็น
ใช้ฟังก์ชัน `ensemble` ของโค้ดที่ 11.9 คำนวณสัดส่วนของภาพที่ความน่าจะเป็นสูงสุดเกิน 0.9 สำหรับภาพทดสอบปกติ เก็บใน `my_conf_test` และสำหรับดาวฤกษ์ เก็บใน `my_conf_star` แล้วเก็บชื่อประเภทที่การรวมกลุ่มทายดาวฤกษ์บ่อยที่สุดใน `my_star_class` (ใช้ `CLASSES`)

In [ ]:
my_conf_test = None    # ✏️ fraction of X_test with max ensemble probability > 0.9
my_conf_star = None    # ✏️ the same for X_star
my_star_class = None   # ✏️ most frequent predicted class name for the stars

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_conf_star', 'my_conf_test', 'my_star_class'])
_Pt, _Ps = ensemble(X_test)[0], ensemble(X_star)[0]
_ct, _cs = np.mean(_Pt.max(1) > 0.9), np.mean(_Ps.max(1) > 0.9)
_sc = CLASSES[np.bincount(_Ps.argmax(1), minlength=5).argmax()]
_say("test  , max prob > 0.9:", "(ยังไม่ได้ตอบ)" if my_conf_test is None else ("✔" if abs(my_conf_test - _ct) < 1e-6 else "✘"), f"— {_ct:.3f}")
_say("stars , max prob > 0.9:", "(ยังไม่ได้ตอบ)" if my_conf_star is None else ("✔" if abs(my_conf_star - _cs) < 1e-6 else "✘"), f"— {_cs:.3f}")
_say("stars called          :", "(ยังไม่ได้ตอบ)" if my_star_class is None else ("✔" if my_star_class == _sc else "✘"), f"— {_sc}")

In [ ]:
#@title แนวคำตอบ 11.G
_say = _gate(['my_conf_star', 'my_conf_test', 'my_star_class'])
_say('''แนวคำตอบ 11.G
my_conf_test = np.mean(ensemble(X_test)[0].max(1) > 0.9) ; my_conf_star = np.mean(ensemble(X_star)[0].max(1) > 0.9)
my_star_class = CLASSES[np.bincount(ensemble(X_star)[0].argmax(1)).argmax()]
ในตำรา: ภาพทดสอบ 0.83 ดาวฤกษ์ 0.94 และทายดาวฤกษ์ทั้ง 200 ภาพเป็น smooth cigar
บนเครื่องอื่นสัดส่วนอาจเปลี่ยน แต่ดาวฤกษ์ยังได้ความมั่นใจพอ ๆ กับหรือสูงกว่าภาพที่โครงข่ายถูกฝึกมา
และเอนโทรปีของดาวฤกษ์ต่ำกว่าของภาพทดสอบที่ทายผิดมาก การรวมกลุ่มจึงไม่เตือนเมื่อพบวัตถุที่ไม่เคยเห็น''')

> **ความเข้าใจคลาดเคลื่อนที่พบบ่อย:** "ความน่าจะเป็นสูงสุดจากซอฟต์แมกซ์ใกล้หนึ่ง แปลว่าโครงข่ายมั่นใจอย่างมีเหตุผล" ซอฟต์แมกซ์บังคับให้ความน่าจะเป็นของประเภทที่รู้จักรวมกันเป็นหนึ่งเสมอ ไม่มีช่องสำหรับคำตอบ "ไม่ใช่ประเภทใดเลย" เมื่อโครงข่ายเห็นภาพที่ไม่เหมือนข้อมูลฝึก มันยังต้องแบ่งความน่าจะเป็นให้ประเภทที่รู้จัก และมักให้ประเภทที่ภาพคล้ายที่สุดในแง่ของลักษณะเด่นที่มันใช้ด้วยความน่าจะเป็นสูง การรวมกลุ่มช่วยได้เฉพาะเมื่อสมาชิกเห็นต่างกัน ซึ่งไม่เกิดขึ้นเมื่อทุกตัวเรียนลักษณะเด่นเดียวกัน บทเรียนเดียวกันนี้ปรากฏมาแล้วในป่าสุ่มนอกช่วงข้อมูลในบทที่ 9 วิธีรับมือที่ใช้ได้จริงคือการคัดกรองข้อมูลเข้าด้วยเกณฑ์ทางฟิสิกส์ก่อน เช่น แยกดาวฤกษ์ด้วยความกว้างของภาพเทียบกับฟังก์ชันการกระจายของจุด หรือเพิ่มประเภท "อื่น ๆ" ที่มีตัวอย่างจริงของวัตถุที่ไม่ใช่เป้าหมายในข้อมูลฝึก

In [ ]:
#@title คำถามทบทวนข้อ 7 (แนวคำตอบ)
print('''คำถามทบทวนข้อ 7: ทำไมการรวมกลุ่มโครงข่ายจึงไม่เตือนเมื่อพบดาวฤกษ์หรือกาแล็กซีคู่ และควรรับมืออย่างไร''')
_say = _gate([])
_say('''คำถามทบทวนข้อ 7 — แนวคำตอบ: ซอฟต์แมกซ์ต้องแบ่งความน่าจะเป็นให้ประเภทที่รู้จักเสมอ และสมาชิกทุกตัวเรียนลักษณะเด่นคล้ายกัน
จึงเห็นตรงกันว่าภาพใหม่คล้ายประเภทใดมากที่สุด ความกระจายระหว่างสมาชิกจึงเล็ก ควรรับมือด้วยการคัดกรอง
ด้วยเกณฑ์ทางฟิสิกส์ก่อน หรือเพิ่มประเภท "อื่น ๆ" ที่มีตัวอย่างจริงในข้อมูลฝึก''')

### สังเคราะห์: สิ่งที่คาบทั้งสี่แสดง

| คำถาม | สิ่งที่วัดได้ในโน้ตบุ๊กนี้ | บทเรียน |
|---|---|---|
| โครงสร้างช่วยอะไร | CNN ใช้พารามิเตอร์น้อยกว่าเกือบเก้าเท่า และแทบไม่เสียความแม่นยำเมื่อภาพเลื่อน ขณะที่ MLP เสียไปกว่าครึ่ง | ความสมมาตรต่อการเลื่อนเป็นข้อจำกัดแบบแข็ง ส่วนการหมุนต้องสอนด้วยการเพิ่มข้อมูลที่ตรงกับฟิสิกส์ |
| ความผิดพลาดอยู่ที่ใด | เมทริกซ์ความสับสนผิดมากในคู่ที่มีความกำกวมทางกายภาพ | อ่านความผิดพลาดด้วยฟิสิกส์ของภาพ |
| ตัวเลขที่รายงานเชื่อได้แค่ไหน | ค่าสูงสุดกับข้อมูลตรวจสอบสูงกว่าข้อมูลทดสอบหลายจุด สมาชิกต่างกันหลายจุดจากเมล็ดสุ่ม | รายงานข้อมูลทดสอบที่ไม่ได้ใช้เลือก พร้อมช่วงความไม่แน่นอน |
| เมื่อข้อมูลน้อย | การถ่ายโอนการเรียนรู้ช่วย แต่เส้นฐานทางกายภาพดีกว่า | เทียบกับเส้นฐานทางกายภาพที่ดีที่สุดทุกครั้ง |
| โครงข่ายมองอะไร | Grad-CAM จับทางลัดที่มุมภาพได้ | ใช้หาปัญหา ไม่ใช่ยืนยันความถูกต้อง ยืนยันด้วยการทดลองควบคุม |
| โครงข่ายรู้ตัวหรือไม่ว่าไม่รู้ | การรวมกลุ่มมั่นใจกับดาวฤกษ์และกาแล็กซีคู่ | คัดกรองข้อมูลเข้าด้วยเกณฑ์ทางฟิสิกส์ และบันทึกขอบเขตในการ์ดแบบจำลอง |

บทที่ 12 จะรวมเครื่องมือทั้งหมดของตำราเข้าเป็นโครงงานเดียว พร้อมกรอบของการทำซ้ำได้และจริยธรรม ความต่างของตัวเลขระหว่างเครื่องที่พบในบทนี้และบทที่ 10 จะเป็นตัวอย่างจริงว่าการกำหนดเมล็ดสุ่มอย่างเดียวไม่รับประกันตัวเลขเดิมข้ามเครื่อง

---
# ส่วนที่ 8 งานนอกชั้น: E11.1–E11.7 ส่วนที่เหลือ

ส่วนนี้ทำนอกเวลาเรียน (รวมราว 7 ชั่วโมงตามคู่มือผู้สอน ซึ่ง 3–4 ชั่วโมงเป็นโครงงาน E11.4 เลือกทำตามที่ผู้สอนมอบหมาย) ต้องรันส่วนที่ 0 ก่อน (คลิกเซลล์หัวคาบที่ 1 แล้วเลือก Runtime → Run before) จากนั้นรันเซลล์เตรียมถัดไป ซึ่งเตรียมภาพกาแล็กซีจำลอง โครงข่ายที่ฝึกแล้ว และฟังก์ชัน `grad_cam` ของโค้ดที่ 11.8 (ถ้ายังไม่มีไฟล์ใน `outputs/` จะฝึกใหม่ ⏱ ราวสิบนาที) แต่ละข้อจึงทำได้โดยไม่ต้องรันคาบที่ 1–4 ใหม่

In [ ]:
#@title เตรียมโค้ดที่ 11.1–11.3 โครงข่ายที่ฝึกแล้ว และ grad_cam ของโค้ดที่ 11.8 (ดับเบิลคลิกเพื่อดูโค้ด)
import os, time
import numpy as np
# ---- Listing 11.1: GalaxyCNN and MLP
import torch
import torch.nn as nn

class GalaxyCNN(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 64 -> 32
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 32 -> 16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 16 -> 8
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU())                   # 64 maps, 8x8
        self.head = nn.Linear(64, n_classes)
    def forward(self, x):
        return self.head(self.features(x).mean(dim=(2, 3)))   # global average pooling

class MLP(nn.Module):                                           # for comparison
    def __init__(self, n_classes=5):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(),
                                 nn.Linear(64 * 64, 128), nn.ReLU(),
                                 nn.Linear(128, 64), nn.ReLU(),
                                 nn.Linear(64, n_classes))
    def forward(self, x):
        return self.net(x)

# ---- Listing 11.2: toy galaxies (from outputs/galaxies.npz if saved in period 2)
import numpy as np
from scipy.ndimage import gaussian_filter
CLASSES = ['smooth round', 'smooth cigar', 'edge-on disk', 'spiral', 'barred spiral']

def make_galaxy(cls, rng, size=64):
    """One noisy 64x64 image of a toy galaxy of class cls (0-4)."""
    y, x = np.mgrid[:size, :size] - (size - 1) / 2
    x0, y0 = rng.normal(0, 1.5, 2)                     # small centring error
    x, y = x - x0, y - y0
    pa = rng.uniform(0, np.pi)                          # position angle
    xr = x * np.cos(pa) + y * np.sin(pa)
    yr = -x * np.sin(pa) + y * np.cos(pa)
    def sersic(re, n, q):
        r = np.sqrt(xr**2 + (yr / q)**2)
        b = 2 * n - 1 / 3
        return np.exp(-b * ((r / re)**(1 / n) - 1))
    if cls == 0:
        img = sersic(rng.uniform(6, 10), 4, rng.uniform(0.8, 1.0))
    elif cls == 1:
        img = sersic(rng.uniform(6, 10), rng.uniform(1, 4), rng.uniform(0.2, 0.4))
    elif cls == 2:
        img = sersic(rng.uniform(10, 15), 1, rng.uniform(0.08, 0.15)) \
              + 0.3 * sersic(2.5, 2, 0.6)
    else:
        q = rng.uniform(0.45, 1.0)                      # cos(inclination)
        re = rng.uniform(10, 15)
        r = np.sqrt(xr**2 + (yr / q)**2)
        theta = np.arctan2(yr / q, xr)
        pitch = rng.uniform(0.25, 0.45)                 # arm winding
        arms = 0.5 * (1 + np.cos(2 * (theta - np.log(r + 1e-3) / np.tan(pitch))))
        arms = arms * (1 - np.exp(-(r / (0.4 * re))**2))  # arms start outside the centre
        disk = np.exp(-r / (re / 1.68)) * (0.4 + 1.2 * arms**2)
        img = disk + 1.5 * sersic(2.0, 2, 0.7 + 0.3 * q) / np.exp(11 / 3)   # bulge, rounder than disk
        if cls == 4:                                     # bar along the disk
            bar_len = rng.uniform(0.4, 0.6) * re
            img += 1.2 * np.exp(-(xr / bar_len)**4 - (yr / (0.3 * bar_len))**2)
    img = np.arcsinh(img / 0.3)                        # astronomical asinh stretch
    img = img / img.max() * rng.uniform(0.6, 1.0)
    img = gaussian_filter(img, rng.uniform(0.8, 1.5))   # seeing (PSF)
    img = img + rng.normal(0, 0.05, img.shape)          # sky + read noise
    return img.astype(np.float32)

def make_galaxy_set(n_per_class, seed=0):
    rng = np.random.default_rng(seed)
    X = np.array([make_galaxy(c, rng) for c in range(5) for _ in range(n_per_class)])
    y = np.repeat(np.arange(5), n_per_class)
    p = rng.permutation(len(y))
    return X[p], y[p]

if os.path.exists("outputs/galaxies.npz"):
    _d = np.load("outputs/galaxies.npz")
    X_train, y_train, X_val, y_val, X_test, y_test = (_d[k] for k in ("X_train", "y_train", "X_val", "y_val", "X_test", "y_test"))
else:
    X_train, y_train = make_galaxy_set(1000, seed=1)
    X_val, y_val = make_galaxy_set(250, seed=2)
    X_test, y_test = make_galaxy_set(400, seed=3)
    np.savez_compressed("outputs/galaxies.npz", X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
                        X_test=X_test, y_test=y_test)
print("toy galaxies (SYNTHETIC):", X_train.shape, X_val.shape, X_test.shape)

# ---- Listing 11.3: augment, fit, predict, and the trained MLP and CNN
def augment(xb):
    """Random 90-degree rotation and mirror flip: exact symmetries of the pixel grid."""
    xb = torch.rot90(xb, np.random.randint(4), dims=(2, 3))
    if np.random.rand() < 0.5:
        xb = torch.flip(xb, dims=(3,))
    return xb

def fit(model, X, y, X_val, y_val, epochs=30, lr=2e-3, aug=False, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(X[:, None]), torch.tensor(y)),
        batch_size=64, shuffle=True)
    Xv = torch.tensor(X_val[:, None])
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    val_acc = []
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            if aug:
                xb = augment(xb)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val_acc.append(np.mean(model(Xv).argmax(1).numpy() == y_val))
    return val_acc

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(torch.tensor(X[:, None])), dim=1).numpy()

models = {}
for name, Model, lr, epochs in [('MLP', MLP, 1e-3, 15), ('CNN', GalaxyCNN, 2e-3, 30)]:
    torch.manual_seed(0)
    models[name] = Model()
    if os.path.exists(f"outputs/{name}.pt"):
        models[name].load_state_dict(torch.load(f"outputs/{name}.pt"))
        print(f"loaded outputs/{name}.pt")
    else:
        print(f"outputs/{name}.pt not found: training {name} as in Listing 11.3 ...")
        _t0 = time.time()
        fit(models[name], X_train, y_train, X_val, y_val, epochs=epochs, lr=lr)
        torch.save(models[name].state_dict(), f"outputs/{name}.pt")
        print(f"  trained and saved outputs/{name}.pt ({time.time() - _t0:.0f} s)")
    models[name].eval()

# ---- Listing 11.8: grad_cam only
import torch.nn.functional as F

def grad_cam(model, x, target):
    """Grad-CAM map (64 x 64, unit sum) of class `target` for one image x (1,1,64,64)."""
    model.eval()
    A = model.features(x)                          # last feature maps (1, 64, 8, 8)
    A.retain_grad()
    score = model.head(A.mean(dim=(2, 3)))[0, target]
    model.zero_grad()
    score.backward()
    w = A.grad.mean(dim=(2, 3), keepdim=True)      # importance of each feature map
    cam = F.relu((w * A).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)
    cam = cam[0, 0].detach().numpy()
    return cam / (cam.sum() + 1e-12)

print('ready: GalaxyCNN, MLP, X_train ... y_test, fit, predict, models, grad_cam')

### ✏️ E11.1 โครงข่ายคอนโวลูชันกับเมทริกซ์ระยะห่างของสายพอลิเมอร์ (40 นาที)
ใช้ข้อมูลสายพอลิเมอร์จากโค้ดที่ 10.3 สร้างเมทริกซ์ระยะห่าง $30\times30$ ของแต่ละโครงแบบ ฝึกโครงข่ายคอนโวลูชันขนาดเล็กด้วยวิธีเรียนรู้จากปลายทั้งสองข้าง แล้วหาจุดที่ความน่าจะเป็นเฉลี่ยเท่ากับ 0.5 เปรียบเทียบจำนวนพารามิเตอร์ ความแม่นยำกับข้อมูลตรวจสอบ และจุดตัดกับโครงข่ายหลายชั้นของบทที่ 10 (ตาราง tab:polymer-nn ซึ่งมีพารามิเตอร์ 32,194 ตัว) แล้วอธิบายว่าการมองข้อมูลเป็นภาพช่วยหรือไม่ช่วยเพราะเหตุใด

เซลล์ซ่อนถัดไปอ่านข้อมูลสายพอลิเมอร์จาก `outputs/chain.npz` (ถ้ายังไม่มี จะรันโค้ดที่ 10.3 ใหม่ ⏱ ราวเจ็ดนาทีบน Colab) และนิยาม `train_phase_classifier` กับ `crossing` ของโค้ดที่ 10.5 ในรุ่นที่ใช้โครงข่ายคอนโวลูชัน `MapCNN` แทนโครงข่ายหลายชั้น (สามชั้นคอนโวลูชัน 8, 16 และ 32 ตัวกรอง กับการรวมแบบเฉลี่ยทั้งหมด)

ข้อมูลสายพอลิเมอร์ไวต่อการปัดเศษของเครื่องคำนวณ (ดูบทที่ 10) บนเครื่องอื่นจึงอาจได้จุดตัดต่างจากตำราราว 0.1

In [ ]:
#@title ข้อมูลสายพอลิเมอร์ (โค้ดที่ 10.3) และโค้ดที่ 10.5 ที่ใช้ MapCNN (ดับเบิลคลิกเพื่อดูโค้ด)
import time
if os.path.exists("outputs/chain.npz"):
    _d = np.load("outputs/chain.npz")
    conf, ener, T_lab, T_grid = _d["conf"], _d["ener"], _d["T_lab"], _d["T_grid"]
    print("loaded outputs/chain.npz", conf.shape)
else:
    print("outputs/chain.npz not found: running Listing 10.3 (about 7 minutes on Colab) ...")
    _t0 = time.time()
    import numpy as np

    def make_chain_ensemble(N=30, T=2.0, n_samples=400, sweeps_between=10,
                            n_equil=2000, seed=0):
        """Flexible chain: bond length 1, Lennard-Jones (eps = sigma = 1) between
        beads with |i - j| >= 2. Metropolis sampling with pivot and crankshaft moves.
        Returns centred configurations (n_samples, N, 3) and energies."""
        rng = np.random.default_rng(seed)
        iu, ju = np.triu_indices(N, k=2)
        def energy(r):
            s6 = 1.0 / np.sum((r[iu] - r[ju])**2, axis=1)**3
            return np.sum(4.0 * (s6 * s6 - s6))
        def rotation(axis, angle):                         # Rodrigues' formula
            a = axis / np.linalg.norm(axis)
            K = np.array([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]])
            return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * K @ K
        r = np.zeros((N, 3)); r[:, 0] = np.arange(N)       # straight initial chain
        E, beta = energy(r), 1.0 / T
        confs, energies = [], []
        for m in range((n_equil + n_samples * sweeps_between) * N):
            new = r.copy()
            if rng.random() < 0.5:                         # pivot move
                k = rng.integers(1, N - 1)
                R = rotation(rng.normal(size=3), rng.uniform(-np.pi, np.pi) * 0.5)
                if rng.random() < 0.5:
                    new[k+1:] = (r[k+1:] - r[k]) @ R.T + r[k]
                else:
                    new[:k] = (r[:k] - r[k]) @ R.T + r[k]
            else:                                          # crankshaft or end move
                i = rng.integers(0, N)
                if 0 < i < N - 1:
                    R = rotation(r[i+1] - r[i-1], rng.uniform(-np.pi, np.pi))
                    new[i] = (r[i] - r[i-1]) @ R.T + r[i-1]
                else:
                    j = 1 if i == 0 else N - 2
                    v = rng.normal(size=3)
                    new[i] = r[j] + v / np.linalg.norm(v)
            E_new = energy(new)
            if E_new <= E or rng.random() < np.exp(-beta * (E_new - E)):
                r, E = new, E_new
            sweep, rest = divmod(m + 1, N)
            if rest == 0 and sweep > n_equil and (sweep - n_equil) % sweeps_between == 0:
                confs.append(r - r.mean(axis=0))
                energies.append(E)
        return np.array(confs), np.array(energies)

    T_grid = np.round(np.arange(0.6, 3.41, 0.2), 1)       # 15 temperatures
    data = [make_chain_ensemble(T=T, seed=100 + k) for k, T in enumerate(T_grid)]
    conf = np.concatenate([d[0] for d in data])            # (6000, 30, 3)
    ener = np.concatenate([d[1] for d in data])
    T_lab = np.repeat(T_grid, 400)

    np.savez("outputs/chain.npz", conf=conf, ener=ener, T_lab=T_lab, T_grid=T_grid)
    print(f"saved outputs/chain.npz {conf.shape} ({time.time() - _t0:.0f} s)")

D = np.linalg.norm(conf[:, :, None, :] - conf[:, None, :, :], axis=3).astype(np.float32)   # (6000, 30, 30)

class MapCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 30 -> 15
                               nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 15 -> 7
                               nn.Conv2d(16, 32, 3, padding=1), nn.ReLU())
        self.h = nn.Linear(32, 2)

    def forward(self, x):
        return self.h(self.f(x.view(-1, 1, 30, 30)).mean(dim=(2, 3)))

# Listing 10.5 with the MLP replaced by MapCNN
POLYNN_SRC = '''def train_phase_classifier(F, T, T_low, T_high, seed=0, hidden=64, epochs=60):
    """Learning by extremes: label T <= T_low as globule (1), T >= T_high as coil (0)."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    train = (T <= T_low) | (T >= T_high)
    y = (T <= T_low).astype(np.int64)
    mu, sd = F[train].mean(axis=0), F[train].std(axis=0) + 1e-9
    X_all = torch.tensor((F - mu) / sd, dtype=torch.float32)
    y_all = torch.tensor(y)
    idx = rng.permutation(np.where(train)[0])
    i_tr, i_va = idx[:int(0.8 * len(idx))], idx[int(0.8 * len(idx)):]
    model = nn.Sequential(nn.Linear(F.shape[1], hidden), nn.ReLU(), nn.Dropout(0.2),
                          nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.2),
                          nn.Linear(hidden, 2))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        perm = rng.permutation(i_tr)
        for b in range(0, len(perm), 64):
            bi = perm[b:b + 64]
            loss = loss_fn(model(X_all[bi]), y_all[bi])
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        P = torch.softmax(model(X_all), dim=1)[:, 1].numpy()      # P(globule)
    val_acc = np.mean((P[i_va] > 0.5) == y[i_va])
    return model, X_all, P, val_acc

def crossing(P, T, T_grid):
    """Temperature where the mean P(globule) crosses 0.5 (linear interpolation)."""
    Pm = np.array([P[T == t].mean() for t in T_grid])
    k = np.where((Pm[:-1] - 0.5) * (Pm[1:] - 0.5) <= 0)[0][0]
    return T_grid[k] + (0.5 - Pm[k]) * (T_grid[k+1] - T_grid[k]) / (Pm[k+1] - Pm[k]), Pm
'''
_OLD = '''    model = nn.Sequential(nn.Linear(F.shape[1], hidden), nn.ReLU(), nn.Dropout(0.2),
                          nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.2),
                          nn.Linear(hidden, 2))'''
exec(POLYNN_SRC.replace(_OLD, "    model = MapCNN()"))
print("distance matrices:", D.shape, "| ready: MapCNN, train_phase_classifier, crossing")

In [ ]:
idx_ = [np.where(np.isclose(T_lab, t))[0][0] for t in (0.6, 1.6, 3.4)]
vmax = D[idx_].max()                                        # one colour scale for all panels
fig, ax = plt.subplots(1, 3, figsize=(10, 3))
for a_, t, i in zip(ax, (0.6, 1.6, 3.4), idx_):
    im = a_.imshow(D[i], cmap="viridis", origin="upper", vmin=0, vmax=vmax); a_.grid(False)
    a_.set_title(f"distance matrix, T = {t} (SYNTHETIC)", fontsize=9); fig.colorbar(im, ax=a_, shrink=0.8)
fig.tight_layout(); plt.show()

In [ ]:
# ⏱ ราวห้าถึงแปดนาทีบน Colab (ฝึก 9 ครั้ง)
F_map = D.reshape(len(D), -1)                  # (not F: grad_cam uses F = torch.nn.functional)
allT = []
for lo, hi in [(1.0, 3.0), (0.8, 3.2), (1.2, 2.8)]:
    ts, accs = [], []
    for s in range(3):
        _, _, P, a = train_phase_classifier(F_map, T_lab, lo, hi, seed=s)
        ts.append(crossing(P, T_lab, T_grid)[0]); accs.append(a)
    allT += ts
    print(f"ends <= {lo}, >= {hi}: validation {np.round(accs, 3)}, crossings {np.round(ts, 2)}")
print(f"all ends and seeds: {min(allT):.2f}-{max(allT):.2f}")

my_n_cnn = None    # ✏️ number of parameters of MapCNN
my_spread = None   # ✏️ max(allT) - min(allT): spread of the crossing over all ends and seeds

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_n_cnn', 'my_spread'])
_n = sum(q.numel() for q in MapCNN().parameters())
_say("MapCNN parameters:", "(ยังไม่ได้ตอบ)" if my_n_cnn is None else ("✔" if my_n_cnn == _n else "✘"), f"— {_n:,} (MLP of Listing 10.5: 32,194)")
_say("spread of T*     :", "(ยังไม่ได้ตอบ)" if my_spread is None else ("✔" if abs(my_spread - (max(allT) - min(allT))) < 1e-9 else "✘"),
      f"— {max(allT) - min(allT):.2f}")

In [ ]:
#@title แนวคำตอบ E11.1
_say = _gate(['my_n_cnn', 'my_spread'])
_say('''แนวคำตอบ E11.1
- MapCNN มีพารามิเตอร์ 5,954 ตัว เทียบกับ 32,194 ตัวของโครงข่ายหลายชั้นในโค้ดที่ 10.5 น้อยกว่าราวห้าเท่า
- ความแม่นยำกับข้อมูลตรวจสอบของปลายทั้งสามแบบคือ 0.978 ถึง 0.997 ใกล้เคียงกับโครงข่ายหลายชั้น
- จุดตัดราว 1.6, 1.3 และ 1.7 สำหรับปลาย (1.0, 3.0), (0.8, 3.2) และ (1.2, 2.8) และแกว่งระหว่างเมล็ดสุ่ม
  มากกว่าโครงข่ายหลายชั้น เช่น 1.47-1.60 สำหรับปลายแรก (ตัวเลขบนเครื่องอื่นอาจต่างราว 0.1)
- สรุป: การมองข้อมูลเป็นภาพลดพารามิเตอร์ลงโดยความแม่นยำไม่ลด แต่ไม่ได้แก้ปัญหาหลักของบทที่ 10
  จุดตัดยังขึ้นกับการเลือกปลายมาก และช่วงเมื่อรวมทุกปลายและทุกเมล็ดสุ่ม (ราว 1.2 ถึง 1.8) กว้างกว่าเดิมเสียอีก
  เพราะปัญหามาจากความกว้างของการเปลี่ยนในสายสั้น ไม่ใช่จากการเลือกแบบจำลอง''')

### ✏️ E11.2 ส่วนที่เหลือ: เทียบกับแบบในร่างเดิม (20 นาที)
แบบในร่างเดิมรับภาพ $69\times69$ สามแถบสี ใช้คอนโวลูชัน 32, 64 และ 128 ตัวกรอง (แต่ละชั้นตามด้วย ReLU และการรวมแบบสูงสุดขนาด 2) แล้วแผ่แผนที่ขนาด $128\times8\times8$ เข้าชั้นเชื่อมต่อเต็ม 256 หน่วยและ 10 หน่วย เซลล์ถัดไปสร้างแบบนี้ไว้ใน `draft` คำนวณด้วยมือแล้วเก็บจำนวนพารามิเตอร์ของทั้งโครงข่ายใน `my_draft_total` และสัดส่วนที่อยู่ในชั้นเชื่อมต่อเต็มแรกใน `my_dense_share` แล้วอธิบายว่าชั้นใดมีพารามิเตอร์มากที่สุดในแต่ละแบบ และทำไมการรวมแบบเฉลี่ยทั้งหมดจึงดีกว่า

In [ ]:
draft = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                      nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                      nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                      nn.Flatten(), nn.Linear(128 * 8 * 8, 256), nn.ReLU(), nn.Linear(256, 10))
print("draft output shape for a 69x69 image before flattening:", tuple(draft[:9](torch.zeros(1, 3, 69, 69)).shape))

my_draft_total = None   # ✏️ parameters of `draft`
my_dense_share = None   # ✏️ fraction of them in nn.Linear(8192, 256)

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_dense_share', 'my_draft_total'])
_per = [sum(q.numel() for q in l.parameters()) for l in draft if isinstance(l, (nn.Conv2d, nn.Linear))]
_tot = sum(_per); _sh = _per[3] / _tot
_say("parameters per layer:", _per)
_say("total      :", "(ยังไม่ได้ตอบ)" if my_draft_total is None else ("✔" if my_draft_total == _tot else "✘"), f"— {_tot:,}")
_say("dense share:", "(ยังไม่ได้ตอบ)" if my_dense_share is None else ("✔" if abs(my_dense_share - _sh) < 1e-3 else "✘"), f"— {_sh:.3f}")
_g = GalaxyCNN(n_classes=10); _g.features[0] = nn.Conv2d(3, 16, 3, padding=1)
_say(f"ratio draft / 3-band GalaxyCNN = {_tot / sum(q.numel() for q in _g.parameters()):.1f}")

In [ ]:
#@title แนวคำตอบ E11.2
_say = _gate(['my_dense_share', 'my_draft_total'])
_say('''แนวคำตอบ E11.2
แบบในร่างเดิม: คอนโวลูชัน 896, 18,496 และ 73,856 ตัว ชั้นเชื่อมต่อเต็มแรก 8,192 x 256 + 256 = 2,097,408 ตัว
และชั้นสุดท้าย 2,570 ตัว รวม 2,193,226 ตัว ซึ่งร้อยละ 95.6 อยู่ในชั้นเชื่อมต่อเต็มแรกชั้นเดียว
(ภาพ 69 x 69 ผ่านการรวมสามครั้งเหลือ 8 x 8 เพราะปัดเศษลง)
GalaxyCNN สามแถบสีมี 61,162 ตัว ชั้นที่มากที่สุดคือคอนโวลูชันชั้นสุดท้าย แบบเดิมมากกว่าราว 36 เท่า
การแผ่แผนที่ลักษณะเด่นเข้าชั้นเชื่อมต่อเต็มนำปัญหาเดิมของโครงข่ายหลายชั้นกลับมา คือน้ำหนักแยกกันสำหรับแต่ละ
ตำแหน่งบนแผนที่ 8 x 8 ซึ่งเพิ่มพารามิเตอร์มหาศาลและทำให้ผลขึ้นกับตำแหน่ง ส่วนการรวมแบบเฉลี่ยทั้งหมด
ใช้พารามิเตอร์น้อยกว่ามาก และคงความไม่แปรเปลี่ยนต่อการเลื่อนไว้ได้''')

### E11.3 ส่วนที่เหลือ: ปัญหา ผลกระทบ และวิธีแก้ของทุกขั้น (20 นาที)
สำหรับสี่ขั้นของการเตรียมข้อมูลในแบบฝึก E11.3 (ส่วนที่ 3) ให้ระบุปัญหาของแต่ละขั้น บอกว่าปัญหานั้นทำให้ผลการประเมินดีเกินจริงหรือแย่ลง และเสนอวิธีแก้ เขียนคำตอบในเซลล์ข้อความของตนเอง แล้วเทียบกับแนวคำตอบ

In [ ]:
#@title แนวคำตอบ E11.3
_say = _gate(['my_dense_share', 'my_draft_total'])
_say('''แนวคำตอบ E11.3
ขั้นที่ 1: ข้อมูลรั่วไหลเล็กน้อย เพราะสถิติของชุดทดสอบถูกใช้ในการเตรียมข้อมูลฝึก ผลกับการปรับมาตรฐานมักเล็ก
  แต่หลักที่ถูกต้องคือคำนวณสถิติจากชุดฝึกเท่านั้น แล้วนำไปใช้กับชุดอื่น (ผลดีเกินจริงเล็กน้อย)
ขั้นที่ 2: การรั่วไหลที่รุนแรงกว่ามาก โครงข่ายอาจจำกาแล็กซีดวงนั้นได้จากรายละเอียดเฉพาะตัว ผลกับชุดทดสอบจึงดีเกินจริง
  วิธีแก้คือแบ่งตามวัตถุ ไม่ใช่ตามภาพ ซึ่งเป็นการแบ่งตามกลุ่มในบทที่ 9
ขั้นที่ 3: มีสองปัญหา การเพิ่มข้อมูลกับชุดทดสอบทำให้ชุดทดสอบไม่ใช่ตัวแทนของข้อมูลจริง และการปรับความสว่างร้อยละ 30
  อาจทำลายข้อมูลที่มีความหมาย เช่น ความต่างของความสว่างพื้นผิวระหว่างกาแล็กซีรีกับกาแล็กซีกังหัน
  การเพิ่มข้อมูลควรใช้กับชุดฝึกเท่านั้น และเลือกเฉพาะการแปลงที่ไม่เปลี่ยนฟิสิกส์
ขั้นที่ 4: ผิดโดยหลักการ ภาพเวลา-ความถี่มีแกนที่มีความหมายต่างกัน การหมุนภาพสร้างสัญญาณที่ความถี่เปลี่ยนตามเวลา
  ในแบบที่ไม่มีระบบดาวคู่ใดทำได้ โครงข่ายจะเรียนจากข้อมูลที่ขัดกับฟิสิกส์ ผลจึงแย่ลงหรือแปลกไป โดยอาจไม่มีตัวเลขใดเตือน
สรุป: ทุกการแปลงที่ใช้เพิ่มข้อมูลคือการอ้างว่า "คำตอบไม่เปลี่ยนภายใต้การแปลงนี้" ซึ่งต้องมีเหตุผลทางฟิสิกส์รองรับ''')

### E11.4 โครงงานจำแนกกาแล็กซีด้วย Galaxy10 DECaLS (3–4 ชั่วโมง)
ดาวน์โหลด Galaxy10 DECaLS รุ่นที่ตัดภาพซ้ำออก (`Galaxy10_DECals_NoDuplicated.h5` ราว 2.7 GB, CC BY 4.0) จาก Zenodo (https://doi.org/10.5281/zenodo.10845026) แล้วทำตามขั้นต่อไปนี้

1. อ่านด้วยโค้ดที่ 11.6 (ส่วนที่ 4) โดยเปลี่ยนชื่อไฟล์ให้ตรง และห้ามแปลงภาพทั้งหมดเป็นเลขทศนิยมพร้อมกัน (ต้องใช้หน่วยความจำราว 14 GB) บน Colab ให้ดาวน์โหลดไฟล์ไว้ใน Google Drive ก่อน แล้วเชื่อม Drive กับ Colab
2. ดัดแปลง `GalaxyCNN` ให้รับสามแถบสีและสิบประเภทตามแบบฝึก E11.2
3. แบ่งชุดฝึกของโค้ดที่ 11.6 ออกเป็นชุดฝึกย่อยและชุดตรวจสอบ และเก็บชุดทดสอบไว้ใช้ครั้งเดียวตอนท้าย
4. ฝึกด้วยการเพิ่มข้อมูลแบบหมุนและสะท้อน (`fit(..., aug=True)` ของโค้ดที่ 11.3 ใช้ได้เมื่อแปลงข้อมูลให้อยู่ในรูป `(N, 3, 64, 64)` และปรับบรรทัด `X[:, None]` ซึ่งเพิ่มแกนช่องสัญญาณสำหรับภาพขาวดำ)
5. รายงานความแม่นยำ ความแม่นยำแบบสมดุล (`sklearn.metrics.balanced_accuracy_score`) และเมทริกซ์ความสับสน ตีความคู่ประเภทที่สับสนกันมากที่สุดสามคู่ด้วยฟิสิกส์ของรูปร่างและมุมมอง
6. ฝึกโครงข่ายห้าตัวด้วยเมล็ดสุ่มต่างกันเพื่อประเมินความไม่แน่นอน และเขียนรายงานหนึ่งหน้าพร้อมการ์ดแบบจำลอง (E11.7)

การฝึกกับภาพจริง 14,000 ภาพบนหน่วยประมวลผลกลางใช้เวลานานกว่าภาพจำลองหลายเท่า ควรทดลองกับภาพส่วนหนึ่งและเอพอกน้อย ๆ ก่อน

In [ ]:
#@title แนวคำตอบ E11.4
_say = _gate([])
_say('''แนวคำตอบ E11.4
แบบฝึกหัดนี้ไม่มีคำตอบตัวเลขเดียว เพราะผลขึ้นกับขนาดภาพ จำนวนเอพอก และทรัพยากรของเครื่อง ผู้เขียนจึงไม่ระบุความแม่นยำเป้าหมาย
สิ่งที่ควรพบในเชิงคุณภาพ:
- คู่ที่สับสนกันมากมักมีความกำกวมทางกายภาพจริง ได้แก่ เรียบกลมกับเรียบระหว่างกลาง (ระดับความรีที่ต่อเนื่องกัน)
  กังหันแขนแน่นกับแขนหลวม (ช่วงของมุมพิตช์ที่ต่อเนื่องกัน) และถูกรบกวนกับกำลังรวมตัว (ขั้นต่างกันของกระบวนการเดียวกัน)
- ประเภทเรียบรูปซิการ์ซึ่งมีภาพน้อยที่สุดมักมีค่าการเรียกคืนต่ำที่สุด จึงต้องรายงานความแม่นยำแบบสมดุลคู่กับความแม่นยำรวม
- ความแตกต่างระหว่างโครงข่ายทั้งห้าบอกความไม่แน่นอนจากการฝึก ซึ่งควรรายงานเป็นช่วง
- รายงานที่ดีระบุว่าป้ายกำกับมาจากการลงคะแนนของอาสาสมัคร ซึ่งมีความไม่แน่นอนของตัวเอง
  ภาพที่อาสาสมัครเห็นไม่ตรงกันคือภาพที่ยากสำหรับโครงข่ายเช่นกัน''')

### E11.5 ส่วนที่เหลือ: ความแม่นยำเชิงบวกและการตรวจความบริสุทธิ์ (25 นาที)
ต่อจากส่วนในชั้นเรียน (ส่วนที่ 5) ให้อธิบายว่าต้องมีข้อมูลใดเพิ่มจึงจะคำนวณความแม่นยำเชิงบวกของพันธุ์ขาวดอกมะลิ 105 ได้ และถ้าจะใช้แบบจำลองนี้ตรวจว่าข้าวเปลือกหนึ่งกระสอบมีความบริสุทธิ์ถึงร้อยละ 95 หรือไม่ ต้องคำนึงถึงอะไรบ้าง

In [ ]:
#@title แนวคำตอบ E11.5
_say = _gate([])
_say('''แนวคำตอบ E11.5
ความแม่นยำเชิงบวกของขาวดอกมะลิ 105 ต้องรู้ "หลัก" ของเมทริกซ์ความสับสน คือจำนวนเมล็ดพันธุ์อื่นที่ถูกทายเป็นขาวดอกมะลิ 105
ข้อความในบทความบอกเพียงแถวของ กข23 และปทุมธานี 1 บางส่วน จึงต้องใช้เมทริกซ์เต็มจากรูปในบทความ
สำหรับการตรวจความบริสุทธิ์ มีเรื่องที่ต้องคิดอย่างน้อยสามเรื่อง
1 ค่าการเรียกคืนร้อยละ 94 หมายความว่าแม้ข้าวบริสุทธิ์ร้อยละร้อย แบบจำลองก็จะนับเมล็ดราวร้อยละ 6 เป็นพันธุ์อื่น
  ความบริสุทธิ์ที่ประมาณได้โดยตรงจึงต่ำกว่าเกณฑ์ร้อยละ 95 เสมอ ต้องแก้ด้วยเมทริกซ์ความสับสนก่อนสรุป
2 พันธุ์ที่ใช้ปลอมปนจริงอาจไม่ใช่สามพันธุ์ที่แบบจำลองรู้จัก ซึ่งเป็นปัญหาข้อมูลนอกการแจกแจงในส่วนที่ 7
3 ตัวเลขร้อยละ 84 มาจากการเลือกรอบด้วยชุดตรวจสอบ ความแม่นยำจริงกับข้าวใหม่จึงน่าจะต่ำกว่านี้บ้าง
ทั้งหมดนี้ต้องรายงานพร้อมความไม่แน่นอนจากจำนวนเมล็ดที่ตรวจ''')

### ✏️ E11.6 โครงข่ายมองส่วนใดของกาแล็กซีแต่ละประเภท (30 นาที)
ใช้ฟังก์ชัน `grad_cam` ของโค้ดที่ 11.8 กับ `models['CNN']` (ฝึกด้วยข้อมูลสะอาด) คำนวณสัดส่วนของแผนที่ที่อยู่ภายในรัศมี 20 พิกเซลจากกลางภาพ (`disc` ในเซลล์ถัดไป) สำหรับภาพทดสอบ 50 ภาพแรกของแต่ละประเภท โดยใช้ประเภทจริงเป็นเป้าหมาย เก็บค่าเฉลี่ยของทั้งห้าประเภทเป็นลิสต์ใน `my_shares` ตามลำดับ `CLASSES` แล้วเปรียบเทียบกับสัดส่วนพื้นที่ของวงกลม `disc.mean()` ตีความผลของแต่ละประเภทด้วยรูปร่างของกาแล็กซี ผลใดที่ไม่คาดคิด และควรทดลองอะไรต่อเพื่อตรวจว่าผลนั้นเป็นทางลัดหรือไม่

ตัวอย่างการใช้: `grad_cam(models['CNN'], torch.tensor(im[None, None]), c)[disc].sum()` ให้สัดส่วนในวงกลมของภาพ `im` สำหรับประเภท `c`

In [ ]:
yy, xx = np.mgrid[:64, :64]
disc = (yy - 31.5)**2 + (xx - 31.5)**2 < 20**2
print(f"area share of the r < 20 disc: {disc.mean():.3f}")

my_shares = None   # ✏️ list of 5 mean shares (classes 0-4), 50 test images each

In [ ]:
#@markdown ✅ ตรวจคำตอบ
_say = _gate(['my_shares'])
_sh = [np.mean([grad_cam(models['CNN'], torch.tensor(im[None, None]), c)[disc].sum() for im in X_test[y_test == c][:50]])
       for c in range(5)]
for c in range(5):
    _say(f"{CLASSES[c]:14s}: share inside {_sh[c]:.2f}")
_say("my_shares:", "(ยังไม่ได้ตอบ)" if my_shares is None else ("✔" if np.allclose(my_shares, _sh, atol=1e-4) else "✘"))

In [ ]:
#@title แนวคำตอบ E11.6
_say = _gate(['my_shares'])
_say('''แนวคำตอบ E11.6
วงกลมรัศมี 20 พิกเซลกินพื้นที่ราวร้อยละ 31 ของภาพ ในตำรา สัดส่วนของแผนที่ภายในวงกลมคือราวร้อยละ 86 สำหรับกังหัน
ร้อยละ 68 สำหรับจานที่เห็นด้านข้าง และร้อยละ 59 สำหรับกังหันมีคาน ซึ่งสมเหตุสมผล เพราะแขน คาน และจานอยู่ในบริเวณนี้
ผลที่ไม่คาดคิดคือเรียบกลมและเรียบรูปซิการ์ ซึ่งมีสัดส่วนภายในวงกลมเพียงราวร้อยละ 15 และ 22 น้อยกว่าสัดส่วนพื้นที่เสียอีก
แปลว่าโครงข่ายให้น้ำหนักกับบริเวณรอบนอกของภาพ (ตัวเลขบนเครื่องอื่นต่างไปบ้าง แต่รูปแบบนี้มักคงอยู่)
คำอธิบายที่เป็นไปได้ทางฟิสิกส์: โพรไฟล์ n = 4 ของเรียบกลมมีปีกที่ลดลงช้าและกว้างไปถึงขอบภาพ การมีหรือไม่มีแสงจาง ๆ
ที่ขอบจึงเป็นหลักฐานของประเภทนี้ได้จริง แต่ก็เป็นไปได้ว่าโครงข่ายใช้ระดับของท้องฟ้าหรือสัญญาณรบกวนที่ขอบ
การทดลองควบคุมที่ควรทำ: ตัดปีกของโพรไฟล์ที่รัศมีหนึ่งแล้วดูว่าการทำนายเปลี่ยนหรือไม่ หรือแทนบริเวณรอบนอกด้วย
สัญญาณรบกวนล้วน ๆ ถ้าการทำนายเปลี่ยนมาก แสดงว่าโครงข่ายพึ่งปีกของโพรไฟล์ ซึ่งในข้อมูลจริงอาจถูกรบกวนด้วย
ท้องฟ้าที่สว่างหรือกาแล็กซีข้างเคียง''')

### E11.7 เขียนการ์ดแบบจำลอง (30 นาที)
เขียนการ์ดแบบจำลองหนึ่งหน้าสำหรับโครงข่ายที่ฝึกในแบบฝึกหัด E11.4 ด้วย Galaxy10 DECaLS ตามโครงของตารางในส่วนที่ 7 โดยให้ส่วน "นอกขอบเขต" และ "ข้อจำกัดที่รู้" มีเนื้อหาเฉพาะของข้อมูลจริง และระบุว่าข้อจำกัดแต่ละข้อได้มาจากการทดลองหรือเป็นการคาดการณ์ (ถ้ายังไม่ได้ทำ E11.4 ให้เขียนการ์ดของโครงข่ายที่ฝึกบน Colab ในคาบที่ 2 แทน โดยใช้ตัวเลขจากเครื่องของตนเอง)

In [ ]:
#@title แนวคำตอบ E11.7
_say = _gate(['my_shares'])
_say('''แนวคำตอบ E11.7
การ์ดที่ดีมีลักษณะสามข้อ
1 ส่วนการใช้งานระบุเงื่อนไขของข้อมูลเข้าให้ชัด เช่น ภาพสามแถบสีจาก DECaLS ที่ลดขนาดเป็น 64 x 64 ของกาแล็กซีที่
  Galaxy Zoo เลือกแล้ว ไม่ใช่ "ภาพกาแล็กซีทั่วไป"
2 ส่วนข้อจำกัดแยกสิ่งที่วัดแล้วออกจากสิ่งที่คาดการณ์ สิ่งที่วัดได้ใน E11.4 เช่น ค่าการเรียกคืนต่ำของประเภทซิการ์
  ความสับสนระหว่างประเภทที่ต่อเนื่องกัน และความแปรปรวนระหว่างโครงข่ายห้าตัว สิ่งที่คาดการณ์แต่ยังไม่ได้ทดสอบ เช่น
  ประสิทธิภาพที่ลดลงกับกาแล็กซีที่อยู่ไกลกว่า กับภาพจากกล้องหรือการสำรวจอื่นที่มีฟังก์ชันการกระจายของจุดต่างกัน
  และกับกาแล็กซีที่อาสาสมัครเห็นไม่ตรงกันซึ่งถูกคัดออกจากชุดข้อมูล
3 ส่วนจริยธรรมระบุสัญญาอนุญาต CC BY 4.0 การอ้างถึงผู้จัดทำชุดข้อมูล การสำรวจ DECaLS และ Galaxy Zoo
  และระบุว่าป้ายกำกับมาจากการลงคะแนนของอาสาสมัคร
การ์ดที่อ่อนมักมีเพียงตัวเลขความแม่นยำ และไม่บอกเลยว่าแบบจำลองจะผิดพลาดในสถานการณ์ใด''')

### โค้ดที่ 11.9 ฉบับเต็ม (ห้าโครงข่าย)
ในคาบที่ 4 ใช้การรวมกลุ่มฉบับย่อสามตัวเพราะเวลาไม่พอ เซลล์ถัดไปคือโค้ดที่ 11.9 ตามตำราทุกบรรทัด ซึ่งฝึกโครงข่ายห้าตัวด้วยเมล็ดสุ่ม 0 ถึง 4 (⏱ ราวหนึ่งชั่วโมงบน Colab ควรเปิดทิ้งไว้แล้วกลับมาดูผล) เมื่อรันเสร็จ ให้เปรียบเทียบกับฉบับย่อในคาบที่ 4 ว่าความแม่นยำของสมาชิกกระจายเท่าใด การรวมกลุ่มดีกว่าสมาชิกกี่ตัว และข้อสรุปเรื่องดาวฤกษ์และกาแล็กซีคู่เปลี่ยนหรือไม่

### โค้ดที่ 11.9 — การรวมกลุ่มโครงข่ายคอนโวลูชันกับภาพนอกการแจกแจง

In [ ]:
# ⏱ ราวหนึ่งชั่วโมงบน Colab (ฝึกโครงข่ายคอนโวลูชันห้าตัว)
_t0 = time.time()
def point_source(rng):                              # a star: PSF only, no extended light
    img = np.zeros((64, 64))
    c = (31.5 + rng.normal(0, 1.5, 2)).astype(int)
    img[c[0], c[1]] = rng.uniform(20, 60)
    img = np.arcsinh(gaussian_filter(img, 1.2) / 0.3)
    img = img / img.max() * rng.uniform(0.6, 1.0)
    return (img + rng.normal(0, 0.05, img.shape)).astype(np.float32)

def merger(rng):                                    # two smooth galaxies side by side
    a, b = make_galaxy(0, rng), make_galaxy(0, rng)
    return np.maximum(a, np.roll(b, rng.integers(10, 16), axis=1))

rng = np.random.default_rng(11)
X_star = np.array([point_source(rng) for _ in range(200)])
X_merge = np.array([merger(rng) for _ in range(200)])

members = []
for s in range(5):                                  # deep ensemble of 5 CNNs
    torch.manual_seed(s)
    members.append(GalaxyCNN())
    fit(members[-1], X_train, y_train, X_val, y_val, seed=s)
    print("member", s, np.mean(predict(members[-1], X_test).argmax(1) == y_test).round(3))

def ensemble(X):
    P = np.mean([predict(m, X) for m in members], axis=0)
    return P, -(P * np.log(P + 1e-12)).sum(axis=1)   # mean probability, entropy

P, H = ensemble(X_test)
ok = P.argmax(1) == y_test
print("ensemble accuracy", ok.mean().round(3),
      "| entropy: correct", H[ok].mean().round(3), "wrong", H[~ok].mean().round(3))
print("test images with max prob > 0.9:", np.mean(P.max(1) > 0.9).round(3))
for name, X in [('stars', X_star), ('mergers', X_merge)]:
    Pq, Hq = ensemble(X)
    print(name, "max prob > 0.9:", np.mean(Pq.max(1) > 0.9).round(2),
          "| predicted classes:", np.bincount(Pq.argmax(1), minlength=5))

print(f"five members trained and evaluated in {time.time() - _t0:.0f} s")

---
### สรุปบท
โครงข่ายคอนโวลูชันสร้างความรู้สองข้อเกี่ยวกับภาพไว้ในโครงสร้าง คือรูปแบบที่มีความหมายอยู่ในบริเวณเล็ก และรูปแบบเดียวกันมีความหมายเดียวกันทุกตำแหน่ง การเชื่อมต่อเฉพาะที่และการใช้น้ำหนักร่วมกันทำให้โครงข่ายในบทนี้มีพารามิเตอร์น้อยกว่าโครงข่ายหลายชั้นเกือบเก้าเท่า จำแนกภาพกาแล็กซีจำลองได้ดีกว่า และที่สำคัญกว่านั้น เสียความแม่นยำเพียงเล็กน้อยหรือไม่เสียเลยเมื่อกาแล็กซีเลื่อนออกจากกลางภาพ ขณะที่โครงข่ายหลายชั้นเสียไปกว่าครึ่ง ความสมมาตรต่อการเลื่อนนี้เป็นข้อจำกัดแบบแข็ง ส่วนความสมมาตรอื่น เช่น การหมุน ต้องสอนผ่านการเพิ่มข้อมูล ซึ่งต้องเลือกให้ตรงกับความสมมาตรของระบบที่วัดเสมอ

การอ่านผลของโครงข่ายต้องทำอย่างนักฟิสิกส์ เมทริกซ์ความสับสนของกาแล็กซีจำลองและของเมล็ดข้าวไทยแสดงว่าความผิดพลาดส่วนใหญ่อยู่ระหว่างประเภทที่มีความกำกวมทางกายภาพจริง การเลือกเอพอกด้วยข้อมูลตรวจสอบทำให้ตัวเลขดูดีเกินจริง และเป้าหมายของการใช้งาน เช่น การตรวจการปลอมปนข้าวหอมมะลิ กำหนดว่าตัวชี้วัดใดสำคัญ เมื่อข้อมูลน้อย การถ่ายโอนการเรียนรู้ช่วยโครงข่ายได้ แต่ลักษณะเด่นทางกายภาพที่ตรงกับความต่างจริงของข้อมูลอาจดีกว่า

Grad-CAM ช่วยจับทางลัดที่โครงข่ายเรียนจากความบังเอิญในข้อมูลฝึกได้อย่างชัดเจน แต่แผนที่ที่ชี้ไปยังบริเวณที่ดูสมเหตุสมผลไม่ได้พิสูจน์ว่าโครงข่ายใช้ฟิสิกส์ที่ถูกต้อง การรวมกลุ่มโครงข่ายเพิ่มความแม่นยำและบอกภาพที่ยากได้ แต่ไม่ได้เตือนเมื่อพบวัตถุที่ไม่เคยเห็น ซึ่งโครงข่ายยังคงตอบด้วยความมั่นใจสูง ท้ายที่สุด แบบจำลองต้องมีการ์ดที่บอกขอบเขตการใช้งานและข้อจำกัด และข้อมูลต้องใช้ตามสัญญาอนุญาตพร้อมให้เครดิตแก่ผู้ที่สร้างมัน บทสุดท้ายของตำราจะรวมเครื่องมือทั้งหมดเข้าเป็นโครงงานเดียว พร้อมกรอบของการทำซ้ำได้และจริยธรรม

### เอกสารอ้างอิง
- Adebayo, J., Gilmer, J., Muelly, M., Goodfellow, I., Hardt, M., & Kim, B. (2018). Sanity checks for saliency maps. In *Advances in Neural Information Processing Systems 31* (pp. 9505–9515).
- Cohen, T., & Welling, M. (2016). Group equivariant convolutional networks. In *Proceedings of the 33rd International Conference on Machine Learning* (PMLR Vol. 48, pp. 2990–2999).
- Dey, A., Schlegel, D. J., Lang, D., et al. (2019). Overview of the DESI Legacy Imaging Surveys. *The Astronomical Journal, 157*(5), 168. https://doi.org/10.3847/1538-3881/ab089d
- Gebru, T., Morgenstern, J., Vecchione, B., et al. (2021). Datasheets for datasets. *Communications of the ACM, 64*(12), 86–92. https://doi.org/10.1145/3458723
- Geirhos, R., Jacobsen, J.-H., Michaelis, C., Zemel, R., Brendel, W., Bethge, M., & Wichmann, F. A. (2020). Shortcut learning in deep neural networks. *Nature Machine Intelligence, 2*(11), 665–673. https://doi.org/10.1038/s42256-020-00257-z
- Hendrycks, D., & Gimpel, K. (2017). A baseline for detecting misclassified and out-of-distribution examples in neural networks. In *5th International Conference on Learning Representations (ICLR)*. https://doi.org/10.48550/arXiv.1610.02136
- Leung, H. W., & Bovy, J. (2024). *Galaxy10 DECaLS* (Version v1) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.10845026
- Lintott, C. J., Schawinski, K., Slosar, A., Land, K., Bamford, S., Thomas, D., Raddick, M. J., Nichol, R. C., Szalay, A., Andreescu, D., Murray, P., & Vandenberg, J. (2008). Galaxy Zoo: Morphologies derived from visual inspection of galaxies from the Sloan Digital Sky Survey. *Monthly Notices of the Royal Astronomical Society, 389*(3), 1179–1189. https://doi.org/10.1111/j.1365-2966.2008.13689.x
- Lupton, R., Blanton, M. R., Fekete, G., Hogg, D. W., O'Mullane, W., Szalay, A., & Wherry, N. (2004). Preparing red-green-blue images from CCD data. *Publications of the Astronomical Society of the Pacific, 116*(816), 133–137. https://doi.org/10.1086/382245
- Mehta, P., & Schwab, D. J. (2014). *An exact mapping between the variational renormalization group and deep learning* (arXiv:1410.3831). arXiv. https://doi.org/10.48550/arXiv.1410.3831
- Mitchell, M., Wu, S., Zaldivar, A., et al. (2019). Model cards for model reporting. In *Proceedings of the Conference on Fairness, Accountability, and Transparency (FAT\* '19)* (pp. 220–229). https://doi.org/10.1145/3287560.3287596
- Nguyen, A., Yosinski, J., & Clune, J. (2015). Deep neural networks are easily fooled: High confidence predictions for unrecognizable images. In *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR)* (pp. 427–436). https://doi.org/10.1109/CVPR.2015.7298640
- Selvaraju, R. R., Cogswell, M., Das, A., Vedantam, R., Parikh, D., & Batra, D. (2017). Grad-CAM: Visual explanations from deep networks via gradient-based localization. In *Proceedings of the IEEE International Conference on Computer Vision (ICCV)* (pp. 618–626). https://doi.org/10.1109/ICCV.2017.74
- Walmsley, M., Lintott, C., Géron, T., Kruk, S., et al. (2022). Galaxy Zoo DECaLS: Detailed visual morphology measurements from volunteers and deep learning for 314,000 galaxies. *Monthly Notices of the Royal Astronomical Society, 509*(3), 3966–3988. https://doi.org/10.1093/mnras/stab2093
- Yosinski, J., Clune, J., Bengio, Y., & Lipson, H. (2014). How transferable are features in deep neural networks? In *Advances in Neural Information Processing Systems 27* (pp. 3320–3328).
- Zhou, B., Khosla, A., Lapedriza, A., Oliva, A., & Torralba, A. (2016). Learning deep features for discriminative localization. In *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR)* (pp. 2921–2929). https://doi.org/10.1109/CVPR.2016.319
- ณภัทร กาญจนา, รัญชน์ นาคจีน, กฤษฎิ์ เนตยากร และบุษรา พัฒนศิริ. (2564). การจำแนกพันธุ์เมล็ดข้าวไทยด้วยโครงข่ายประสาทเทียมแบบคอนโวลูชัน. ใน *การประชุมวิชาการระดับชาติ ครั้งที่ 18 มหาวิทยาลัยเกษตรศาสตร์ วิทยาเขตกำแพงแสน* (หน้า 143–150). นครปฐม. (อ้างในโน้ตบุ๊กเป็น Kanchana et al., 2021)